# Benchmark v4 — Robust & Progressive Adaptive INT8
**Финальная независимая проверка**, две GPU Kaggle. Исходники исследовательского стенда включены в notebook. Включите T4 ×2 и Internet. Профили: `final_smoke` для проверки окружения, `final` для основного эксперимента, `final_full` для расширенного эксперимента.

## Протокол (зафиксирован до новых результатов)
- Новые checkpoint: EfficientNet-B0, ViT-Tiny, MiniLM-L6; CIFAR-10 и AG News не использовались в v3. Изображения CIFAR-10 проходят ImageNet-модели: оцениваем **fidelity и agreement**, а не точность классификации CIFAR-10. MiniLM — fidelity embeddings, не качество downstream retrieval.
- Дополнительные контролируемые модели: Digits MLP, PCA autoencoder, multi-output, Diabetes regression, обученная LSTM (квантуется только поддерживаемая выходная голова, recurrent ONNX LSTM не поддержан).
- Fit строит кандидатов; исходный dev делится 50%/25%/25% на local-dev, global-search, global-check. Test и shift только для отчёта; все политики заморожены заранее. Test фиксирован по seed. На малых датасетах фактические размеры меньше профиля.
- Astra prior из v3 заморожен. Progressive: Single → Paired → SDR, остановка при робастной границе ≤ −25% локальной MinMax dev-MSE. Это заранее заданная гипотеза, не подобранная по новому тесту. При отсутствии выигрыша — MinMax.
- Все политики, кроме `*-Local`, проходят независимую проверку всех численных выходов против полной MinMax-модели. `*-Local` — абляции без глобального отката. Progressive-Global пробует оригинальный V2 для классификации или ограниченный точный greedy для других выходов, не более 4 операторов. V2 отдельно доступен как исторический baseline только классификации.
- Один полный банк кандидатов строится реально, политики **replay** читают только разрешённые элементы. Стоимость отдельного построения оценивается по измеренным временам нужных операторов, не является фактическим standalone временем. Политики с одинаковым выбором могут переиспользовать измерения одной и той же ONNX-модели. Фактические время replay и глобального поиска отдельно записываются. Формат INT8 одинаков.
- Основная гипотеза: Progressive с global-check выигрывает у Single-Robust с таким же check при приемлемой цене. Простые методы и полный банк — обязательные контроли. Робастная гарантия условна относительно множества неопределённости; проверка на dev не гарантирует test.
- Критерии инженерного завершения: корректная модель, одинаковый размер с MinMax, медианная по seed ошибка **каждого** выхода ≤ 1.02 от MinMax на test и ≤ 1.05 на shift; ≥5% выигрыша на test хотя бы на двух моделях (по худшему отношению среди выходов); для доступных task-метрик accuracy не хуже на >1 п.п., RMSE не хуже на >2%. Среди прошедших защищённых стратегий выбирается по ошибке/стоимости, либо MinMax по умолчанию. Это инженерные пороги, а не доказательство статистической значимости или универсальной гарантии. Новый независимый тест для утверждения превосходства уже выбранной политики потребуется при публикации.


In [ ]:
from pathlib import Path
import sys, os, subprocess, json, zipfile, shutil, time, threading

PROFILE = "final_smoke"  # final_smoke / final / final_full
INSTALL_MISSING = True
CPU_THREADS = 2        # на одного GPU-воркера
GPU_COUNT = 2
PROVIDER = "cuda"
MATH_BACKEND = "torch"  # numpy — оригинальный эталон
CAPTURE_GROUP = 4
ALLOW_CONV_CPU_FALLBACK = True  # False = только CUDA-инференс, ошибка CNN не скрывается
RUN_PREFLIGHT = True   # дополнительные unit-тесты; проверка GPU обязательна независимо от флага
MODELS = None
SEEDS = None
METHODS = ["FP32", "MinMax", "Single-3", "Paired-3", "SDR", "V2",
           "Single-Dev", "Single-Robust", "Paired-Robust", "SDR-Robust",
           "Astra-Adaptive", "Progressive", "FullBank-Robust", "Progressive-Global",
           "Progressive-Local", "Single-Robust-Local", "Single-Dev-Local"]
ROWS_OVERRIDE = None
CHANNEL_CAP = None    # 0 = все каналы с учётом бюджета
V2_LAYER_CAP = None   # 0 = все различающиеся тензоры
SDR_WORK_OVERRIDE = None
SAVE_MODELS = False

ROOT = Path("/kaggle/working/quant_benchmark_v4") if Path("/kaggle/working").exists() else Path.cwd()/"quant_benchmark_v4"
ROOT.mkdir(parents=True, exist_ok=True)
OUT = ROOT / "results" / PROFILE
OUT.mkdir(parents=True, exist_ok=True)
print("Project:", ROOT, "Results:", OUT)


## 1. Развёртывание встроенных исходников
Повторное выполнение сохраняет результаты с тем же fingerprint. После полностью завершённого model/seed большие банки и активации удаляются, отчёты сохраняются. При SAVE_MODELS=True они сохраняются вместе с ONNX.

### Алгоритмы квантования и тесты

In [ ]:
SOURCES = {'algorithms/__init__.py': '', 'algorithms/activation_int8.py': '"""Симметричное INT8-квантование весов с учётом калибровочных активаций."""\n\nimport numpy as np\n\n\ndef _matrix(values):\n    values = np.asarray(values)\n    if values.ndim != 2 or 0 in values.shape or values.dtype.kind not in "fiu":\n        raise ValueError("Ожидается непустая двумерная вещественная матрица.")\n    with np.errstate(over="ignore", invalid="ignore"):\n        values = values.astype(np.float32, copy=False)\n    if not np.isfinite(values).all():\n        raise ValueError("Элементы должны быть конечными в формате FP32.")\n    return values.astype(np.float64)\n\n\ndef _scale32(values):\n    # Любой из 255 симметричных кодов должен деквантоваться в конечный FP32.\n    smallest = np.nextafter(np.float32(0), np.float32(1))\n    largest = float(np.finfo(np.float32).max)\n    upper = np.float32(largest / 127)\n    if float(upper) * 127 > largest:\n        upper = np.nextafter(upper, np.float32(0))\n    return np.clip(values, float(smallest), float(upper)).astype(np.float32)\n\n\ndef _pairs(x):\n    pairs = []\n    for start in range(0, x.shape[1], 32):\n        block = x[:, start:start + 32]\n        gram = block.T @ block\n        energy = gram.diagonal()\n        total = energy[:, None] + energy\n        scores = np.divide(2 * abs(gram), total, out=np.zeros_like(gram),\n                           where=total > 0)\n        for i in range(len(gram) - 1):\n            j = i + 1 + int(np.argmax(scores[i, i + 1:]))\n            if scores[i, j] > 0:\n                sign = -1 if gram[i, j] > 0 else 1\n                pairs.append((start + i, start + j, sign))\n                scores[:, j] = 0\n                scores[j, :] = 0\n    return pairs\n\n\ndef _move(residual, direction, codes, scales, i, j=None, sign=1):\n    energy = direction @ direction\n    if energy == 0:\n        return\n    low, high = -127 - codes[i], 127 - codes[i]\n    if j is not None:\n        second = sign * codes[j]\n        low = np.maximum(low, -127 - second)\n        high = np.minimum(high, 127 - second)\n    projection = np.einsum("n,nb->b", direction, residual)\n    step = np.rint(np.clip(projection / (energy * scales), low, high)).astype(np.int16)\n    if np.any(step):\n        codes[i] += step\n        if j is not None:\n            codes[j] += sign * step\n        residual -= direction[:, None] * (scales * step)\n\n\ndef _loss(x, weights, codes, scales):\n    decoded = np.multiply(codes, scales, dtype=np.float32)\n    residual = x @ (weights - decoded)\n    return np.einsum("nb,nb->b", residual, residual)\n\n\ndef quantize_int8(activations, weights, *, sweeps=3):\n    """Возвращает (коды INT8, масштабы FP32) для произведения X @ W.\n\n    X имеет форму (наблюдения, входы), W — (входы, выходы). Вещественные\n    входные данные приводятся к FP32. Коды имеют форму W, масштабы — (выходы,).\n    Параметр sweeps ограничивает число проходов. Ноль возвращает MinMax.\n    """\n    x, w = _matrix(activations), _matrix(weights)\n    if x.shape[1] != w.shape[0]:\n        raise ValueError("Число входных признаков X и W должно совпадать.")\n    integer = isinstance(sweeps, (int, np.integer)) and not isinstance(sweeps, (bool, np.bool_))\n    if not integer or sweeps < 0:\n        raise ValueError("Число проходов должно быть целым неотрицательным.")\n\n    peak = np.max(abs(w), axis=0)\n    scales = _scale32(np.where(peak == 0, 1.0, peak / 127))\n    codes = np.clip(np.rint(w / scales), -127, 127).astype(np.int8)\n    if sweeps == 0:\n        return codes, scales\n    pairs = _pairs(x)\n\n    for start in range(0, w.shape[1], 32):\n        cols = slice(start, start + 32)\n        weights_block, best, stored = w[:, cols], codes[:, cols], scales[cols]\n        best_loss = _loss(x, weights_block, best, stored)\n        for _ in range(sweeps):\n            if not np.any(best_loss):\n                break\n            scale = stored.astype(np.float64)\n            trial = best.astype(np.int16)\n            residual = x @ (weights_block - trial * scale)\n            for i in range(x.shape[1]):\n                _move(residual, x[:, i], trial, scale, i)\n            for i, j, sign in pairs:\n                _move(residual, x[:, i] + sign * x[:, j], trial, scale, i, j, sign)\n\n            # Пересчитываем остаток перед одномерной задачей по масштабу.\n            residual = x @ (weights_block - trial * scale)\n            response = x @ trial\n            energy = np.einsum("nb,nb->b", response, response)\n            cross = np.einsum("nb,nb->b", response, residual)\n            delta = np.divide(cross, energy, out=np.zeros_like(cross), where=energy > 0)\n            fitted = _scale32(scale + delta)\n\n            improved = False\n            for candidate in (stored.copy(), fitted):\n                loss = _loss(x, weights_block, trial, candidate)\n                keep = loss < best_loss\n                if np.any(keep):\n                    best[:, keep] = trial[:, keep]\n                    stored[keep] = candidate[keep]\n                    best_loss[keep] = loss[keep]\n                    improved = True\n            if not improved:\n                break\n    return codes, scales\n', 'algorithms/adaptive_selector.py': '"""Выбор алгоритма для одного формата симметричных весов INT8."""\n\nfrom dataclasses import dataclass\nfrom itertools import combinations\nimport numpy as np\n\nMETHODS = ("single", "paired", "sdr")\nBASE = "minmax"\n\n\ndef features(d, n, p, stable, corr, peak, rms):\n    """Общие для архива и исполнения безразмерные признаки до построения кандидатов."""\n    if min(d, n, p) <= 0 or min(stable, corr, peak, rms) < 0:\n        raise ValueError("Некорректные размеры или характеристики.")\n    f = np.array([np.log1p(d/n), stable/p, corr, np.log1p(peak/rms if rms else 0.)])\n    if not np.isfinite(f).all():\n        raise ValueError("Неконечный профиль.")\n    return f\n\n\ndef _matrix(a):\n    a = np.asarray(a)\n    if a.dtype != np.float32 or a.ndim != 2 or not a.size:\n        raise ValueError("Нужна непустая двумерная матрица FP32.")\n    for part in np.nditer(a, flags=["buffered", "external_loop"], buffersize=8192):\n        if not np.isfinite(part).all():\n            raise ValueError("Неконечный вход.")\n    return a\n\n\ndef profile(w, x):\n    """Профиль по калибровке; спектр относится только к выбранным координатам."""\n    w, x = _matrix(w), _matrix(x)\n    d = w.shape[0]\n    if x.shape[1] != d:\n        raise ValueError("Несогласованные размерности.")\n    p = min(64, d)\n    cols = np.linspace(0, d-1, p, dtype=int)\n    gram = np.zeros((p, p))\n    for i in range(0, len(x), 512):\n        part = x[i:i+512, cols].astype(float)\n        gram += part.T @ part\n    values, vectors = np.linalg.eigh(gram)\n    values = values.clip(0)\n    total = values.sum()\n    stable = total/values[-1] if total else 0.\n    norm = np.sqrt(gram.diagonal())\n    denom = norm[:, None]*norm\n    corr = np.divide(gram, denom, out=np.zeros_like(gram), where=denom>0)\n    np.fill_diagonal(corr, 0.)\n    correlation = abs(corr).sum()/max(1, p*(p-1))\n    energy = 0.\n    for part in np.nditer(w, flags=["buffered", "external_loop"], buffersize=8192):\n        a = part.astype(float)\n        energy += a @ a\n    rms = np.sqrt(energy/w.size)\n    peak = max(abs(float(w.min())), abs(float(w.max())))\n    probs = values[values>0]/total if total else np.array([])\n    top = vectors[:, values >= values[-1]*(1-1e-10)] if total else vectors[:, :0]\n    coherence = p*np.max(np.sum(top**2, axis=1))/top.shape[1] if total else 0.\n    return features(d, len(x), p, stable, correlation, peak, rms), dict(\n        sampled=p, stable=stable, entropy_rank=np.exp(-probs @ np.log(probs)) if total else 0.,\n        coherence=coherence, gram=gram/len(x))\n\n\nclass Portfolio:\n    """История по доменам: одна строка профиля и три выигрыша на уникальный оператор."""\n\n    def __init__(self, domains):\n        self.domains = []\n        for x, g in domains:\n            x, g = np.asarray(x, float), np.asarray(g, float)\n            if (x.ndim != 2 or x.shape[1] != 4 or not len(x) or g.shape != (len(x), 3)\n                    or not np.isfinite(x).all() or not np.isfinite(g).all()):\n                raise ValueError("Неверная форма или значения обучающей истории.")\n            self.domains.append((x.copy(), np.clip(g, 0, 1)))\n        if self.domains:\n            all_x = np.concatenate([x for x, _ in self.domains])\n            self.low, self.high = all_x.min(0), all_x.max(0)\n            spread = np.diff(np.quantile(all_x, [.25, .75], axis=0), axis=0)[0]\n            self.scale = np.where(spread>1e-12, spread, np.maximum(self.high-self.low, 1e-12))\n\n    def plan(self, f, quotes, budget):\n        """Резервирует подмножество по сметам без вызовов квантователей."""\n        f = np.asarray(f, float)\n        if f.shape != (4,) or not np.isfinite(f).all() or not np.isfinite(budget) or budget < 0:\n            raise ValueError("Неверный профиль или бюджет.")\n        if any(k not in METHODS or not np.isfinite(v) or v < 0 for k,v in quotes.items()):\n            raise ValueError("Недопустимая смета.")\n        if not self.domains:\n            return (), "no_history"\n        if np.any((f < self.low-1e-10) | (f > self.high+1e-10)):\n            return (), "outside_history"\n        neighbors = []\n        for x, g in self.domains:\n            order = np.argsort(np.sum(((x-f)/self.scale)**2, axis=1), kind="stable")\n            neighbors.append(g[order[:int(np.ceil(np.sqrt(len(x))))]])\n        best, chosen = (0., 0., 0), ()\n        for count in range(1, 4):\n            for ids in combinations(range(3), count):\n                names = tuple(METHODS[i] for i in ids)\n                if any(k not in quotes for k in names):\n                    continue\n                cost = sum(quotes[k] for k in names)\n                if cost > budget:\n                    continue\n                gain = min(np.max(g[:, ids], axis=1).mean() for g in neighbors)\n                score = (float(gain), -cost, -count)\n                if score > best:\n                    best, chosen = score, names\n        return chosen, "covered" if chosen else "budget_or_no_gain"\n\n\n@dataclass(frozen=True)\nclass Loss:\n    """Средние квадраты на fit/dev и энергия ошибки весов на выходной канал."""\n    fit: float\n    dev: float\n    energy: float\n\n\ndef paired_upper(differences, span, candidates, alpha=.05):\n    """Верхняя граница Хёффдинга для средних разностей независимых единиц.\n\n    span — заранее известная ширина диапазона, а не размах данной выборки.\n    candidates — число проверяемых альтернатив до просмотра этих данных.\n    """\n    d = np.asarray(differences, float)\n    if (d.ndim != 1 or not d.size or not np.isfinite(d).all() or not 0 < alpha < 1\n            or type(candidates) is not int or candidates < 1 or not np.isfinite(span)\n            or span < 0 or np.ptp(d) > span):\n        raise ValueError("Неверные предпосылки для оценки.")\n    return d.mean() + span*np.sqrt(np.log(candidates/alpha)/(2*len(d)))\n\n\ndef choose(losses, radius=None, confidence=None):\n    """Выбирает по верхней границе приращения потерь относительно MinMax."""\n    b = losses[BASE]\n    if any(not np.isfinite(v) or v < 0 for l in losses.values() for v in vars(l).values()):\n        raise ValueError("Потери и энергии должны быть конечными и неотрицательными.")\n    if radius is not None and (not np.isfinite(radius) or radius < 0):\n        raise ValueError("Недопустимый радиус.")\n    rho = (abs(b.dev-b.fit)/b.energy if b.energy else 0.) if radius is None else radius\n    bounds = {BASE: 0.}\n    for name, l in losses.items():\n        if name == BASE:\n            continue\n        fit, dev = l.fit-b.fit, l.dev-b.dev\n        bounds[name] = dev + abs(dev-fit) + rho*(l.energy+b.energy)\n        if confidence is not None:\n            upper = confidence.get(name, np.inf)\n            bounds[name] = max(bounds[name], upper) if np.isfinite(upper) else np.inf\n    name = min(bounds, key=bounds.get)\n    return name, dict(radius=rho, upper=bounds, reason="robust_gain" if name != BASE else "baseline")\n\n\ndef _payload(pair, shape):\n    q, s = pair\n    q, s = np.asarray(q), np.asarray(s)\n    if (q.shape != shape or q.dtype != np.int8 or s.dtype != np.float32\n            or s.shape != (shape[1],) or np.any(q == -128)\n            or not np.all(np.isfinite(s) & (s>0))):\n        raise ValueError("Некорректный формат INT8.")\n    peak = np.maximum(q.max(axis=0), -q.min(axis=0))\n    with np.errstate(over="ignore"):\n        if not np.isfinite(peak*s).all():\n            raise ValueError("Переполнение деквантизации.")\n    return q, s\n\n\ndef _loss(w, pair, fit, dev):\n    q, s = _payload(pair, w.shape)\n    energy = 0.\n    sums = [0., 0.]\n    for j in range(0, w.shape[1], 32):\n        sl = slice(j, j+32)\n        with np.errstate(over="raise", invalid="raise"):\n            e = (q[:, sl]*s[sl]).astype(float)-w[:, sl]\n        energy += np.sum(e*e)\n        for k, x in enumerate((fit, dev)):\n            for i in range(0, len(x), 128):\n                r = x[i:i+128].astype(float) @ e\n                sums[k] += np.sum(r*r)\n    m = w.shape[1]\n    return Loss(sums[0]/(len(fit)*m), sums[1]/(len(dev)*m), energy/m)\n\n\ndef select(w, fit, dev, baseline, builders, prior, quotes, budget=1., radius=None,\n           confidence_fn=None):\n    """Строит MinMax и только допущенные варианты. Функции принимают (W, Xfit).\n    \n    Возвращает имя, выбранную пару, банк допустимых пар и диагностику.\n    Сметы в единицах вызывающей стороны; бюджет покрывает дополнительные построения.\n    """\n    w = _matrix(w)\n    base = _payload(baseline(w, fit), w.shape)\n    bank = {BASE: base}\n    if fit is None or dev is None:\n        return BASE, base, bank, {"reason": "missing_split", "attempted": ()}\n    try:\n        fit, dev = _matrix(fit), _matrix(dev)\n    except ValueError:\n        return BASE, base, bank, {"reason": "invalid_split", "attempted": ()}\n    if dev.shape[1] != w.shape[0] or fit.shape[1] != w.shape[0]:\n        raise ValueError("Несогласованные входы оператора.")\n    try:\n        vec, diag = profile(w, fit)\n    except np.linalg.LinAlgError:\n        return BASE, base, bank, {"reason": "profile_failure", "attempted": ()}\n    allowed, pre = prior.plan(vec, {k:v for k,v in quotes.items() if k in builders}, budget)\n    if not allowed:\n        return BASE, base, bank, {"reason": pre, "attempted": (), "profile": diag}\n    losses = {BASE: _loss(w, base, fit, dev)}\n    failed = {}\n    for name in allowed:\n        try:\n            pair = _payload(builders[name](w, fit), w.shape)\n            losses[name] = _loss(w, pair, fit, dev)\n            bank[name] = pair\n        except (ArithmeticError, TypeError, ValueError, np.linalg.LinAlgError, MemoryError) as err:\n            failed[name] = str(err)\n    confidence = None if confidence_fn is None else {\n        k: confidence_fn(w, base, pair, dev, len(allowed)) for k, pair in bank.items() if k != BASE}\n    name, reason = choose(losses, radius, confidence)\n    reason.update(profile=diag, pre=pre, attempted=allowed, failed=failed,\n                  reserved=sum(quotes[k] for k in allowed), losses=losses)\n    return name, bank[name], bank, reason\n\n\ndef search_priority(losses, radius=None):\n    """Оценивает ширину интервала относительного выигрыша для очереди проб V2."""\n    _, info = choose(losses, radius)\n    b = losses[BASE]\n    if not b.dev:\n        return 0.\n    widths = []\n    for k, l in losses.items():\n        if k != BASE:\n            center = l.dev-b.dev\n            half = info["upper"][k]-center\n            if center-half < 0:\n                widths.append(min(1., half/b.dev))\n    return max(widths, default=0.)\n\n\ndef global_plan(options, priority, budget, limit=12, probe_price=0.):\n    """Отбирает небольшую область для V2 и резервирует окончательные проверки."""\n    if (type(budget) is not int or budget < 0 or type(limit) is not int or limit < 0\n            or not np.isfinite(probe_price) or probe_price < 0):\n        raise ValueError("Нужны целые неотрицательные лимиты.")\n    remaining = budget-6\n    active = []\n    def density(k):\n        return priority.get(k, 0.)/max(1, len(set(options[k]))-1)\n    for name in sorted(options, key=lambda k: -density(k)):\n        probes = len(set(options[name]))-1\n        if 0 < probes <= remaining and density(name) > probe_price and len(active)<limit:\n            active.append(name)\n            remaining -= probes\n    extra_cost = 3 + sum(len(set(options[k]))-1 for k in active)\n    return tuple(active) if sum(priority[k] for k in active) > probe_price*extra_cost else ()\n\n\ndef confirm_outputs(reference, baseline, proposal):\n    """Возвращает вектор MSE по именованным численным выходам фиксированного сравнения."""\n    if not reference or set(reference) != set(baseline) or set(reference) != set(proposal):\n        raise ValueError("Не совпадают выходы.")\n    old, new = [], []\n    for k in sorted(reference):\n        arrays = [np.asarray(z[k]) for z in (reference, baseline, proposal)]\n        if (any(a.dtype.kind not in "fiu" for a in arrays)\n                or any(a.shape != arrays[0].shape for a in arrays) or not arrays[0].size):\n            raise ValueError("Нужны согласованные численные выходы.")\n        if not all(np.isfinite(a).all() for a in arrays):\n            raise ValueError("Неконечный выход.")\n        big_int = any(a.dtype.kind in "iu" and\n                      (int(a.min()) < -(2**53) or int(a.max()) > 2**53) for a in arrays)\n        if big_int and any(a.dtype.kind == "f" for a in arrays):\n            raise ValueError("Нужен адаптер единиц для смешанных типов с большими целыми.")\n        ref, b, p = (a.astype(object if big_int else float) for a in arrays)\n        with np.errstate(over="raise", invalid="raise"):\n            old.append(np.mean(np.asarray(b-ref, dtype=float)**2))\n            new.append(np.mean(np.asarray(p-ref, dtype=float)**2))\n    return np.array(old), np.array(new)\n\n\ndef finalize(local, options, priority, infer_fit, infer_check, budget, suggest=None, limit=12,\n             probe_price=0.):\n    """Проверяет локальную сборку и необязательное предложение внешнего V2.\n    \n    Конфигурации — словари имён вариантов. None обозначает FP32. suggest\n    принимает локальную сборку, варианты активных операторов и лимитированный\n    вызов infer_fit. infer_check использует отдельную фиксированную выборку.\n    """\n    base = dict.fromkeys(local, BASE)\n    if set(options) != set(local) or any(BASE not in options[k] or v not in options[k]\n                                         for k,v in local.items()):\n        raise ValueError("Несогласованный банк.")\n    planned = global_plan(options, priority, budget, limit, probe_price)\n    active = planned if suggest else ()\n    if budget < 3 or (local == base and not active):\n        return base, {"calls": 0, "reason": "baseline"}\n    calls = 0\n    proposals = [dict(local)] if local != base else []\n    if active:\n        cap = 2 + sum(len(set(options[k]))-1 for k in active)\n        def valid(config):\n            return (isinstance(config, dict) and set(config) == set(local)\n                    and all(v in options[k] for k,v in config.items())\n                    and all(config[k] == local[k] for k in local if k not in active))\n        def probe(config):\n            nonlocal calls\n            if calls >= cap:\n                raise ValueError("Исчерпан бюджет V2.")\n            if config is not None and not valid(config):\n                raise ValueError("Запрошена конфигурация вне активного банка.")\n            calls += 1\n            return infer_fit(config)\n        try:\n            candidate = suggest(dict(local), {k: tuple(options[k]) for k in active}, probe)\n            if valid(candidate) and candidate != base and candidate not in proposals:\n                proposals.append(dict(candidate))\n        except (ArithmeticError, TypeError, ValueError, np.linalg.LinAlgError):\n            pass\n    if not proposals:\n        return base, {"calls": calls, "reason": "empty"}\n    # Копии нужны, если адаптер переиспользует свои буферы вывода.\n    def check(config):\n        nonlocal calls\n        calls += 1\n        return {k: np.array(v, copy=True) for k,v in infer_check(config).items()}\n    winner = base\n    best = None\n    try:\n        ref, original = check(None), check(base)\n        for candidate in proposals:\n            try:\n                old, new = confirm_outputs(ref, original, check(candidate))\n            except (ArithmeticError, ValueError):\n                continue\n            if np.all(new <= old) and np.any(new < old):\n                score = np.mean(np.divide(new, old, out=np.zeros_like(new), where=old>0))\n                if best is None or score < best:\n                    best, winner = score, candidate\n    except (ArithmeticError, ValueError):\n        return base, {"calls": calls, "reason": "invalid_reference"}\n    return winner, {"calls": calls, "active": active, "reason": "checked"}', 'algorithms/logit_int8.py': '"""Послойный выбор вариантов INT8 по откликам конечных логитов."""\n\nimport numpy as np\n\n\ndef _validate(bank):\n    if not bank:\n        raise ValueError("Нужна хотя бы одна матрица.")\n    for name, variants in bank.items():\n        if not variants:\n            raise ValueError(f"{name}: отсутствует базовый вариант.")\n        shape = None\n        for codes, scales in variants:\n            if (codes.dtype != np.int8 or codes.ndim != 2 or 0 in codes.shape\n                    or (shape is not None and codes.shape != shape)):\n                raise ValueError(f"{name}: нужны матрицы INT8 одной формы.")\n            shape = codes.shape\n            if (scales.dtype != np.float32 or scales.shape != (shape[1],)\n                    or not np.all(np.isfinite(scales) & (scales > 0))\n                    or np.any(codes == -128)):\n                raise ValueError(f"{name}: неверные коды или масштабы.")\n            peak = np.maximum(codes.max(axis=0), -codes.min(axis=0))\n            with np.errstate(over="ignore"):\n                if not np.isfinite(peak * scales).all():\n                    raise ValueError(f"{name}: переполнение при деквантизации.")\n\n\ndef _logits(infer, weights, data, shape=None):\n    raw = np.asarray(infer(weights, data))\n    if (raw.ndim != 2 or raw.shape[0] == 0 or raw.shape[1] < 2\n            or raw.dtype.kind != "f" or (shape is not None and raw.shape != shape)):\n        raise ValueError("Ожидаются логиты формы (примеры, классы) в едином порядке.")\n    z = raw.astype(np.float64, copy=True)\n    if not np.isfinite(z).all() or np.max(np.abs(z)) > float(np.finfo(np.float32).max):\n        raise ValueError("Логиты должны быть конечными и в диапазоне FP32.")\n    return z\n\n\nclass _Target:\n    def __init__(self, reference):\n        self.reference = reference\n        self.labels = reference.argmax(axis=1)\n        self.rows = np.arange(len(reference))\n        self.rivals = np.arange(reference.shape[1]) != self.labels[:, None]\n        self.floor = 0.5 * self.margin(reference)\n\n    def margin(self, z):\n        rival = np.max(z, axis=1, where=self.rivals, initial=-np.inf)\n        return z[self.rows, self.labels] - rival\n\n    def loss(self, z):\n        mse = np.mean(np.square(z - self.reference))\n        deficit = np.maximum(self.floor - self.margin(z), 0.0)\n        return mse + np.mean(np.square(deficit)), mse\n\n\ndef select_int8(bank, infer, calibration, development, *, width=4):\n    """Возвращает словарь индексов выбранных вариантов по именам матриц.\n\n    bank задаёт порядок обхода. Для каждого имени нужна последовательность\n    пар (коды INT8 [входы, выходы], масштабы FP32 [выходы]). Варианты\n    строятся на калибровке. Нулевые индексы задают исходную INT8-модель.\n    infer(None, data) запускает FP32, infer(полный_словарь_пар, data) —\n    модель с указанными весами. Каждый вызов независим и возвращает логиты\n    всей выборки в одном порядке. Выборки должны быть непересекающимися\n    и пригодными для повторного прохода. При равных логитах побеждает\n    первый класс. width ограничивает число частичных решений в поиске.\n    """\n    if (isinstance(width, (bool, np.bool_)) or not isinstance(width, (int, np.integer))\n            or width < 1):\n        raise ValueError("Ширина поиска должна быть целым положительным числом.")\n    _validate(bank)\n    base = {name: variants[0] for name, variants in bank.items()}\n    zeros = dict.fromkeys(bank, 0)\n    fit = _Target(_logits(infer, None, calibration))\n    anchor = _logits(infer, base, calibration, fit.reference.shape)\n    base_loss = fit.loss(anchor)[0]\n    beam = [((), anchor)]\n\n    for name, variants in bank.items():\n        shifts = [None]\n        for variant in variants[1:]:\n            model = {**base, name: variant}\n            shifts.append(_logits(infer, model, calibration, anchor.shape) - anchor)\n\n        ranked = []\n        for parent, (_, z) in enumerate(beam):\n            for k, shift in enumerate(shifts):\n                trial = z if k == 0 else z + shift\n                ranked.append((fit.loss(trial)[0], parent, k))\n        ranked.sort()\n        survivors = []\n        for _, parent, k in ranked[:width]:\n            path, z = beam[parent]\n            survivors.append((path + (k,), z if k == 0 else z + shifts[k]))\n        beam = survivors\n\n    path, estimate = beam[0]\n    if not any(path) or fit.loss(estimate)[0] >= base_loss:\n        return zeros\n    indices = dict(zip(bank, path))\n    proposal = {name: bank[name][k] for name, k in indices.items()}\n    dev = _Target(_logits(infer, None, development))\n    if dev.reference.shape[1] != anchor.shape[1]:\n        raise ValueError("Наборы классов на калибровке и проверке должны совпадать.")\n    old = _logits(infer, base, development, dev.reference.shape)\n    new = _logits(infer, proposal, development, dev.reference.shape)\n    old_loss, old_mse = dev.loss(old)\n    new_loss, new_mse = dev.loss(new)\n    old_ok = old.argmax(axis=1) == dev.labels\n    new_ok = new.argmax(axis=1) == dev.labels\n    if new_loss > old_loss or new_mse > old_mse or np.any(old_ok & ~new_ok):\n        return zeros\n    if new_loss < old_loss or new_mse < old_mse or np.any(new_ok & ~old_ok):\n        return indices\n    return zeros\n', 'algorithms/qk_svd_int8.py': '"""Квантование одной обычной головы внимания через факторизацию формы QK."""\n\nimport numpy as np\n\n\ndef _fp32(value, ndim):\n    a = np.asarray(value)\n    if a.dtype != np.float32 or a.ndim != ndim or not a.size or not np.isfinite(a).all():\n        raise ValueError("Ожидается непустой конечный массив FP32 заданной размерности.")\n    return a\n\n\ndef minmax_int8(weights):\n    """Возвращает матрицу кодов и вектор масштабов по её выходным столбцам."""\n    w = _fp32(weights, 2).astype(np.float64)\n    peak = np.max(np.abs(w), axis=0)\n    limit = float(np.finfo(np.float32).max)\n    upper = np.float32(limit / 127)\n    if 127 * float(upper) > limit:\n        upper = np.nextafter(upper, np.float32(0))\n    lower = np.nextafter(np.float32(0), np.float32(1))\n    s = np.clip(np.where(peak > 0, peak / 127, 1), lower, upper).astype(np.float32)\n    return np.rint(w / s).clip(-127, 127).astype(np.int8), s\n\n\ndef _augment(w, bias):\n    if bias is None:\n        return w\n    b = _fp32(bias, 1)\n    if b.shape != (w.shape[1],):\n        raise ValueError("Длина смещения должна совпадать с шириной головы.")\n    return np.vstack((w, b))\n\n\ndef _balanced(a, b):\n    qa, ra = np.linalg.qr(a)\n    qb, rb = np.linalg.qr(b)\n    u, s, vt = np.linalg.svd(ra @ rb.T, full_matrices=False)\n    root = np.sqrt(s)\n    left, right = np.zeros_like(a), np.zeros_like(b)\n    left[:, :s.size] = (qa @ u) * root\n    right[:, :s.size] = (qb @ vt.T) * root\n    return left, right\n\n\ndef _pack(factor, rows):\n    q, s = minmax_int8(factor[:rows])\n    bias = factor[-1].copy() if factor.shape[0] > rows else None\n    return q, s, bias\n\n\ndef _decoded(packed):\n    q, s, bias = packed\n    return _augment(q * s, bias)\n\n\ndef _core_blocks(matrices):\n    joined = np.concatenate(matrices, axis=1).astype(np.float64, copy=False)\n    return np.hsplit(np.linalg.qr(joined, mode="r"), len(matrices))\n\n\ndef quantize_qk_head(query, key, query_bias=None, key_bias=None):\n    """Возвращает две тройки (коды, масштабы, смещение) и признак выбора SVD.\n\n    Входы — FP32-матрицы одной головы в ориентации (входы, ширина головы).\n    Смещения необязательны. Допустим прямой скалярный продукт Q и K без\n    дополнительных потребителей, позиционных вращений и нормализации Q/K.\n    """\n    wq, wk = _fp32(query, 2), _fp32(key, 2)\n    if wq.shape[1] != wk.shape[1]:\n        raise ValueError("Ширины проекций одной головы должны совпадать.")\n    a, b = _augment(wq, query_bias), _augment(wk, key_bias)\n    base_a, base_b = _pack(a, len(wq)), _pack(b, len(wk))\n    fallback = base_a, base_b, False\n    a, b = a.astype(np.float64), b.astype(np.float64)\n    try:\n        left, right = _balanced(a, b)\n        limit = float(np.finfo(np.float32).max)\n        if max(np.max(abs(left)), np.max(abs(right))) > limit:\n            return fallback\n        left, right = left.astype(np.float32), right.astype(np.float32)\n        test_a, test_b = _pack(left, len(wq)), _pack(right, len(wk))\n        ar = _core_blocks((a, left, _decoded(base_a), _decoded(test_a)))\n        br = _core_blocks((b, right, _decoded(base_b), _decoded(test_b)))\n    except np.linalg.LinAlgError:\n        return fallback\n\n    ref, folded, old, new = [x @ y.T for x, y in zip(ar, br)]\n    norm = np.linalg.norm\n    size = max(*a.shape, *b.shape, 4 * a.shape[1])\n    magnitude = max(norm(x) * norm(y) for x, y in zip(ar, br))\n    guard = 16 * np.finfo(np.float64).eps * size * magnitude\n    if norm(folded - ref) + guard > 8 * float(np.finfo(np.float32).eps) * norm(ref):\n        return fallback\n    if norm(new - ref) + guard >= norm(old - ref):\n        return fallback\n    return test_a, test_b, True\n', 'algorithms/spectral_rounding.py': '"""Спектральное зависимое округление весовых матриц в симметричный INT8."""\n\nfrom dataclasses import dataclass\nimport numpy as np\n\n\n@dataclass(frozen=True)\nclass Config:\n    """Предел ширины блока, расчётной работы и дополнительной памяти в байтах."""\n    block_cap: int = 12\n    work_limit: int = 150_000_000\n    memory_limit: int = 64 << 20\n\n\n@dataclass\nclass Result:\n    """Коды, масштабы, факт принятия хотя бы одного канала и диагностика."""\n    codes: np.ndarray\n    scales: np.ndarray\n    applied: bool\n    diagnostics: dict\n\n\ndef _matrix(a):\n    if not isinstance(a, np.ndarray) or a.ndim != 2 or a.dtype != np.float32:\n        raise ValueError("Нужна двумерная матрица NumPy в формате FP32.")\n    return a\n\n\ndef _finite(a):\n    chunks = np.nditer(a, flags=["external_loop", "buffered", "zerosize_ok"], buffersize=4096)\n    return all(np.isfinite(v).all() for v in chunks)\n\n\ndef _minmax(w):\n    q = np.empty(w.shape, np.int8)\n    scales = np.empty(w.shape[1], np.float32)\n    limit = float(np.finfo(np.float32).max)\n    upper = np.float32(limit / 127)\n    if float(upper) * 127 > limit:\n        upper = np.nextafter(upper, np.float32(0))\n    lower = np.nextafter(np.float32(0), np.float32(1))\n    for j in range(w.shape[1]):\n        col = w[:, j]\n        peak = max(float(col.max()), -float(col.min()))\n        s = np.float32(np.clip(peak / 127 if peak else 1., lower, upper))\n        scales[j] = s\n        for i in range(0, len(col), 4096):\n            rows = slice(i, i + 4096)\n            q[rows, j] = np.rint(col[rows].astype(float) / s).clip(-127, 127)\n    return q, scales\n\n\ndef _plan(n, d, m, cfg):\n    for b in range(min(cfg.block_cap, d), 1, -1):\n        work = 6*n*d*m + 2*n*d*b + d*m*(b**3 + 12*b)\n        memory = 8*(4*n*b + 8*n + 32*d + 4*d*b + 24*b*b) + 131072\n        if work <= cfg.work_limit and memory <= cfg.memory_limit:\n            return b, work, memory\n    return None\n\n\ndef _geometry(x, b):\n    blocks = []\n    norms = np.zeros(x.shape[1])\n    for i in range(0, len(norms), b):\n        sl = slice(i, min(i+b, len(norms)))\n        part = x[:, sl].astype(float)\n        length = np.linalg.norm(part, axis=0)\n        norms[sl] = length\n        part /= np.where(length > 0, length, 1)\n        gram = part.T @ part\n        np.fill_diagonal(gram, 1.)\n        spectrum = np.linalg.eigvalsh(gram)\n        tol = 64 * np.finfo(float).eps * max(x.shape[0], len(length))\n        blocks.append((sl, gram, int(np.sum(spectrum > 1+tol)), tol))\n    return blocks, norms\n\n\ndef _matvec(x, v, blocks):\n    out = np.zeros(x.shape[0])\n    for sl, _, _, _ in blocks:\n        out += x[:, sl].astype(float) @ v[sl]\n    return out\n\n\ndef _endpoint(v, lo, hi, u):\n    edge = np.where(u > 0, hi, lo)\n    steps = np.full(v.size, np.inf)\n    np.divide(edge-v, u, where=u != 0, out=steps)\n    hit = int(np.argmin(steps))\n    result = np.clip(v + steps[hit]*u, lo, hi)\n    result[hit] = edge[hit]\n    return result\n\n\ndef _round_block(start, lo, hi, gram, grad, coupled, tol):\n    v = start.copy()\n    active = (lo < v) & (v < hi)\n    while active.any():\n        f = np.flatnonzero(active)\n        c = gram[np.ix_(f, f)]\n        u = np.zeros(len(f))\n        u[0] = 1.\n        if coupled and len(f) > 1:\n            vals, basis = np.linalg.eigh(c)\n            if vals[0] < 1-tol:\n                u = basis[:, 0].copy()\n                u[np.abs(u) < 16*np.finfo(float).eps] = 0.\n                if u[np.argmax(np.abs(u))] < 0:\n                    u *= -1\n                if u @ (c @ u-u) > 0:\n                    u.fill(0.)\n                    u[0] = 1.\n        vf, low, high = v[f], lo[f], hi[f]\n        lin = 2*grad[f] + (high-vf) + (low-vf)\n        endpoints = [_endpoint(vf, low, high, direction) for direction in (u, -u)]\n        costs = []\n        for point in endpoints:\n            step = point-vf\n            costs.append(lin @ step + step @ (c @ step-step))\n        new = endpoints[int(costs[1] < costs[0])]\n        grad += gram[:, f] @ (new-vf)\n        v[f] = new\n        active[f] = (low < new) & (new < high)\n    return v\n\n\ndef _column(x, w, base, s, blocks, norms):\n    low_code = np.floor(w / float(s)).clip(-127, 127).astype(np.int16)\n    high_code = np.minimum(low_code+1, 127)\n    lo = np.multiply(low_code, s, dtype=np.float32).astype(float)\n    hi = np.multiply(high_code, s, dtype=np.float32).astype(float)\n    center = np.clip(w, lo, hi)\n    residual = _matvec(x, center-w, blocks)\n    candidate = base.copy()\n    for sl, gram, coupled, tol in blocks:\n        length = norms[sl]\n        part = x[:, sl].astype(float)\n        grad = (part.T @ residual) / np.where(length > 0, length, 1)\n        low = (lo[sl]-w[sl])*length\n        high = (hi[sl]-w[sl])*length\n        start = np.clip(np.zeros_like(low), low, high)\n        end = _round_block(start, low, high, gram, grad, coupled, tol)\n        candidate[sl] = np.where(length == 0, base[sl],\n                                 np.where(end == high, high_code[sl], low_code[sl]))\n        decoded = np.multiply(candidate[sl], s, dtype=np.float32)\n        residual += part @ (decoded.astype(float)-center[sl])\n    def loss(codes):\n        err = _matvec(x, (codes*s).astype(float)-w, blocks)\n        return float(err @ err)\n    return candidate, loss(base), loss(candidate)\n\n\ndef quantize(weights, activations, config=None):\n    """Квантует W[вход, выход] по X[наблюдение, вход]. None включает MinMax.\n\n    Требуются массивы FP32. Неконечные веса и несогласованные формы ошибочны.\n    Пустая или неконечная калибровка даёт MinMax. Бюджеты относятся к ветви\n    спектрального поиска. Входы, обязательный результат и память BLAS отделены.\n    """\n    cfg = Config() if config is None else config\n    if not isinstance(cfg, Config) or any(type(v) is not int or v < 0 for v in vars(cfg).values()):\n        raise ValueError("Пределы ресурсов должны быть целыми неотрицательными числами.")\n    w = _matrix(weights)\n    if not w.size or not _finite(w):\n        raise ValueError("Нужны непустые конечные веса.")\n    q, s = _minmax(w)\n    info = {"reason": "no_calibration", "accepted": 0}\n    result = Result(q, s, False, info)\n    if activations is None:\n        return result\n    x = _matrix(activations)\n    if x.shape[1] != w.shape[0]:\n        raise ValueError("Число признаков не совпадает.")\n    if not x.size or not _finite(x):\n        info["reason"] = "invalid_calibration"\n        return result\n    plan = _plan(len(x), *w.shape, cfg)\n    if plan is None:\n        info["reason"] = "budget_or_width"\n        return result\n    b, work, memory = plan\n    info.update(block=b, work_estimate=work, workspace_estimate=memory)\n    try:\n        blocks, norms = _geometry(x, b)\n        info["protected_ranks"] = [r for _, _, r, _ in blocks]\n        if not any(info["protected_ranks"]):\n            info["reason"] = "flat_blocks"\n            return result\n        for j in range(w.shape[1]):\n            v, old, new = _column(x, w[:, j].astype(float), q[:, j], s[j], blocks, norms)\n            if np.isfinite(new) and new < old*(1-64*np.finfo(float).eps):\n                q[:, j] = v\n                info["accepted"] += 1\n    except np.linalg.LinAlgError:\n        result.codes, result.scales = _minmax(w)\n        info.update(reason="factorization_failed", accepted=0)\n        return result\n    result.applied = info["accepted"] > 0\n    info["reason"] = "accepted" if result.applied else "no_gain"\n    return result\n', 'algorithms/test_activation_int8.py': '"""Минимальные синтетические проверки квантования весов."""\n\nimport unittest\n\nimport numpy as np\n\nfrom activation_int8 import quantize_int8\n\n\nclass QuantizationTests(unittest.TestCase):\n    def test_paired_move_escapes_single_coordinate_minimum(self):\n        w = np.array([[0.25], [0], [127]], dtype=np.float32)\n        for sign in (-1, 1):\n            with self.subTest(sign=sign):\n                x = np.array([[4, sign * 5, 0], [0, 0, 1]], dtype=np.float32)\n                q, s = quantize_int8(x, w, sweeps=1)\n                np.testing.assert_array_equal(q[:, 0], [-1, sign, 127])\n                np.testing.assert_array_equal(s, [1])\n                np.testing.assert_array_equal(x @ (q * s), x @ w)\n\n    def test_least_squares_scale(self):\n        x = np.eye(2, dtype=np.float32)\n        w = np.array([[127], [64.49]], dtype=np.float32)\n        q, s = quantize_int8(x, w, sweeps=1)\n        expected = (127**2 + 64 * float(w[1, 0])) / (127**2 + 64**2)\n        np.testing.assert_array_equal(q[:, 0], [127, 64])\n        np.testing.assert_array_equal(s, [np.float32(expected)])\n\n    def test_minmax_bound_storage_and_repeatability(self):\n        rng = np.random.default_rng(17)\n        x = rng.normal(size=(7, 35)).astype(np.float32)\n        x[:, 1] = x[:, 0]\n        x[:, 2] = 0\n        w = rng.normal(size=(35, 33)).astype(np.float32)\n        w[:, -1] = 0\n        peak = np.max(abs(w.astype(np.float64)), axis=0)\n        base_s = np.where(peak == 0, 1, peak / 127).astype(np.float32)\n        base_q = np.rint(w.astype(np.float64) / base_s).clip(-127, 127).astype(np.int8)\n        q0, s0 = quantize_int8(x, w, sweeps=0)\n        np.testing.assert_array_equal(q0, base_q)\n        np.testing.assert_array_equal(s0, base_s)\n        q, s = quantize_int8(x, w)\n        self.assertEqual((q.shape, s.shape), (w.shape, (33,)))\n        self.assertEqual((q.dtype, s.dtype), (np.int8, np.float32))\n        self.assertTrue(np.all(q >= -127) and np.all(s > 0))\n        self.assertEqual(q.nbytes + s.nbytes, w.size + 4 * w.shape[1])\n        for actual, repeated in zip((q, s), quantize_int8(x, w)):\n            np.testing.assert_array_equal(actual, repeated)\n\n        def loss(codes, scales):\n            diff = w.astype(np.float64) - (codes * scales)\n            return np.sum((x.astype(np.float64) @ diff) ** 2, axis=0)\n\n        self.assertTrue(np.all(loss(q, s) <= loss(base_q, base_s)))\n\n    def test_degenerate_and_extreme_inputs(self):\n        tiny = np.nextafter(np.float32(0), np.float32(1))\n        w = np.array([[tiny, np.finfo(np.float32).max, 0]], dtype=np.float32)\n        with np.errstate(over="raise", invalid="raise", divide="raise"):\n            for value in (0, 1):\n                q, s = quantize_int8([[value]], w)\n                self.assertTrue(np.isfinite(q * s).all() and np.all(s > 0))\n                self.assertEqual((q * s)[0, 0], tiny)\n                self.assertEqual((q[0, 2], s[2]), (0, 1))\n                if value == 0:\n                    q0, s0 = quantize_int8([[0]], w, sweeps=0)\n                    np.testing.assert_array_equal(q, q0)\n                    np.testing.assert_array_equal(s, s0)\n        for x, weights in (([[np.nan]], [[0]]), ([[1, 2]], [[3]])):\n            with self.assertRaises(ValueError):\n                quantize_int8(x, weights)\n\n\nif __name__ == "__main__":\n    unittest.main()\n', 'algorithms/test_adaptive_selector.py': '"""Проверки математической границы, разграничения этапов и лимитов инференса."""\n\nimport unittest\nimport numpy as np\nfrom adaptive_selector import (BASE, Loss, Portfolio, choose, confirm_outputs, finalize,\n                               global_plan, paired_upper, profile, search_priority, select)\n\n\nclass SelectorTests(unittest.TestCase):\n    def test_bound_covers_stress_set(self):\n        rng=np.random.default_rng(73)\n        x,y=rng.normal(size=(2,8,3))\n        hf,hd=x.T@x/8,y.T@y/8\n        ea,eb=rng.normal(size=(2,3,2))\n        def loss(e):\n            return Loss(*(np.trace(e.T@h@e)/2 for h in (hf,hd)),np.sum(e**2)/2)\n        rho=.13\n        l={BASE:loss(eb),"single":loss(ea)}\n        _,diag=choose(l,rho)\n        direction=np.linalg.eigh(ea@ea.T-eb@eb.T)\n        delta=(direction[1]*np.sign(direction[0]))@direction[1].T*rho\n        for t in (-1,0,.5,1):\n            h=hd+t*(hd-hf)+delta\n            diff=(np.trace(ea.T@h@ea)-np.trace(eb.T@h@eb))/2\n            self.assertLessEqual(diff,diag["upper"]["single"]+1e-12)\n        self.assertEqual(choose(l,rho,confidence={"single":1.})[0],BASE)\n        self.assertEqual(paired_upper([-1.,-1.],0.,1),-1.)\n        self.assertGreaterEqual(search_priority(l),0.)\n\n    def test_only_reserved_builder_runs(self):\n        w=np.array([[.25],[0],[127]],np.float32)\n        x=np.array([[4,5,0],[0,0,1]],np.float32)\n        f,diag=profile(w,x)\n        self.assertEqual(diag["sampled"],3)\n        prior=Portfolio([(f[None,:],[[.1,1.,.2]])])\n        def variant(values):\n            return np.array(values,np.int8)[:,None],np.ones(1,np.float32)\n        base=lambda w,x:variant([0,0,127])\n        calls=[]\n        builders={"single":lambda *_:self.fail(),"sdr":lambda *_:self.fail()}\n        def paired(*args):\n            calls.append(1)\n            return variant([-1,1,127])\n        builders["paired"]=paired\n        for budget,expected in ((0.,BASE),(1.,"paired")):\n            name,_,bank,info=select(w,x,x.copy(),base,builders,prior,\n                                    dict.fromkeys(builders,1.),budget)\n            self.assertEqual(name,expected)\n            self.assertEqual(len(bank),1+int(budget))\n        self.assertEqual(calls,[1])\n        builders["paired"]=lambda *_:variant([-128,1,127])\n        self.assertEqual(select(w,x,x.copy(),base,builders,prior,{"paired":1})[0],BASE)\n        self.assertEqual(prior.plan(f+50,{"paired":1},1)[0],())\n        self.assertEqual(Portfolio([]).plan(f,{"paired":1},1)[0],())\n\n    def test_global_budget_and_multioutput_rollback(self):\n        local={"a":"single"}; options={"a":[BASE,"single","sdr"]}\n        def infer(c):\n            values=(0.,0.) if c is None else {BASE:(2.,2.),"single":(1.,3.),"sdr":(.5,1.)}[c["a"]]\n            return {k:np.array([v]) for k,v in zip(("signal","aux"),values)}\n        self.assertEqual(finalize(local,options,{},infer,infer,2)[1]["calls"],0)\n        self.assertEqual(finalize(local,options,{},infer,infer,3)[0]["a"],BASE)\n        probes=[]\n        def propose(anchor,active,probe):\n            for c in (None,anchor,{"a":BASE},{"a":"sdr"}):\n                probes.append(probe(c))\n            return {"a":"sdr"}\n        name,diag=finalize(local,options,{"a":1},infer,infer,8,propose)\n        self.assertEqual((name["a"],diag["calls"],len(probes)),("sdr",8,4))\n        def excessive(anchor,active,probe):\n            for _ in range(20):\n                probe(anchor)\n        self.assertEqual(finalize(local,options,{"a":1},infer,infer,8,excessive)[1]["calls"],7)\n        self.assertEqual(global_plan(options,{"a":1},7),())\n        z=np.array([2**60],np.uint64)\n        a,b=confirm_outputs({"v":z},{"v":z+1},{"v":z})\n        self.assertEqual((a[0],b[0]),(1.,0.))\n\n\nif __name__=="__main__":\n    unittest.main()', 'algorithms/test_logit_int8.py': '"""Синтетические проверки выбора INT8-весов по конечным логитам."""\n\nimport unittest\n\nimport numpy as np\n\nfrom logit_int8 import select_int8\n\n\ndef variant(codes, scale=1 / 64):\n    q = np.array(codes, dtype=np.int8, ndmin=2)\n    return q, np.full(q.shape[1], scale, dtype=np.float32)\n\n\ndef decoded(pair):\n    return pair[0] * pair[1]\n\n\nclass LogitTests(unittest.TestCase):\n    def test_additive_cancellation_and_probe_budget(self):\n        bank = {"a": [variant(65), variant(68)], "b": [variant(64), variant(60)]}\n        fit = np.array([[1], [2]], dtype=np.float32)\n        dev = np.array([[-1], [3]], dtype=np.float32)\n        calls = []\n\n        def infer(weights, x):\n            calls.append("fit" if x is fit else "dev")\n            slope = 2 if weights is None else decoded(weights["a"]) + decoded(weights["b"])\n            return np.column_stack((x * slope, np.zeros_like(x)))\n\n        for _ in range(2):\n            calls.clear()\n            self.assertEqual(select_int8(bank, infer, fit, dev, width=2), {"a": 1, "b": 1})\n            self.assertEqual(calls, ["fit"] * 4 + ["dev"] * 3)\n\n    def test_nonlinear_interaction_rolls_back(self):\n        bank = {"a": [variant(64), variant(80)], "b": [variant(64), variant(50)]}\n        fit = np.array([[1]], dtype=np.float32)\n        dev = np.array([[2]], dtype=np.float32)\n        checked = []\n\n        def infer(weights, x):\n            gain = 66 / 64 if weights is None else decoded(weights["a"]) * decoded(weights["b"])\n            if x is dev and weights is not None:\n                checked.append(float(gain.item()))\n            return np.column_stack((x * gain, np.zeros_like(x)))\n\n        self.assertEqual(select_int8(bank, infer, fit, dev), {"a": 0, "b": 0})\n        self.assertEqual(checked, [1.0, (80 / 64) * (50 / 64)])\n\n    def test_development_rejects_new_flip_despite_lower_mse(self):\n        bank = {"w": [variant([[8], [32]]), variant([[-1], [64]])]}\n        reference = np.array([[0.125], [1]], dtype=np.float32)\n        fit = np.array([[0, 1]], dtype=np.float32)\n        dev = np.eye(2, dtype=np.float32)\n\n        def infer(weights, x):\n            value = x @ (reference if weights is None else decoded(weights["w"]))\n            return np.column_stack((value, np.zeros_like(value)))\n\n        z0, z1 = (infer({"w": pair}, dev) for pair in bank["w"])\n        target = infer(None, dev)\n        self.assertLess(np.mean((z1 - target)**2), np.mean((z0 - target)**2))\n        self.assertEqual(select_int8(bank, infer, fit, dev), {"w": 0})\n\n    def test_multiclass_margin_changes_choice(self):\n        ref = np.array([[0.5, -0.5, 0]], dtype=np.float32)\n        bank = {"w": [variant([64, 0, 32]), variant([16, -32, 12]), variant([44, -20, 12])]}\n        fit = np.array([[1]], dtype=np.float32)\n        dev = np.array([[2]], dtype=np.float32)\n\n        def infer(weights, x):\n            return x @ (ref if weights is None else decoded(weights["w"]))\n\n        first, second = (decoded(pair) for pair in bank["w"][1:])\n        self.assertLess(np.mean((first - ref)**2), np.mean((second - ref)**2))\n        self.assertEqual(select_int8(bank, infer, fit, dev), {"w": 2})\n\n\nif __name__ == "__main__":\n    unittest.main()\n', 'algorithms/test_qk_svd_int8.py': '"""Малые синтетические проверки квантования формы QK."""\n\nimport unittest\n\nimport numpy as np\n\nfrom qk_svd_int8 import _balanced, minmax_int8, quantize_qk_head\n\n\nclass HeadTests(unittest.TestCase):\n    def test_affine_scores_and_bound(self):\n        a = np.array([[1., 3.], [0., 2.], [2., -1.]])\n        b = np.array([[1., -1.], [2., 1.], [0., 3.], [1., 0.]])\n        left, right = _balanced(a, b)\n        x = np.array([[1., -2., 1.], [3., 1., 1.]])\n        y = np.array([[0., 1., 2., 1.], [3., 0., -2., 1.]])\n        np.testing.assert_allclose((x @ left) @ (y @ right).T,\n                                   (x @ a) @ (y @ b).T, rtol=1e-13, atol=1e-13)\n        bound = np.linalg.norm(left, axis=0) @ np.linalg.norm(right, axis=0)\n        self.assertAlmostEqual(bound, np.linalg.svd(a @ b.T, compute_uv=False).sum())\n\n    def test_shear_improves_operator_and_preserves_layout(self):\n        a = np.array([[1, 1024], [0, 2]], np.float32)\n        b = np.array([[1, 0], [-1024, 1]], np.float32)\n        bias = np.zeros(2, np.float32)\n        qa, qb, used = quantize_qk_head(a, b, bias)\n        self.assertTrue(used)\n        self.assertIsNone(qb[2])\n        self.assertEqual(qa[2].dtype, np.float32)\n        for weight, (codes, scales, _) in zip((a, b), (qa, qb)):\n            self.assertEqual((codes.dtype, scales.dtype), (np.int8, np.float32))\n            self.assertEqual((codes.shape, scales.shape), (weight.shape, (2,)))\n            self.assertEqual(codes.nbytes + scales.nbytes, weight.size + 4 * weight.shape[1])\n        qa0, sa0 = minmax_int8(a)\n        qb0, sb0 = minmax_int8(b)\n        target = a.astype(float) @ b.T\n        old = (qa0 * sa0).astype(float) @ (qb0 * sb0).T\n        new = (qa[0] * qa[1]).astype(float) @ (qb[0] * qb[1]).T\n        self.assertGreater(np.linalg.norm(old - target), 2)\n        self.assertLess(np.linalg.norm(new - target), 1e-5)\n        again = quantize_qk_head(a, b, bias)\n        for before, after in zip((qa, qb), again[:2]):\n            for value, repeated in zip(before, after):\n                np.testing.assert_array_equal(value, repeated)\n\n    def test_zero_and_exact_minmax(self):\n        for a in (np.zeros((2, 4), np.float32), np.eye(2, 4, dtype=np.float32)):\n            b = np.eye(3, 4, dtype=np.float32)\n            qa, qb, used = quantize_qk_head(a, b)\n            self.assertFalse(used)\n            for w, (q, s, bias) in zip((a, b), (qa, qb)):\n                np.testing.assert_array_equal(q * s, w)\n                self.assertIsNone(bias)\n                self.assertTrue(np.all(s > 0))\n\n    def test_bias_is_packed_in_original_slots(self):\n        a = np.array([[1, 1024], [0, 2]], np.float32)\n        b = np.array([[1, 0], [-1024, 1]], np.float32)\n        qa, qb, used = quantize_qk_head(a, b, a[1], b[0])\n        self.assertTrue(used)\n        x = np.array([[1., 2.], [-2., 1.]])\n        expected = (x @ a + a[1]) @ (x @ b + b[0]).T\n        actual = (x @ (qa[0] * qa[1]) + qa[2]) @ (x @ (qb[0] * qb[1]) + qb[2]).T\n        np.testing.assert_allclose(actual, expected, atol=1e-5)\n\n\nif __name__ == "__main__":\n    unittest.main()\n', 'algorithms/test_spectral_rounding.py': '"""Синтетические проверки геометрии, защитного механизма и формата INT8."""\nimport unittest\nfrom unittest.mock import patch\nimport numpy as np\nfrom spectral_rounding import Config, _round_block, quantize\n\nclass SpectralTests(unittest.TestCase):\n    def test_potential_and_coordinated_rounding(self):\n        gram = np.ones((6, 6))\n        low, high = np.full(6, -.49), np.full(6, .51)\n        end = _round_block(np.zeros(6), low, high, gram, np.zeros(6), 1, 1e-12)\n        self.assertTrue(np.all((end == low) | (end == high)))\n        self.assertLessEqual(end @ gram @ end, -low @ high + 1e-12)\n        self.assertLess(end @ gram @ end, low @ gram @ low)\n        self.assertGreater(np.count_nonzero(end == high), 1)\n    def test_full_score_and_determinism(self):\n        rng = np.random.default_rng(31)\n        x = rng.normal(size=(5, 22)).astype(np.float32)\n        x[:, 1:4] = x[:, :1]; x[:, -1] = 0\n        w = rng.normal(size=(22, 5)).astype(np.float32); w[:, -1] = 0\n        base = quantize(w, None); new = quantize(w, x)\n        def loss(result):\n            residual = x.astype(float) @ ((result.codes*result.scales).astype(float)-w)\n            return np.sum(residual**2, axis=0)\n        self.assertTrue(np.all(loss(new) <= loss(base)))\n        np.testing.assert_array_equal(new.scales, base.scales)\n        np.testing.assert_array_equal(new.codes[-1], base.codes[-1])\n        self.assertTrue(new.applied)\n        np.testing.assert_array_equal(new.codes, quantize(w, x).codes)\n        self.assertEqual((new.codes.dtype, new.scales.dtype), (np.int8, np.float32))\n        self.assertEqual(new.codes.nbytes+new.scales.nbytes, w.size+4*w.shape[1])\n    def test_resource_and_degeneracy_fallback(self):\n        w = np.eye(3, dtype=np.float32)\n        for x, cfg, reason in ((w, Config(memory_limit=0), "budget_or_width"), (w, Config(work_limit=0), "budget_or_width"), (w, Config(), "flat_blocks"), (w[:0], Config(), "invalid_calibration"), (w*np.nan, Config(), "invalid_calibration")):\n            result = quantize(w, x, cfg)\n            self.assertFalse(result.applied); self.assertEqual(result.diagnostics["reason"], reason)\n            np.testing.assert_array_equal(result.codes, quantize(w, None).codes)\n        with patch("numpy.linalg.eigvalsh", side_effect=np.linalg.LinAlgError):\n            self.assertEqual(quantize(w, w).diagnostics["reason"], "factorization_failed")\n    def test_extreme_weights_and_shape(self):\n        tiny = np.nextafter(np.float32(0), np.float32(1)); big = np.finfo(np.float32).max\n        w = np.array([[big, tiny, 0], [-big/2, -tiny, 0], [1, tiny, 0]], np.float32)\n        x = np.array([[big, big, 0], [tiny, tiny, 0]], np.float32)\n        with np.errstate(invalid="raise", divide="raise", over="raise"):\n            result = quantize(w, x)\n        self.assertTrue(np.isfinite(result.codes*result.scales).all())\n        self.assertTrue((result.scales > 0).all() and result.codes.min() >= -127)\n        for bad in (w.astype(float), w.ravel(), w*np.nan):\n            with self.assertRaises(ValueError): quantize(bad, x)\n        with self.assertRaises(ValueError): quantize(w, x[:, :-1])\n\nif __name__ == "__main__": unittest.main()\n'}
for name, source in SOURCES.items():
    path = ROOT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source, encoding="utf-8")
print("Written", len(SOURCES), "files")

### ONNX-движок, модели, политики и замороженный prior

In [ ]:
SOURCES = {'adaptive_policies.py': '"""Замороженные политики v4; выбор по fit/local-dev, без доступа к test."""\nfrom algorithms.adaptive_selector import BASE, Loss, Portfolio, choose, features, confirm_outputs\nimport numpy as np\n\nVARIANTS={\'minmax\':\'MinMax\',\'single\':\'Single-3\',\'paired\':\'Paired-3\',\'sdr\':\'SDR\'}\nPOLICIES=[\'Single-Dev\',\'Single-Robust\',\'Paired-Robust\',\'SDR-Robust\',\'Astra-Adaptive\',\'Progressive\',\'FullBank-Robust\',\'Progressive-Global\']\n\ndef aggregate(layers):\n    """Объединяет группы оператора с весами по числу выходов и строк."""\n    gs=layers[\'groups\']; mf=np.array([g[\'channels\']*g[\'fit_rows\'] for g in gs],float); md=np.array([g[\'channels\']*g[\'dev_rows\'] for g in gs],float)\n    m=np.array([g[\'channels\'] for g in gs],float)\n    f=float(np.average([g[\'fit_mse\'] for g in gs],weights=mf));d=float(np.average([g[\'dev_mse\'] for g in gs],weights=md))\n    e=float(np.average([g[\'shape\'][0]*g[\'weight_mse\'] for g in gs],weights=m))\n    med=lambda k:float(np.median([g[k] for g in gs]))\n    vec=features(max(g[\'shape\'][0] for g in gs),min(g[\'fit_rows\'] for g in gs),med(\'profile_sampled_features\'),med(\'stable_rank_sample\'),med(\'mean_abs_corr_sample\'),med(\'weight_abs_max\'),med(\'weight_rms\'))\n    return Loss(f,d,e),vec\n\ndef policy(name,losses,vec,prior,target=.25):\n    """Replay банка: политика читает только результаты разрешённых построений.\n    Progressive: Single, затем Paired, затем SDR до робастного выигрыша 25%.\n    Порог заморожен до независимого теста, не оптимизирован по архиву.\n    """\n    seen={BASE:losses[BASE]}; planned=[]; pre=None\n    def add(k):\n        if k in losses and k not in seen:seen[k]=losses[k];planned.append(k)\n    if name==\'Single-Dev\':\n        add(\'single\'); winner=min(seen,key=lambda k:seen[k].dev);info={\'reason\':\'dev_only\'}\n    elif name==\'Astra-Adaptive\':\n        selected,pre=prior.plan(vec,{k:1. for k in (\'single\',\'paired\',\'sdr\') if k in losses},1.)\n        for k in selected:add(k)\n        winner,info=choose(seen)\n    else:\n        if name in (\'Single-Robust\',\'Paired-Robust\',\'SDR-Robust\'):\n            add(name.split(\'-\')[0].lower())\n        elif name==\'FullBank-Robust\':\n            for k in (\'single\',\'paired\',\'sdr\'):add(k)\n        elif name in (\'Progressive\',\'Progressive-Global\'):\n            if seen[BASE].dev>0:\n                for k in (\'single\',\'paired\',\'sdr\'):\n                    add(k); win,diag=choose(seen)\n                    if diag[\'upper\'][win] <= -target*seen[BASE].dev:break\n        else:raise ValueError(name)\n        winner,info=choose(seen)\n    return winner,tuple(planned),dict(info,pre=pre)\n\ndef check(reference,baseline,proposal):\n    old,new=confirm_outputs(reference,baseline,proposal)\n    accept=bool(np.all(new<=old) and np.any(new<old))\n    ratios=[float(n/o) if o>0 else (0. if n==0 else None) for o,n in zip(old,new)]\n    return accept,dict(outputs=sorted(reference),old=old.tolist(),new=new.tolist(),ratios=ratios,accepted=accept)\n', 'benchmark.py': '"""ONNX-бенчмарк с изолированными GPU-воркерами и checkpoint по слоям."""\nimport argparse, hashlib, json, os, sys, time, subprocess, gc, traceback\nfrom pathlib import Path\nfrom dataclasses import dataclass,asdict\nfrom collections import Counter\nfrom unittest.mock import patch\nimport numpy as np\nfrom threadpoolctl import threadpool_limits\nfrom algorithms import activation_int8 as v1, spectral_rounding as sdr, logit_int8 as v2\nfrom graph import discover, with_outputs, sample_rows, export\nfrom models import make_case,custom_case\n\nfrom mixed import EXTRA, mixed_runs\nVERSION=\'4.0.0-kaggle\'\nMETHODS=[\'FP32\',\'MinMax\',\'Single-3\',\'Paired-3\',\'SDR\',\'V2\']+EXTRA\n@dataclass(frozen=True)\nclass Profile:\n    models: tuple\n    seeds: tuple\n    fit: int\n    dev: int\n    test: int\n    rows: int\n    threads: int\n    batch: int\n    sdr_work: int=150_000_000\n    v1_work: int=100_000_000\n    beam: int=4\n    channel_cap: int=0\n    v2_layers: int=0\n    cache_mib: int=128\n    provider: str="cpu"\n    math_backend: str=\'numpy\'\n    capture_group: int=4\nPROFILES={\n \'final_smoke\':Profile((\'efficientnet_b0\',\'minilm_embeddings\',\'multi_output\'),(42,),12,12,16,32,2,4,500_000_000,100_000_000,2,4,2,256,\'cuda\'),\n \'final\':Profile((\'efficientnet_b0\',\'vit_tiny\',\'minilm_embeddings\',\'mlp\',\'regression\',\'autoencoder\',\'multi_output\',\'lstm_head\'),(42,123,731),128,96,512,512,2,8,2_000_000_000,500_000_000,4,128,4,768,\'cuda\'),\n \'final_full\':Profile((\'efficientnet_b0\',\'vit_tiny\',\'minilm_embeddings\',\'mlp\',\'regression\',\'autoencoder\',\'multi_output\',\'lstm_head\'),(42,123,731,2026,91),256,144,1024,1024,2,8,6_000_000_000,2_000_000_000,4,256,4,1024,\'cuda\'),\n \'gpu_smoke\':Profile((\'resnet18\',\'distilbert_sst2\'),(42,),8,8,16,32,2,4,500_000_000,100_000_000,2,4,2,256,\'cuda\'),\n \'research\':Profile((\'resnet18\',\'mobilenet_v2\',\'squeezenet1_1\',\'distilbert_sst2\',\'mlp\',\'regression\',\'autoencoder\',\'multi_output\'),(42,123,731),128,64,512,512,2,8,2_000_000_000,500_000_000,4,128,8,768,\'cuda\'),\n \'research_full\':Profile((\'resnet18\',\'mobilenet_v2\',\'squeezenet1_1\',\'distilbert_sst2\',\'mlp\',\'regression\',\'autoencoder\',\'multi_output\',\'rnn\'),(42,123,731,2026,91),256,128,1024,1024,2,8,6_000_000_000,2_000_000_000,8,256,12,1024,\'cuda\'),\n \'smoke\':Profile((\'cnn\',\'depthwise_cnn\',\'regression\'),(42,),8,8,12,24,1,4),\n \'quick\':Profile((\'bert_tiny\',\'mlp\',\'cnn\',\'regression\'),(42,),64,32,128,256,2,8),\n \'full\':Profile((\'bert_tiny\',\'mlp\',\'cnn\',\'depthwise_cnn\',\'regression\',\'autoencoder\'),(42,123,731),128,64,360,512,2,8),\n \'extended\':Profile((\'bert_tiny\',\'mlp\',\'cnn\',\'depthwise_cnn\',\'regression\',\'autoencoder\',\'multi_output\',\'rnn\'),(42,123,731,2026,91),192,64,872,2048,4,8,350_000_000,200_000_000,8)}\n\ndef atomic(path,obj):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);tmp=path.with_suffix(path.suffix+\'.tmp\')\n    tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2,allow_nan=False),encoding=\'utf-8\');os.replace(tmp,path)\n\ndef hash_feeds(h,feeds):\n    for k,v in sorted(feeds.items()):\n        a=np.ascontiguousarray(v);h.update(k.encode());h.update(str((a.shape,a.dtype)).encode());h.update(memoryview(a).cast(\'B\'))\n\ndef session(model,threads,profiling=False):\n    import onnxruntime as ort\n    opt=ort.SessionOptions();opt.intra_op_num_threads=threads;opt.inter_op_num_threads=1;opt.execution_mode=ort.ExecutionMode.ORT_SEQUENTIAL\n    gpu=os.environ.get(\'QUANT_INFER_PROVIDER\',os.environ.get(\'QUANT_PROVIDER\',\'cpu\'))==\'cuda\'\n    if profiling:opt.enable_profiling=True\n    if gpu:\n        import torch  # Загружает совместимые CUDA/cuDNN библиотеки до ORT.\n        if not torch.cuda.is_available() or \'CUDAExecutionProvider\' not in ort.get_available_providers():\n            raise RuntimeError(\'CUDA недоступна. Проверьте Kaggle GPU / onnxruntime-gpu.\')\n        if hasattr(ort,\'preload_dlls\'):ort.preload_dlls()\n        providers=[(\'CUDAExecutionProvider\',{\'device_id\':0,\'use_tf32\':0,\'cudnn_conv_algo_search\':os.environ.get(\'QUANT_CONV_ALGO\',\'HEURISTIC\'),\'cudnn_conv_use_max_workspace\':1,\'prefer_nhwc\':int(os.environ.get(\'QUANT_NHWC\',\'0\'))}),\'CPUExecutionProvider\']\n    else:providers=[\'CPUExecutionProvider\']\n    sess=ort.InferenceSession(model.SerializeToString(),sess_options=opt,providers=providers)\n    if gpu and sess.get_providers()[0]!=\'CUDAExecutionProvider\':raise RuntimeError(\'ORT silently fell back to CPU\')\n    sess.disable_fallback()\n    return sess\n\ndef batches(feeds,batch):\n    if not feeds or any(v.ndim==0 for v in feeds.values()): raise ValueError(\'Все входы должны иметь общую первую batch-ось.\')\n    lengths={len(v) for v in feeds.values()}\n    if len(lengths)!=1 or not next(iter(lengths)): raise ValueError(\'Несогласованные batch-размеры входов.\')\n    n=next(iter(lengths))\n    for i in range(0,n,batch): yield {k:v[i:i+batch] for k,v in feeds.items()}\n\ndef infer(sess,feeds,batch):\n    names=[o.name for o in sess.get_outputs()]; out={n:[] for n in names}\n    for f in batches(feeds,batch):\n        for n,v in zip(names,sess.run(names,f)):\n            if v.ndim==0 or v.shape[0]!=len(next(iter(f.values()))) or v.dtype.kind not in \'fiu b\'.replace(\' \',\'\'):\n                raise ValueError(f\'Выход {n}: нужен числовой тензор с первой batch-осью.\')\n            if not np.isfinite(v).all(): raise ValueError(f\'Неконечный выход {n}\')\n            out[n].append(v)\n    return {k:np.concatenate(v,axis=0) for k,v in out.items()}\n\ndef capture_many(model, indexed, feeds, cfg, seed, dev=False):\n    """Одна ORT-сессия на группу, независимый reservoir на каждый слой."""\n    names=list(dict.fromkeys(v for _,t in indexed for v in (t.input,t.output)))\n    sess=session(with_outputs(model,names),cfg.threads)\n    states={i:[np.random.default_rng(seed+i*101+int(dev)),None,np.empty(0),0] for i,t in indexed}\n    errors={}\n    for feed in batches(feeds,cfg.batch):\n        outputs=dict(zip(names,sess.run(names,feed)))\n        for i,t in indexed:\n            if i in errors:continue\n            rng,pools,keys,total=states[i]\n            try:\n                rows,n=sample_rows(outputs[t.input],outputs[t.output],t,cfg.rows,rng,feed.get(\'attention_mask\'))\n                if not n:continue\n                k=len(rows[0]);priorities=[];prev=0.\n                for j in range(k):\n                    prev+=(1-prev)*rng.beta(1,n-j);priorities.append(prev)\n                priorities=np.array(priorities);rng.shuffle(priorities)\n                merged=np.concatenate([keys,priorities]);keep=np.argsort(merged,kind=\'stable\')[:cfg.rows]\n                pools=[np.concatenate([a,b],axis=0)[keep] for a,b in zip(pools,rows)] if pools is not None else [a[keep] for a in rows]\n                states[i]=[rng,pools,merged[keep],total+n]\n            except (ValueError,ArithmeticError) as exc:errors[i]=str(exc)\n        del outputs\n    del sess\n    return {i:(v[1],v[3]) for i,v in states.items() if i not in errors and v[1] is not None},errors\n\n\ndef capture(model,target,feeds,cfg,seed):\n    data,errors=capture_many(model,[(0,target)],feeds,cfg,seed)\n    if 0 not in data:raise ValueError(errors.get(0,\'Нет допустимых строк активаций\'))\n    return data[0]\n\n\ndef matrix_profile(w,x):\n    # Диагностика на фиксированной подвыборке признаков, не полный спектр слоя.\n    idx=np.linspace(0,x.shape[1]-1,min(64,x.shape[1]),dtype=int)\n    a=x[:,idx].astype(float);g=a.T@a;ev=np.maximum(np.linalg.eigvalsh(g),0);total=ev.sum()\n    h=np.sqrt(np.diag(g));c=np.divide(g,h[:,None]*h[None,:],out=np.zeros_like(g),where=h[:,None]*h[None,:]>0)\n    np.fill_diagonal(c,0.)\n    return {\'profile_sampled_features\':len(idx),\'stable_rank_sample\':float(total/ev[-1]) if ev[-1]>0 else 0.,\'top_energy_sample\':float(ev[-1]/total) if total>0 else 0.,\'mean_abs_corr_sample\':float(np.abs(c).sum()/max(1,len(idx)*(len(idx)-1))),\'weight_abs_max\':float(np.abs(w).max()),\'weight_rms\':float(np.sqrt(np.mean(w.astype(float)**2)))}\n\ndef local_metrics(w,x,q,s):\n    err=(q*s).astype(np.float64)-w; residual=x.astype(np.float64)@err\n    return float(np.mean(err**2)),float(np.mean(residual**2))\n\ndef _quant_core(w,x,method,cfg):\n    base=v1.quantize_int8(x,w,sweeps=0); info={\'reason\':\'baseline\',\'accepted\':0}\n    if method==\'MinMax\':return base,{**info,\'math_backend\':\'numpy\'}\n    if cfg.math_backend==\'torch\':\n        from cuda_math import v1_torch,sdr_torch\n        device=\'cuda\' if cfg.provider==\'cuda\' else \'cpu\'\n        if method in (\'Single-3\',\'Paired-3\'):\n            if int(x.shape[0])*int(w.size)>cfg.v1_work:return base,{\'reason\':\'budget\',\'accepted\':0,\'math_backend\':\'numpy\'}\n            q,s=v1_torch(x,w,paired=method==\'Paired-3\',device=device)\n            changed=np.any(q!=base[0],axis=0)|(s!=base[1]);return (q,s),{\'reason\':\'accepted\' if changed.any() else \'no_gain\',\'accepted\':int(changed.sum()),\'math_backend\':\'torch_\'+device}\n        if method==\'SDR\':\n            pair,info=sdr_torch(x,w,sdr.Config(work_limit=cfg.sdr_work),device=device)\n            return pair,{**info,\'math_backend\':\'torch_\'+device}\n    if method in (\'Single-3\',\'Paired-3\'):\n        if int(x.shape[0])*int(w.size)>cfg.v1_work:\n            return base,{\'reason\':\'budget\',\'accepted\':0}\n        if method==\'Single-3\':\n            with patch.object(v1,\'_pairs\',return_value=[]): q,s=v1.quantize_int8(x,w,sweeps=3)\n        else:q,s=v1.quantize_int8(x,w,sweeps=3)\n        changed=np.any(q!=base[0],axis=0)|(s!=base[1]);return (q,s),{\'reason\':\'accepted\' if changed.any() else \'no_gain\',\'accepted\':int(changed.sum())}\n    if method==\'SDR\':\n        r=sdr.quantize(w.astype(np.float32,copy=False),x.astype(np.float32,copy=False),sdr.Config(work_limit=cfg.sdr_work))\n        return (r.codes,r.scales),r.diagnostics\n    raise ValueError(method)\n\ndef quant(w,x,method,cfg):\n    # Одинаковый детерминированный набор каналов для всех методов; остальные MinMax.\n    cap=cfg.channel_cap\n    if method==\'MinMax\' or not cap or w.shape[1]<=cap:\n        pair,info=_quant_core(w,x,method,cfg)\n        return pair,{**info,\'attempted_channels\':0 if method==\'MinMax\' or info[\'reason\'] in (\'budget\',\'budget_or_width\',\'flat_blocks\') else w.shape[1],\'channel_cap\':cap}\n    (q,s),_=_quant_core(w,x,\'MinMax\',cfg)\n    idx=np.linspace(0,w.shape[1]-1,cap,dtype=int)\n    (qq,ss),info=_quant_core(w[:,idx],x,method,cfg)\n    q[:,idx]=qq;s[idx]=ss\n    return (q,s),{**info,\'attempted_channels\':0 if info[\'reason\'] in (\'budget\',\'budget_or_width\',\'flat_blocks\') else len(idx),\'channel_cap\':cap,\'selected_channels\':idx.tolist()}\n\ndef fidelity(ref,pred,task,labels):\n    if set(ref)!=set(pred):raise ValueError(\'Набор выходов отличается\')\n    metrics={}; sample=[]\n    for k,r in ref.items():\n        p=pred[k]\n        if r.shape!=p.shape:raise ValueError(f\'Размерность выхода {k} изменилась\')\n        r=r.astype(float);p=p.astype(float); e=p-r; energy=float(np.mean(r*r)); mse=float(np.mean(e*e))\n        sample.append(np.mean(e.reshape(len(e),-1)**2,axis=1))\n        metrics[k]={\'mse\':mse,\'mae\':float(np.mean(abs(e))),\'max_abs\':float(np.max(abs(e))),\'nmse\':mse/energy if energy>1e-20 else None}\n    result={\'outputs\':metrics,\'mse_equal_output_mean\':float(np.mean([m[\'mse\'] for m in metrics.values()])), \'samples\':len(next(iter(ref.values())))}\n    if len(ref)==1:\n        r,p=next(iter(ref.values())),next(iter(pred.values()))\n        if task==\'classification\' and r.ndim==2 and r.shape[1]>=2:\n            a=r.argmax(1);b=p.argmax(1);result.update(agreement=float(np.mean(a==b)),changed=int(np.sum(a!=b)))\n            if labels is not None:\n                from sklearn.metrics import f1_score\n                y=np.asarray(labels).reshape(-1);result[\'top5_accuracy\']=float(np.mean(np.any(np.argsort(p,axis=1)[:,-min(5,p.shape[1]):]==y[:,None],axis=1))) if p.shape[1]>=5 else None;result.update(accuracy=float(np.mean(b==y)),macro_f1=float(f1_score(y,b,average=\'macro\',zero_division=0)),lost_correct=int(np.sum((a==y)&(b!=y))),restored_correct=int(np.sum((a!=y)&(b==y))))\n        if task in (\'regression\',\'reconstruction\') and labels is not None:\n            y=np.asarray(labels).reshape(p.shape);e=p.astype(float)-y;result.update(task_rmse=float(np.sqrt(np.mean(e*e))),task_mae=float(np.mean(abs(e))))\n        if task==\'generic\' and r.ndim==2 and r.shape[1]>1:\n            a=r.astype(float);b=p.astype(float);den=np.linalg.norm(a,axis=1)*np.linalg.norm(b,axis=1)\n            if np.all(den>0):result[\'mean_cosine_to_fp32\']=float(np.mean(np.sum(a*b,axis=1)/den))\n    elif task==\'multi\':\n        if \'logits\' in ref:result[\'logits_agreement\']=float(np.mean(ref[\'logits\'].argmax(1)==pred[\'logits\'].argmax(1)))\n    return result,np.mean(np.stack(sample),axis=0)\n\ndef save_bank(path,bank):\n    data={}\n    for i,(k,(q,s)) in enumerate(bank.items()):data[f\'q{i}\']=q;data[f\'s{i}\']=s\n    data[\'names\']=np.array(list(bank))\n    tmp=path.with_suffix(\'.tmp.npz\');np.savez_compressed(tmp,**data);os.replace(tmp,path)\ndef load_bank(path):\n    with np.load(path,allow_pickle=False) as z:return {str(k):(z[f\'q{i}\'],z[f\'s{i}\']) for i,k in enumerate(z[\'names\'])}\n\ndef benchmark_one(case,cfg,seed,out,methods,save_models):\n    import onnx,onnxruntime as ort\n    h=hashlib.sha256(case.model.SerializeToString())\n    for f in (case.fit,case.dev,case.test,case.shift or {}):hash_feeds(h,f)\n    if case.labels is not None:h.update(np.ascontiguousarray(case.labels).tobytes())\n    if case.shift_labels is not None:h.update(np.ascontiguousarray(case.shift_labels).tobytes())\n    device={\'provider\':cfg.provider,\'inference_provider\':os.environ.get(\'QUANT_INFER_PROVIDER\',cfg.provider),\'conv_algo\':os.environ.get(\'QUANT_CONV_ALGO\',\'HEURISTIC\'),\'nhwc\':os.environ.get(\'QUANT_NHWC\',\'0\'),\'math_backend\':cfg.math_backend}\n    if cfg.provider==\'cuda\':\n        import torch\n        device.update(gpu=torch.cuda.get_device_name(0),torch=torch.__version__,cuda=torch.version.cuda,cudnn=torch.backends.cudnn.version())\n    h.update(json.dumps(device,sort_keys=True).encode());h.update(json.dumps(asdict(cfg),sort_keys=True).encode());h.update(str((VERSION,np.__version__,onnx.__version__,ort.__version__,save_models,case.task)).encode())\n    for p in sorted(Path(__file__).parent.glob(\'*.py\'))+sorted((Path(__file__).parent/\'algorithms\').glob(\'*.py\')):h.update(p.read_bytes())\n    h.update((Path(__file__).parent/\'prior.json\').read_bytes());key=h.hexdigest()[:16];root=out/case.name/f\'seed_{seed}\'/key;root.mkdir(parents=True,exist_ok=True)\n    atomic(root/\'manifest.json\',{\'model\':case.name,\'seed\':seed,\'task\':case.task,\'historical_test\':case.historical,\'note\':case.note,\'config\':asdict(cfg),\'fingerprint\':key,\'versions\':{\'numpy\':np.__version__,\'onnx\':onnx.__version__,\'ort\':ort.__version__},\'provider_requested\':cfg.provider,\'device\':device,\'provenance\':getattr(case,\'provenance\',{}),\'counts\':{k:len(next(iter(v.values()))) for k,v in [(\'fit\',case.fit),(\'dev\',case.dev),(\'test\',case.test)]}})\n    if all((root/(m+\'.json\')).exists() for m in methods): print(f\'CACHE {case.name} seed={seed}\',flush=True);return root\n    start=time.perf_counter();onnx.checker.check_model(case.model);base_sess=session(case.model,cfg.threads)\n    # Local selection, global search, final check use disjoint subsets of original dev.\n    ndev=len(next(iter(case.dev.values())))\n    if ndev<3:raise ValueError(\'Need at least 3 development examples for v4 splits\')\n    a=max(1,ndev//2);b=a+max(1,(ndev-a)//2)\n    local_feeds={k:v[:a] for k,v in case.dev.items()};search_feeds={k:v[a:b] for k,v in case.dev.items()};check_feeds={k:v[b:] for k,v in case.dev.items()}\n    atomic(root/\'splits_v4.json\',{\'fit\':len(next(iter(case.fit.values()))),\'local_dev\':a,\'global_search\':b-a,\'global_check\':ndev-b,\'test\':len(next(iter(case.test.values()))),\'note\':\'Original dev split by position; all selectors frozen before test metrics are used. No test values used in selection.\'})\n    ref=infer(base_sess,case.test,cfg.batch);ref_dev=infer(base_sess,search_feeds,cfg.batch);ref_shift=infer(base_sess,case.shift,cfg.batch) if case.shift else None\n    fit_ref=None\n    def evaluate(method,model,prep,layer_data,extra):\n        reuse=extra.pop(\'reuse_from\',None)\n        if reuse and (root/(reuse+\'.json\')).exists() and (root/(reuse+\'.mse.npy\')).exists():\n            import shutil\n            source=json.loads((root/(reuse+\'.json\')).read_text())\n            # Only inference metrics/latency are reused for byte-identical candidate selections.\n            keys=(\'metrics\',\'development\',\'shift\',\'model_bytes\',\'initializer_bytes\',\'nodes\',\'dq_nodes\',\'median_ms\',\'p95_ms\',\'latency_batch\',\'providers\',\'inference_provider\',\'math_backend_requested\',\'latency_warmup\',\'latency_repetitions\')\n            result={k:source[k] for k in keys};result.update(method=method,model=case.name,seed=seed,status=\'ok\',historical_test=case.historical,prepare_seconds=prep,calibration_seconds=extra.pop(\'calibration_seconds\',0.),layers=layer_data,measurement_reused_from=reuse,**extra)\n            if save_models:onnx.save_model(model,str(root/(method+\'.onnx\')))\n            shutil.copyfile(root/(reuse+\'.mse.npy\'),root/(method+\'.mse.npy\'));atomic(root/(method+\'.json\'),result)\n            print(f\'{case.name} seed={seed} {method}: MSE={result["metrics"]["mse_equal_output_mean"]:.6g} prep={prep:.2f}s (identical selection: {reuse})\',flush=True);return\n        sess=base_sess if method==\'FP32\' else session(model,cfg.threads)\n        pred=infer(sess,case.test,cfg.batch);dev=infer(sess,search_feeds,cfg.batch)\n        metrics,per=fidelity(ref,pred,case.task,case.labels);dev_metrics,_=fidelity(ref_dev,dev,case.task,None)\n        shift_metrics=None\n        if case.shift:shift_metrics,_=fidelity(ref_shift,infer(sess,case.shift,cfg.batch),case.task,case.shift_labels)\n        sample=next(batches(case.test,cfg.batch));times=[]\n        for i in range(25):\n            t=time.perf_counter();sess.run(None,sample);dt=(time.perf_counter()-t)*1000\n            if i>=5:times.append(dt)\n        if save_models:onnx.save_model(model,str(root/(method+\'.onnx\')))\n        # Per-example MSE позволяет вычислять парное сравнение, без повторного инференса.\n        np.save(root/(method+\'.mse.npy\'),per)\n        result={\'method\':method,\'model\':case.name,\'seed\':seed,\'status\':\'ok\',\'historical_test\':case.historical,\'metrics\':metrics,\'development\':dev_metrics,\'shift\':shift_metrics,\'prepare_seconds\':prep,\'calibration_seconds\':extra.pop(\'calibration_seconds\',0.),\'model_bytes\':len(model.SerializeToString()),\'initializer_bytes\':sum(t.raw_data.__len__() or len(t.SerializeToString()) for t in model.graph.initializer),\'nodes\':len(model.graph.node),\'dq_nodes\':sum(n.op_type==\'DequantizeLinear\' for n in model.graph.node),\'median_ms\':float(np.median(times)),\'p95_ms\':float(np.percentile(times,95)),\'latency_batch\':len(next(iter(sample.values()))),\'providers\':sess.get_providers(),\'inference_provider\':os.environ.get(\'QUANT_INFER_PROVIDER\',cfg.provider),\'math_backend_requested\':cfg.math_backend,\'latency_warmup\':5,\'latency_repetitions\':20,\'layers\':layer_data,**extra}\n        atomic(root/(method+\'.json\'),result);print(f\'{case.name} seed={seed} {method}: MSE={metrics["mse_equal_output_mean"]:.6g} prep={prep:.2f}s\',flush=True)\n        if method!=\'FP32\':del sess\n    if \'FP32\' in methods and not (root/\'FP32.json\').exists():evaluate(\'FP32\',case.model,0,[],{})\n    if set(methods)=={\'FP32\'}:return root\n    start=time.perf_counter();targets,skipped=discover(case.model);data={};valid=[]\n    # Группируем выходы, Fit и Dev используют одну сессию на группу каждый.\n    for start_i in range(0,len(targets),cfg.capture_group):\n        indexed=list(enumerate(targets[start_i:start_i+cfg.capture_group],start_i))\n        pending=[(i,t) for i,t in indexed if not (root/f\'activation_{i:04d}.npz\').exists()]\n        if pending:\n            print(f\'  capture {start_i+1}–{start_i+len(indexed)}/{len(targets)}\',flush=True)\n            try:\n                fit,ef=capture_many(case.model,pending,case.fit,cfg,seed)\n                dev,ed=capture_many(case.model,pending,local_feeds,cfg,seed,True)\n            except Exception as exc:\n                # Изолируем неисправный адаптер; ошибки CUDA не скрываем.\n                print(\'  group capture failed, isolating:\',str(exc)[:180],flush=True)\n                fit={};dev={};ef={};ed={}\n                for i,t in pending:\n                    try:\n                        f,_=capture_many(case.model,[(i,t)],case.fit,cfg,seed);d,_=capture_many(case.model,[(i,t)],local_feeds,cfg,seed,True);fit.update(f);dev.update(d)\n                    except Exception as exc:\n                        if os.environ.get(\'QUANT_INFER_PROVIDER\')==\'cuda\' and any(k in str(exc).lower() for k in (\'cudnn\',\'ep_fail\',\'cudaexecutionprovider\')):raise\n                        ef[i]=str(exc)\n            for i,t in pending:\n                if i not in fit or i not in dev:\n                    skipped.append({\'node\':t.node,\'weight\':t.weight,\'reason\':ef.get(i,ed.get(i,\'no_valid_rows\'))[:400]});continue\n                x,n=fit[i];d,nd=dev[i];ap=root/f\'activation_{i:04d}.npz\';tmp=ap.with_suffix(\'.tmp.npz\')\n                np.savez(tmp,counts=np.array([n,nd]),**{f\'x{j}\':v for j,v in enumerate(x)},**{f\'d{j}\':v for j,v in enumerate(d)});os.replace(tmp,ap)\n            del fit,dev\n        for i,t in indexed:\n            ap=root/f\'activation_{i:04d}.npz\'\n            if not ap.exists():continue\n            # Активации загружаются с диска по слою, не отбрасываем большие модели по размеру RAM.\n            data[t.weight]=ap;valid.append(t)\n            print(f\'  {t.op} {t.node}: cached\',flush=True)\n    cal_time=time.perf_counter()-start\n    atomic(root/\'coverage.json\',{\'eligible\':len(targets),\'validated\':len(valid),\'graph_operator_counts\':dict(Counter(n.op_type for n in case.model.graph.node)),\'eligible_parameters\':sum(t.original.size for t in targets),\'fp32_initializer_parameters\':sum(int(np.prod(t.dims)) for t in case.model.graph.initializer if t.data_type==onnx.TensorProto.FLOAT),\'skipped\':skipped,\'quantized_parameters\':sum(t.original.size for t in valid),\'calibration_seconds\':cal_time})\n    if not valid:\n        for m in methods:\n            if m!=\'FP32\':atomic(root/(m+\'.json\'),{\'model\':case.name,\'seed\':seed,\'method\':m,\'status\':\'skipped\',\'reason\':\'no_validated_targets\'})\n        return root\n    banks={}\n    required=set(methods)\n    v2_eligible=case.task==\'classification\' and len(ref)==1 and next(iter(ref.values())).ndim==2 and next(iter(ref.values())).shape[1]>=2\n    if any(m in EXTRA for m in methods):required.update([\'MinMax\',\'Single-3\',\'Paired-3\',\'SDR\'])\n    if \'V2\' in methods and v2_eligible:required.update([\'MinMax\',\'Single-3\',\'Paired-3\',\'SDR\'])\n    for method in (\'MinMax\',\'Single-3\',\'Paired-3\',\'SDR\'):\n        if method not in required:continue\n        bp=root/(method+\'.bank.npz\');jp=root/(method+\'.json\')\n        if bp.exists() and jp.exists():banks[method]=load_bank(bp);continue\n        bank={};layers=[];t0=time.perf_counter()\n        for li,t in enumerate(valid):\n            print(f\'    {method} {li+1}/{len(valid)} {t.node}\',flush=True)\n            layer_started=time.perf_counter()\n            cp=root/f\'{method}.layer_{li:04d}.npz\';cj=cp.with_suffix(\'.json\')\n            if cp.exists() and cj.exists():\n                with np.load(cp,allow_pickle=False) as z:bank[t.weight]=(z[\'q\'],z[\'s\'])\n                layers.append(json.loads(cj.read_text(encoding=\'utf-8\')));continue\n            with np.load(data[t.weight],allow_pickle=False) as z:\n                n,nd=map(int,z[\'counts\']);xs=[z[f\'x{j}\'] for j in range(t.groups)];ds=[z[f\'d{j}\'] for j in range(t.groups)]\n            pairs=[];details=[]\n            for group,(w,x,d) in enumerate(zip(t.matrices(),xs,ds)):\n                try:pair,info=quant(w,x,method,cfg)\n                except (ValueError,ArithmeticError,np.linalg.LinAlgError,RuntimeError) as exc:\n                    pair=v1.quantize_int8(x,w,sweeps=0);info={\'reason\':\'algorithm_error\',\'error\':str(exc)[:400],\'accepted\':0,\'math_backend\':\'numpy_fallback\',\'attempted_channels\':0}\n                q,s=pair;wm,fm=local_metrics(w,x,q,s);_,dm=local_metrics(w,d,q,s)\n                details.append({\'group\':group,\'shape\':list(w.shape),\'fit_rows\':len(x),\'dev_rows\':len(d),\'fit_available\':n,\'dev_available\':nd,\'weight_mse\':wm,\'fit_mse\':fm,\'dev_mse\':dm,\'channels\':w.shape[1],**matrix_profile(w,x),**info});pairs.append(pair)\n            q,s=t.pack(pairs);bank[t.weight]=t.canonical(q,s);layer={\'node\':t.node,\'weight\':t.weight,\'operator\':t.op,\'axis\':t.axis,\'groups\':details,\'layer_wall_seconds\':time.perf_counter()-layer_started};layers.append(layer)\n            tmp=cp.with_suffix(\'.tmp.npz\');np.savez(tmp,q=bank[t.weight][0],s=bank[t.weight][1]);os.replace(tmp,cp);atomic(cj,layer)\n        prep=sum(l.get(\'layer_wall_seconds\',0.) for l in layers);model=export(case.model,valid,bank);save_bank(bp,bank);banks[method]=bank\n        evaluate(method,model,prep,layers,{\'calibration_seconds\':cal_time,\'payload_bytes\':sum(q.nbytes+s.nbytes for q,s in bank.values()),\'quantized_parameters\':sum(q.size for q,s in bank.values())});del model;gc.collect()\n    if \'V2\' in methods and not (root/\'V2.json\').exists():\n        if not v2_eligible:\n            atomic(root/\'V2.json\',{\'model\':case.name,\'seed\':seed,\'method\':\'V2\',\'status\':\'skipped\',\'reason\':\'original V2 requires one classification logits tensor\'})\n        else:\n            variant_names=[\'MinMax\',\'Single-3\',\'Paired-3\',\'SDR\']\n            scores={m:json.loads((root/(m+\'.json\')).read_text(encoding=\'utf-8\'))[\'development\'][\'mse_equal_output_mean\'] for m in variant_names}\n            anchor=min(variant_names,key=lambda m:scores[m]);variant_names=[anchor]+[m for m in variant_names if m!=anchor]\n            bank={t.weight:[banks[m][t.weight] for m in variant_names] for t in valid}\n            # Неизменённый V2 на ограниченном банке: остальные тензоры фиксированы на лучшем по dev MSE anchor.\n            variable=[t for t in valid if any(not (np.array_equal(bank[t.weight][0][0],p[0]) and np.array_equal(bank[t.weight][0][1],p[1])) for p in bank[t.weight][1:])]\n            active={t.weight for t in sorted(variable,key=lambda t:(-t.original.size,t.weight))[:cfg.v2_layers]} if cfg.v2_layers else {t.weight for t in variable}\n            bank={k:(v if k in active else v[:1]) for k,v in bank.items()}\n            calls=0; t0=time.perf_counter()\n            def callback(weights,feeds):\n                nonlocal calls\n                calls+=1\n                p=case.model if weights is None else export(case.model,valid,weights)\n                s=session(p,cfg.threads);z=next(iter(infer(s,feeds,cfg.batch).values()));del s;return z\n            indices=v2.select_int8(bank,callback,case.fit,search_feeds,width=cfg.beam)\n            chosen={k:bank[k][i] for k,i in indices.items()};prep=time.perf_counter()-t0\n            p=export(case.model,valid,chosen);save_bank(root/\'V2.bank.npz\',chosen)\n            evaluate(\'V2\',p,prep,[],{\'calibration_seconds\':cal_time,\'candidate_bank\':variant_names,\'anchor\':anchor,\'anchor_dev_mse\':scores,\'v2_variable_layers\':sorted(active),\'v2_layer_cap\':cfg.v2_layers,\'variant\':\'budgeted V2\' if cfg.v2_layers else \'V2\',\'selection\':{k:variant_names[i] for k,i in indices.items()},\'forward_dataset_calls\':calls,\'payload_bytes\':sum(q.nbytes+s.nbytes for q,s in chosen.values()),\'quantized_parameters\':sum(q.size for q,s in chosen.values()),\'bank_build_seconds\':sum(json.loads((root/(m+\'.json\')).read_text(encoding=\'utf-8\'))[\'prepare_seconds\'] for m in variant_names)})\n    mixed_runs(case,cfg,root,valid,banks,[m for m in methods if m in EXTRA],evaluate,search_feeds,check_feeds,base_sess,cal_time)\n    return root\n\n\ndef summarize(out):\n    import csv\n    all_runs=[]; manifests=[]\n    # Только последний fingerprint каждого model/seed, чтобы не смешивать настройки.\n    for seed_dir in sorted(out.glob(\'*/seed_*\')):\n        roots=[p for p in seed_dir.iterdir() if (p/\'manifest.json\').is_file()]\n        if not roots:continue\n        root=max(roots,key=lambda p:(p/\'manifest.json\').stat().st_mtime)\n        manifest=json.loads((root/\'manifest.json\').read_text(encoding=\'utf-8\'))\n        rows=[json.loads((root/(m+\'.json\')).read_text(encoding=\'utf-8\')) for m in METHODS if (root/(m+\'.json\')).is_file()]\n        base=next((r for r in rows if r[\'method\']==\'MinMax\' and r[\'status\']==\'ok\'),None)\n        for r in rows:\n            if base and r[\'status\']==\'ok\':\n                b=base[\'metrics\'][\'mse_equal_output_mean\'];r[\'ratio_to_minmax\']=r[\'metrics\'][\'mse_equal_output_mean\']/b if b else None\n                if r[\'method\'] not in (\'FP32\',\'MinMax\') and (root/(r[\'method\']+\'.mse.npy\')).exists():\n                    diff=np.load(root/(r[\'method\']+\'.mse.npy\'))-np.load(root/\'MinMax.mse.npy\');rng=np.random.default_rng(42)\n                    means=np.array([rng.choice(diff,len(diff),replace=True).mean() for _ in range(500)])\n                    r[\'paired_bootstrap_delta_ci95\']=[float(i) for i in np.percentile(means,[2.5,97.5])]\n            all_runs.append(r)\n        manifest[\'coverage\']=json.loads((root/\'coverage.json\').read_text(encoding=\'utf-8\')) if (root/\'coverage.json\').exists() else None\n        manifest[\'process_stats\']=json.loads((root/\'process_stats.json\').read_text(encoding=\'utf-8\')) if (root/\'process_stats.json\').exists() else None\n        manifests.append(manifest)\n    atomic(out/\'research_report.json\',{\'version\':VERSION,\'runs\':all_runs,\'manifests\':manifests,\'note\':\'Test-выборки между seed фиксированы. Bootstrap внутри одного seed, не независимый CI по всем seed; BERT test исторический.\'})\n    flat=[]\n    for r in all_runs:\n        if r[\'status\']!=\'ok\':flat.append({k:r.get(k) for k in (\'model\',\'seed\',\'method\',\'status\',\'reason\')});continue\n        groups=[g for l in r[\'layers\'] for g in l[\'groups\']];channels=sum(g[\'channels\'] for g in groups)\n        m=r[\'metrics\'];flat.append({\'model\':r[\'model\'],\'seed\':r[\'seed\'],\'method\':r[\'method\'],\'status\':\'ok\',\'mse\':m[\'mse_equal_output_mean\'],\'ratio_to_minmax\':r.get(\'ratio_to_minmax\'),\'accuracy\':m.get(\'accuracy\'),\'agreement\':m.get(\'agreement\'),\'task_rmse\':m.get(\'task_rmse\'),\'shift_mse\':(r.get(\'shift\') or {}).get(\'mse_equal_output_mean\'),\'prepare_s\':r[\'prepare_seconds\'],\'median_ms\':r[\'median_ms\'],\'model_bytes\':r[\'model_bytes\'],\'channels_applied\':sum(g.get(\'accepted\',0) for g in groups),\'channels_total\':channels,\'channels_attempted\':sum(g.get(\'attempted_channels\',0) for g in groups),\'fallback_groups\':sum(g.get(\'accepted\',0)==0 for g in groups),\'historical_test\':r[\'historical_test\'],\'policy_build_seconds_estimate\':r.get(\'policy_build_seconds_estimate\'),\'policy_replay_seconds\':r.get(\'policy_replay_seconds\'),\'expensive_builds\':r.get(\'expensive_builds\'),\'fallback_to_all_minmax\':r.get(\'fallback_to_all_minmax\')})\n    cols=list(dict.fromkeys(k for row in flat for k in row));out.mkdir(parents=True,exist_ok=True)\n    with (out/\'summary.csv\').open(\'w\',encoding=\'utf-8-sig\',newline=\'\') as f:\n        if cols:w=csv.DictWriter(f,fieldnames=cols);w.writeheader();w.writerows(flat)\n    aggregates=[]\n    for model,method in sorted({(r[\'model\'],r[\'method\']) for r in flat if r[\'status\']==\'ok\'}):\n        selected=[r for r in flat if r[\'model\']==model and r[\'method\']==method and r[\'status\']==\'ok\']\n        ratios=[r[\'ratio_to_minmax\'] for r in selected if r.get(\'ratio_to_minmax\') is not None]\n        aggregates.append({\'model\':model,\'method\':method,\'seeds\':len(selected),\'median_ratio_to_minmax\':float(np.median(ratios)) if ratios else None,\'median_prepare_s\':float(np.median([r[\'prepare_s\'] for r in selected]))})\n    atomic(out/\'aggregate.json\',aggregates)\n    print(\'Report:\',out/\'research_report.json\',flush=True)\n\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--provider\',choices=[\'cpu\',\'cuda\']);p.add_argument(\'--math-backend\',choices=[\'numpy\',\'torch\'],default=\'torch\');p.add_argument(\'--capture-group\',type=int,default=4);p.add_argument(\'--gpus\',type=int,default=2);p.add_argument(\'--conv-cpu-fallback\',action=\'store_true\');p.add_argument(\'--channel-cap\',type=int);p.add_argument(\'--v2-layers\',type=int);p.add_argument(\'--profile\',choices=PROFILES,default=\'quick\');p.add_argument(\'--models\',nargs=\'+\');p.add_argument(\'--seeds\',nargs=\'+\',type=int);p.add_argument(\'--methods\',nargs=\'+\',choices=METHODS,default=METHODS);p.add_argument(\'--archive\');p.add_argument(\'--output\');p.add_argument(\'--threads\',type=int);p.add_argument(\'--rows\',type=int);p.add_argument(\'--sdr-work\',type=int);p.add_argument(\'--save-models\',action=\'store_true\');p.add_argument(\'--worker\',action=\'store_true\');p.add_argument(\'--custom-model\');p.add_argument(\'--fit-npz\');p.add_argument(\'--dev-npz\');p.add_argument(\'--test-npz\');p.add_argument(\'--shift-npz\');p.add_argument(\'--task\',choices=[\'generic\',\'classification\',\'regression\',\'reconstruction\',\'multi\'],default=\'generic\');a=p.parse_args()\n    from dataclasses import replace\n    cfg=PROFILES[a.profile];cfg=replace(cfg,threads=a.threads or cfg.threads,rows=a.rows or cfg.rows,sdr_work=a.sdr_work if a.sdr_work is not None else cfg.sdr_work)\n    cfg=replace(cfg,math_backend=a.math_backend,capture_group=a.capture_group,provider=a.provider or cfg.provider,channel_cap=a.channel_cap if a.channel_cap is not None else cfg.channel_cap,v2_layers=a.v2_layers if a.v2_layers is not None else cfg.v2_layers)\n    if min(cfg.channel_cap,cfg.v2_layers)<0:p.error(\'negative cap\')\n    os.environ[\'QUANT_PROVIDER\']=cfg.provider\n    if cfg.rows<1 or cfg.threads<1 or cfg.sdr_work<0 or a.capture_group<1 or a.gpus<1:p.error(\'Недопустимый бюджет\')\n    root=Path(__file__).resolve().parent;out=Path(a.output).resolve() if a.output else root/\'results\'/a.profile;out.mkdir(parents=True,exist_ok=True)\n    archive=a.archive\n    if archive is None:\n        archive=next((str(z) for z in [root/\'onnx-quant-lab-handoff-v0.5.zip\',root.parent/\'onnx-quant-lab-handoff-v0.5.zip\',root/\'onnx-quant-lab\',root.parent/\'onnx-quant-lab\'] if z.exists()),None)\n    models=[\'custom\'] if a.custom_model else (a.models or cfg.models);seeds=a.seeds or cfg.seeds\n    if a.custom_model and not all([a.fit_npz,a.dev_npz,a.test_npz]):p.error(\'custom требует fit/dev/test NPZ\')\n    if a.worker:\n        name=models[0];seed=seeds[0]\n        if cfg.provider==\'cuda\':\n            from gpu_preflight import configure\n            configure(name,a.conv_cpu_fallback)\n            print(\'Inference:\',os.environ.get(\'QUANT_INFER_PROVIDER\'), \'Math:\',cfg.math_backend, \'GPU:\',os.environ.get(\'CUDA_VISIBLE_DEVICES\'),flush=True)\n        import threading\n        import psutil\n        stop=threading.Event(); proc=psutil.Process(); peak=[proc.memory_info().rss]; t0=time.perf_counter()\n        def monitor():\n            while not stop.wait(.1):\n                peak[0]=max(peak[0],proc.memory_info().rss)\n        thread=threading.Thread(target=monitor,daemon=True);thread.start()\n        try:\n            with threadpool_limits(cfg.threads):\n                from pretrained import VISION,NLP,make_pretrained\n                from independent import VISION_NEW,NLP_NEW,build,lstm_case\n                def load_case():\n                    if a.custom_model:return custom_case(a.custom_model,a.fit_npz,a.dev_npz,a.test_npz,a.task,a.shift_npz)\n                    if name in VISION+NLP+VISION_NEW+NLP_NEW+(\'lstm_head\',):\n                        import fcntl\n                        # Одна загрузка/экспорт общих файлов одновременно; квантование параллельно.\n                        with (root/\'pretrained_cache.lock\').open(\'a\') as lock:\n                            fcntl.flock(lock,fcntl.LOCK_EX)\n                            if name in VISION_NEW+NLP_NEW:return build(name,seed,cfg,root)\n                            if name==\'lstm_head\':return lstm_case(seed,cfg,root)\n                            return make_pretrained(name,seed,cfg,root)\n                    return make_case(name,seed,cfg,root,archive)\n                try:\n                    case=load_case();run_root=benchmark_one(case,cfg,seed,out,a.methods,a.save_models)\n                except Exception as exc:\n                    if not (a.conv_cpu_fallback and name in VISION+VISION_NEW+(\'cnn\',\'depthwise_cnn\',\'custom\') and os.environ.get(\'QUANT_INFER_PROVIDER\')==\'cuda\' and any(k in str(exc).lower() for k in (\'cudnn\',\'ep_fail\',\'cudaexecutionprovider\'))):raise\n                    os.environ[\'QUANT_INFER_PROVIDER\']=\'cpu\'\n                    print(\'EXPLICIT FALLBACK: actual CNN CUDA error; restarting with CPU inference and CUDA math:\',str(exc)[:500],flush=True)\n                    case=load_case();run_root=benchmark_one(case,cfg,seed,out,a.methods,a.save_models)\n            if not a.save_models and all((run_root/(m+\'.json\')).exists() for m in a.methods):\n                for pattern in (\'activation_*.npz\',\'*.bank.npz\',\'*.layer_*.npz\'):\n                    for artifact in run_root.glob(pattern):artifact.unlink(missing_ok=True)\n            if not (run_root/\'process_stats.json\').exists():\n                atomic(run_root/\'process_stats.json\',{\'sampled_peak_rss_mib\':peak[0]/2**20,\'worker_wall_seconds\':time.perf_counter()-t0,\'rss_sample_interval_seconds\':.1})\n        finally:\n            stop.set();thread.join()\n        return\n    # Отдельный процесс на физический GPU, один experiment/worker. Кэш загрузок защищён flock.\n    from concurrent.futures import ThreadPoolExecutor,as_completed\n    import queue\n    if cfg.provider==\'cuda\':\n        import torch\n        devices=list(range(min(a.gpus,torch.cuda.device_count())))\n        if not devices:raise RuntimeError(\'No CUDA devices\')\n        visible=os.environ.get(\'CUDA_VISIBLE_DEVICES\')\n        if visible and len(visible.split(\',\'))>=len(devices):devices=[visible.split(\',\')[i] for i in devices]\n    else:devices=[None]\n    gpu_configs={}\n    if cfg.provider==\'cuda\':\n        for device in devices:\n            env={**os.environ,\'CUDA_VISIBLE_DEVICES\':str(device)}\n            proc=subprocess.run([sys.executable,\'-u\',str(root/\'gpu_preflight.py\')],env=env,capture_output=True,text=True)\n            log=out/f\'gpu_preflight_{device}.log\';log.write_text(proc.stdout+\'\\n\'+proc.stderr,encoding=\'utf-8\')\n            if proc.returncode:raise RuntimeError(f\'GPU {device} preflight failed: {log}\')\n            gpu_configs[device]=next(line[len(\'GPU_PROBE \'):] for line in proc.stdout.splitlines() if line.startswith(\'GPU_PROBE \'))\n            print(\'GPU\',device,gpu_configs[device],flush=True)\n    slots=queue.Queue()\n    for device in devices:slots.put(device)\n    failures=[]\n    def run_job(name,seed):\n        device=slots.get()\n        try:\n            args=[sys.executable,\'-u\',__file__,\'--worker\',\'--profile\',a.profile,\'--models\',name,\'--seeds\',str(seed),\'--methods\',*a.methods,\'--output\',str(out),\'--threads\',str(cfg.threads),\'--rows\',str(cfg.rows),\'--sdr-work\',str(cfg.sdr_work),\'--provider\',cfg.provider,\'--channel-cap\',str(cfg.channel_cap),\'--v2-layers\',str(cfg.v2_layers),\'--math-backend\',cfg.math_backend,\'--capture-group\',str(cfg.capture_group)]\n            if a.conv_cpu_fallback:args+=[\'--conv-cpu-fallback\']\n            if archive:args+=[\'--archive\',archive]\n            if a.save_models:args+=[\'--save-models\']\n            if a.custom_model:\n                for arg,val in [(\'custom-model\',a.custom_model),(\'fit-npz\',a.fit_npz),(\'dev-npz\',a.dev_npz),(\'test-npz\',a.test_npz),(\'shift-npz\',a.shift_npz),(\'task\',a.task)]:\n                    if val:args+=[\'--\'+arg,val]\n            print(f\'\\n=== {name}, seed={seed}, GPU={device} ===\',flush=True)\n            env={**os.environ,\'PYTHONUTF8\':\'1\',**{k:str(cfg.threads) for k in (\'OPENBLAS_NUM_THREADS\',\'MKL_NUM_THREADS\',\'OMP_NUM_THREADS\',\'BLIS_NUM_THREADS\')}}\n            if device is not None:env.update(CUDA_VISIBLE_DEVICES=str(device),QUANT_GPU_CONFIG=gpu_configs[device])\n            log=out/\'logs\'/f\'{name}_{seed}.log\';log.parent.mkdir(exist_ok=True)\n            with log.open(\'w\',encoding=\'utf-8\') as f:\n                proc=subprocess.Popen(args,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)\n                for line in proc.stdout:\n                    f.write(line);f.flush()\n                    if (\' MSE=\' in line or line.startswith(\'    policy\') or \'EXPLICIT FALLBACK\' in line or \'CACHE \' in line):print(f\'[{name}:{seed}:GPU{device}] {line}\',end=\'\',flush=True)\n                rc=proc.wait()\n            return name,seed,rc\n        finally:slots.put(device)\n    with ThreadPoolExecutor(max_workers=len(devices)) as pool:\n        jobs=[pool.submit(run_job,name,seed) for seed in seeds for name in models]\n        for done,f in enumerate(as_completed(jobs),1):\n            name,seed,rc=f.result();print(f\'COMPLETED {done}/{len(jobs)} {name} seed={seed} rc={rc}\',flush=True)\n            if rc:failures.append({\'model\':name,\'seed\':seed,\'returncode\':rc})\n            summarize(out)\n    atomic(out/\'failures.json\',failures)\n    if failures:print(\'Ошибки отдельных запусков:\',failures,flush=True);sys.exit(1)\n\nif __name__==\'__main__\':main()\n', 'cuda_math.py': '"""FP64 Torch-бэкенд PTQ. Оригиналы NumPy неизменны; результат INT8/FP32."""\nimport numpy as np\nfrom algorithms import activation_int8 as ref, spectral_rounding as sr\n\n\ndef _tensor(a, device):\n    import torch\n    return torch.as_tensor(np.ascontiguousarray(a), device=device, dtype=torch.float64)\n\n\ndef _scale(t):\n    import torch\n    lo=float(np.nextafter(np.float32(0),np.float32(1)))\n    hi=np.float32(float(np.finfo(np.float32).max)/127)\n    if float(hi)*127>float(np.finfo(np.float32).max):hi=np.nextafter(hi,np.float32(0))\n    return t.clamp(lo,float(hi)).float().double()\n\n\ndef _baseline(w):\n    import torch\n    peak=w.abs().amax(0)\n    s=_scale(torch.where(peak==0,1.,peak/127))\n    return (w/s).round().clamp(-127,127),s\n\n\ndef _loss(x,w,q,s):\n    return (x@(w-(q.float()*s.float()).double())).square().sum(0)\n\n\ndef _result(q,s):\n    return q.to(dtype=__import__(\'torch\').int8).cpu().numpy(),s.float().cpu().numpy()\n\n\ndef v1_torch(x,w,paired=True,sweeps=3,device=\'cuda\',chunk=128):\n    """Те же координатные шаги, масштаб и защитная проверка; каналы пакетно."""\n    import torch\n    x0=ref._matrix(x);w0=ref._matrix(w)\n    if x0.shape[1]!=w0.shape[0]:raise ValueError(\'shape\')\n    pairs=ref._pairs(x0) if paired else []  # исходное детерминированное правило пар\n    X=_tensor(x0,device);W=_tensor(w0,device)\n    Q,S=_baseline(W);q0=Q.clone();s0=S.clone()\n    # Нулевые направления исключаем один раз, а не синхронизируем CUDA в каждом шаге.\n    directions=[(i,None,1,X[:,i],float(x0[:,i]@x0[:,i])) for i in range(x0.shape[1])]\n    for i,j,sign in pairs:\n        a=x0[:,i]+sign*x0[:,j];directions.append((i,j,sign,X[:,i]+sign*X[:,j],float(a@a)))\n    for begin in range(0,W.shape[1],chunk):\n        sl=slice(begin,begin+chunk);w=W[:,sl];best=Q[:,sl].clone();stored=S[sl].clone();best_loss=_loss(X,w,best,stored)\n        # Оригинал останавливается для блоков 32; сохраняем это правило независимо от ширины GPU-пакета.\n        alive=torch.ones_like(stored,dtype=torch.bool)\n        for _ in range(sweeps):\n            scale=stored.clone();trial=best.clone();res=X@(w-trial*scale)\n            for i,j,sign,a,energy in directions:\n                if not energy:continue\n                low=-127-trial[i];high=127-trial[i]\n                if j is not None:\n                    low=torch.maximum(low,-127-sign*trial[j]);high=torch.minimum(high,127-sign*trial[j])\n                step=(torch.minimum(torch.maximum((a@res)/(energy*scale),low),high)).round()\n                step=torch.where(alive,step,0.)\n                trial[i]+=step\n                if j is not None:trial[j]+=sign*step\n                res-=a[:,None]*(scale*step)\n            res=X@(w-trial*scale);response=X@trial;energy=response.square().sum(0);cross=(response*res).sum(0)\n            fitted=_scale(scale+torch.where(energy>0,cross/energy,0.));improved=torch.zeros_like(alive)\n            for candidate in (stored.clone(),fitted):\n                loss=_loss(X,w,trial,candidate);keep=(loss<best_loss)&alive\n                best=torch.where(keep[None,:],trial,best);stored=torch.where(keep,candidate,stored);best_loss=torch.where(keep,loss,best_loss);improved|=keep\n            # Остановка блока исходной реализации, не отдельных каналов.\n            for i in range(0,len(alive),32):alive[i:i+32]&=improved[i:i+32].any()\n            if not bool(alive.any()):break\n        Q[:,sl]=best;S[sl]=stored\n    keep=_loss(X,W,Q,S)<=_loss(X,W,q0,s0)\n    return _result(torch.where(keep[None,:],Q,q0),torch.where(keep,S,s0))\n\n\ndef _round_batched(start,lo,hi,gram,grad,coupled,tol):\n    import torch\n    v=start.clone();b=v.shape[1];eps=torch.finfo(v.dtype).eps\n    for _ in range(b):\n        active=(lo<v)&(v<hi);any_active=active.any(1)\n        first=active.long().argmax(1);unit=torch.nn.functional.one_hot(first,b).to(v.dtype)*any_active[:,None]\n        u=unit\n        if coupled:\n            C=gram[None,:,:]*(active[:,:,None]&active[:,None,:]) + torch.diag_embed((~active).to(v.dtype)*(b+2))\n            vals,vecs=torch.linalg.eigh(C);eig=vecs[:,:,0]*active\n            eig=torch.where(eig.abs()<16*eps,0.,eig)\n            sign=eig.gather(1,eig.abs().argmax(1,keepdim=True));eig=torch.where(sign<0,-eig,eig)\n            curvature=(eig*(eig@gram-eig)).sum(1)\n            use=(vals[:,0]<1-tol)&(active.sum(1)>1)&(curvature<=0)\n            u=torch.where(use[:,None],eig,unit)\n        lin=2*grad+(hi-v)+(lo-v)\n        def endpoint(direction):\n            edge=torch.where(direction>0,hi,lo);step=torch.where(direction!=0,(edge-v)/direction,torch.inf)\n            a,hit=step.min(1);a=torch.where(any_active,a,0.)\n            point=torch.minimum(torch.maximum(v+a[:,None]*direction,lo),hi)\n            point.scatter_(1,hit[:,None],edge.gather(1,hit[:,None]));point=torch.where(any_active[:,None],point,v)\n            delta=point-v;cost=(lin*delta+delta*(delta@gram-delta)).sum(1)\n            return point,cost\n        p,c=endpoint(u);q,d=endpoint(-u);new=torch.where((d<c)[:,None],q,p)\n        grad=grad+(new-v)@gram;v=new\n    if bool(((lo<v)&(v<hi)).any()):raise ArithmeticError(\'SDR unfinished rounding\')\n    return v\n\n\ndef sdr_torch(x,w,config=None,device=\'cuda\',chunk=128):\n    """Пакетный полёт SDR по активным граням, FP64 и исходный MinMax fallback."""\n    import torch\n    cfg=sr.Config() if config is None else config\n    q0,s0=sr._minmax(w);plan=sr._plan(len(x),*w.shape,cfg)\n    if plan is None:return (q0,s0),{\'reason\':\'budget_or_width\',\'accepted\':0}\n    b,work,memory=plan;X=_tensor(x,device);W=_tensor(w,device);Q=_tensor(q0,device);S=_tensor(s0,device)\n    blocks=[];ranks=[];norm=torch.linalg.vector_norm(X,dim=0)\n    for i in range(0,W.shape[0],b):\n        sl=slice(i,i+b);part=X[:,sl];h=norm[sl];A=part/torch.where(h>0,h,1.);G=A.T@A;G.fill_diagonal_(1.)\n        tol=64*np.finfo(float).eps*max(len(x),len(h));r=int((torch.linalg.eigvalsh(G)>1+tol).sum());ranks.append(r);blocks.append((sl,A,G,h,r,tol))\n    info={\'block\':b,\'work_estimate\':work,\'workspace_estimate\':memory,\'protected_ranks\':ranks,\'accepted\':0}\n    if not any(ranks):return (q0,s0),{**info,\'reason\':\'flat_blocks\'}\n    for begin in range(0,W.shape[1],chunk):\n        slc=slice(begin,begin+chunk);weight=W[:,slc];s=S[slc];base=Q[:,slc].clone()\n        low=(weight/s).floor().clamp(-127,127);high=(low+1).clamp(max=127)\n        lo=(low.float()*s.float()).double();hi=(high.float()*s.float()).double();center=torch.minimum(torch.maximum(weight,lo),hi)\n        residual=X@(center-weight);candidate=base.clone()\n        for sl,A,G,h,r,tol in blocks:\n            a=((lo[sl]-weight[sl])*h[:,None]).T;z=((hi[sl]-weight[sl])*h[:,None]).T\n            start=torch.minimum(torch.maximum(torch.zeros_like(a),a),z)\n            end=_round_batched(start,a,z,G,(A.T@residual).T,r,tol).T\n            candidate[sl]=torch.where(h[:,None]==0,base[sl],torch.where(end==z.T,high[sl],low[sl]))\n            decoded=(candidate[sl].float()*s.float()).double();residual+=X[:,sl]@(decoded-center[sl])\n        old=_loss(X,weight,base,s);new=_loss(X,weight,candidate,s);keep=torch.isfinite(new)&(new<old*(1-64*np.finfo(float).eps))\n        Q[:,slc]=torch.where(keep[None,:],candidate,base);info[\'accepted\']+=int(keep.sum())\n    return _result(Q,S),{**info,\'reason\':\'accepted\' if info[\'accepted\'] else \'no_gain\'}\n', 'gpu_preflight.py': '"""Проверяем MatMul и реальную форму Conv из сбоя ResNet на конкретной GPU."""\nimport os,json,tempfile\nfrom pathlib import Path\nimport numpy as np\n\n\ndef probe(kind,threads=2):\n    import onnx,torch\n    from onnx import helper as h,TensorProto as T,numpy_helper as nh\n    from benchmark import session\n    rng=np.random.default_rng(731)\n    if kind==\'MatMul\':\n        x=rng.normal(size=(16,16)).astype(\'f\');w=np.eye(16,dtype=\'f\');nodes=[h.make_node(\'MatMul\',[\'x\',\'w\'],[\'y\'])];shape=x.shape;ref=x\n    else:\n        x=rng.normal(size=(4,128,28,28)).astype(\'f\');w=rng.normal(0,.02,size=(128,128,3,3)).astype(\'f\')\n        # Тестируем также DQ -> Conv, а не только FP32.\n        if kind==\'DQConv\':\n            q=np.clip(np.rint(w/.001),-127,127).astype(np.int8);s=np.full(128,.001,np.float32);w=q*s[:,None,None,None]\n            nodes=[h.make_node(\'DequantizeLinear\',[\'q\',\'s\'],[\'w\'],axis=0),h.make_node(\'Conv\',[\'x\',\'w\'],[\'y\'],pads=[1,1,1,1])]\n        else:nodes=[h.make_node(\'Conv\',[\'x\',\'w\'],[\'y\'],pads=[1,1,1,1])]\n        shape=x.shape\n        ref=torch.nn.functional.conv2d(torch.from_numpy(x),torch.from_numpy(w),padding=1).numpy()\n    inits=[nh.from_array(q,\'q\'),nh.from_array(s,\'s\')] if kind==\'DQConv\' else [nh.from_array(w,\'w\')]\n    p=h.make_model(h.make_graph(nodes,\'probe\',[h.make_tensor_value_info(\'x\',T.FLOAT,list(x.shape))],[h.make_tensor_value_info(\'y\',T.FLOAT,list(shape))],inits),opset_imports=[h.make_opsetid(\'\',17)],ir_version=8)\n    sess=session(p,threads,profiling=True)\n    try:\n        got=sess.run(None,{\'x\':x})[0];np.testing.assert_allclose(got,ref,atol=3e-4,rtol=2e-3)\n    finally:\n        f=sess.end_profiling();events=json.loads(Path(f).read_text());Path(f).unlink(missing_ok=True)\n    matching=[e for e in events if e.get(\'args\',{}).get(\'provider\')==\'CUDAExecutionProvider\' and (\'MatMul\' in e.get(\'args\',{}).get(\'op_name\',\'\') if kind==\'MatMul\' else \'Conv\' in e.get(\'args\',{}).get(\'op_name\',\'\'))]\n    if not matching:raise RuntimeError(f\'{kind} did not execute on CUDA\')\n    return {\'max_abs\':float(np.max(abs(got-ref))),\'cuda_events\':len(matching)}\n\n\ndef configure(model,allow_cpu=False):\n    import torch,onnxruntime as ort\n    os.environ[\'QUANT_INFER_PROVIDER\']=\'cuda\'\n    torch.set_num_threads(2)\n    if not torch.cuda.is_available():raise RuntimeError(\'PyTorch CUDA missing\')\n    major=str(torch.version.cuda).split(\'.\')[0]\n    # Включаем явное имя runtime для динамического загрузчика cuDNN frontend, если библиотека найдена.\n    import site\n    for base in site.getsitepackages():\n        paths=list(Path(base).glob(f\'nvidia/**/libcudart.so.{major}\'))\n        if paths:\n            os.environ.setdefault(\'CUDNN_FRONTEND_CUDART_LIB_NAME\',str(paths[0]));break\n    cfg=os.environ.get(\'QUANT_GPU_CONFIG\')\n    if cfg:\n        result=json.loads(cfg)\n    else:\n        result={\'device\':torch.cuda.get_device_name(0),\'torch\':torch.__version__,\'cuda\':torch.version.cuda,\'ort\':ort.__version__,\'cudnn\':torch.backends.cudnn.version(),\'matmul\':probe(\'MatMul\'),\'conv\':None,\'attempts\':[]}\n        if model in (\'resnet18\',\'mobilenet_v2\',\'squeezenet1_1\',\'cnn\',\'depthwise_cnn\',\'efficientnet_b0\',\'vit_tiny\',\'custom\',\'all\'):\n            for algo,nhwc in [(\'HEURISTIC\',\'0\'),(\'DEFAULT\',\'0\'),(\'HEURISTIC\',\'1\'),(\'DEFAULT\',\'1\')]:\n                os.environ.update(QUANT_CONV_ALGO=algo,QUANT_NHWC=nhwc)\n                try:\n                    result[\'conv\']={\'fp32\':probe(\'Conv\'),\'dq\':probe(\'DQConv\'),\'algo\':algo,\'nhwc\':nhwc};break\n                except Exception as exc:result[\'attempts\'].append({\'algo\':algo,\'nhwc\':nhwc,\'error\':str(exc)[:500]})\n    print(\'GPU_PROBE\',json.dumps(result),flush=True)\n    if result.get(\'conv\'):\n        os.environ.update(QUANT_CONV_ALGO=result[\'conv\'][\'algo\'],QUANT_NHWC=result[\'conv\'][\'nhwc\'])\n    elif model in (\'resnet18\',\'mobilenet_v2\',\'squeezenet1_1\',\'cnn\',\'depthwise_cnn\',\'efficientnet_b0\',\'vit_tiny\',\'custom\'):\n        if not allow_cpu:raise RuntimeError(\'Conv CUDA probe failed; use --conv-cpu-fallback explicitly or change environment\')\n        os.environ[\'QUANT_INFER_PROVIDER\']=\'cpu\';print(\'EXPLICIT FALLBACK: CNN ONNX inference CPU; math remains CUDA\',flush=True)\n    return result\n\nif __name__==\'__main__\':\n    import subprocess,sys\n    result=configure(\'all\')\n    # Проверяем исходные и пакетные алгоритмы на этой физической GPU.\n    subprocess.run([sys.executable,\'-m\',\'unittest\',\'discover\',\'-s\',\'tests\',\'-p\',\'test_cuda_math.py\',\'-v\'],cwd=Path(__file__).parent,check=True)\n\n', 'graph.py': '"""Адаптеры постоянных FP32-весов ONNX; без зависимости от архитектуры модели."""\nfrom dataclasses import dataclass\nfrom collections import Counter\nimport copy, math\nimport numpy as np\n\n\n@dataclass\nclass Target:\n    weight: str\n    node: str\n    op: str\n    input: str\n    output: str\n    original: np.ndarray\n    attrs: dict\n    bias: np.ndarray | None = None\n\n    @property\n    def axis(self):\n        return 0 if self.op == \'Conv\' or self.attrs.get(\'transB\', 0) else 1\n\n    @property\n    def groups(self):\n        return int(self.attrs.get(\'group\', 1)) if self.op == \'Conv\' else 1\n\n    def matrices(self):\n        w = self.original\n        if self.op == \'Conv\':\n            return [a.reshape(a.shape[0], -1).T.copy() for a in np.split(w, self.groups, axis=0)]\n        return [w.T.copy() if self.axis == 0 else w.copy()]\n\n    def pack(self, pairs):\n        q = np.concatenate([a for a, _ in pairs], axis=1)\n        s = np.concatenate([b for _, b in pairs]).astype(np.float32)\n        if self.op == \'Conv\': q = q.T.reshape(self.original.shape)\n        elif self.axis == 0: q = q.T\n        return np.ascontiguousarray(q), s\n\n    def canonical(self, q, s):\n        """Общий двумерный интерфейс для V2, включая групповые Conv."""\n        if self.op == \'Conv\': return q.reshape(q.shape[0], -1).T.copy(), s\n        return (q.T.copy() if self.axis == 0 else q.copy()), s\n\n    def stored(self, pair):\n        q,s = pair\n        if self.op == \'Conv\': q=q.T.reshape(self.original.shape)\n        elif self.axis == 0: q=q.T\n        return np.ascontiguousarray(q),s\n\n\ndef discover(model):\n    import onnx\n    from onnx import numpy_helper, helper\n    if not any(o.domain in (\'\',\'ai.onnx\') and o.version >= 13 for o in model.opset_import):\n        raise ValueError(\'Нужен стандартный ONNX opset >= 13 для per-axis DQ.\')\n    g=model.graph\n    init={t.name:t for t in g.initializer}\n    uses=Counter(i for n in g.node for i in n.input if i)\n    # Вложенные графы могут ссылаться на внешние инициализаторы.\n    def nested_refs(graph):\n        for n in graph.node:\n            for a in n.attribute:\n                if a.type == onnx.AttributeProto.GRAPH:\n                    uses.update(i for nn in a.g.node for i in nn.input if i)\n                    uses.update(o.name for o in a.g.output)\n                    nested_refs(a.g)\n                elif a.type == onnx.AttributeProto.GRAPHS:\n                    for sub in a.graphs:\n                        uses.update(i for nn in sub.node for i in nn.input if i)\n                        uses.update(o.name for o in sub.output)\n                        nested_refs(sub)\n    nested_refs(g)\n    excluded={v.name for v in list(g.input)+list(g.output)}\n    aliases={n.output[0]:n.input[0] for n in g.node if n.op_type==\'Identity\' and n.domain in (\'\',\'ai.onnx\') and len(n.input)==len(n.output)==1}\n    def constant(name):\n        seen=set()\n        while name in aliases and name not in seen and name not in excluded:\n            seen.add(name);name=aliases[name]\n        return init.get(name) if name not in {v.name for v in g.input} else None\n    targets=[]; skipped=[]\n    for index,n in enumerate(g.node):\n        if n.op_type not in (\'MatMul\',\'Gemm\',\'Conv\') or n.domain not in (\'\',\'ai.onnx\'): continue\n        name=n.name or f\'{n.op_type}_{index}\'\n        reason=None\n        if len(n.input)<2 or n.input[1] not in init: reason=\'nonconstant_rhs\'\n        else:\n            t=init[n.input[1]]\n            if t.name in excluded or uses[t.name]!=1: reason=\'shared_or_public_weight\'\n            elif t.data_type!=onnx.TensorProto.FLOAT: reason=\'not_fp32\'\n            elif n.op_type in (\'MatMul\',\'Gemm\') and len(t.dims)!=2: reason=\'not_rank2\'\n            elif n.op_type==\'Conv\' and len(t.dims) not in (3,4,5): reason=\'unsupported_conv_rank\'\n            elif not all(t.dims): reason=\'empty_weight\'\n        if reason:\n            skipped.append({\'node\':name,\'reason\':reason}); continue\n        attrs={a.name:helper.get_attribute_value(a) for a in n.attribute}\n        bias=None\n        if n.op_type in (\'Gemm\',\'Conv\') and len(n.input)>2 and n.input[2]:\n            bt=constant(n.input[2])\n            if bt is None:\n                skipped.append({\'node\':name,\'reason\':\'nonconstant_bias\'}); continue\n            bias=numpy_helper.to_array(bt)\n        w=numpy_helper.to_array(t).copy()\n        if not np.isfinite(w).all():\n            skipped.append({\'node\':name,\'reason\':\'nonfinite_weight\'}); continue\n        target=Target(t.name,name,n.op_type,n.input[0],n.output[0],w,attrs,bias)\n        if target.groups<1 or (n.op_type==\'Conv\' and w.shape[0]%target.groups):\n            skipped.append({\'node\':name,\'reason\':\'invalid_groups\'}); continue\n        targets.append(target)\n    return targets,skipped\n\n\ndef conv_rows(x, y, target, idx):\n    """Извлечение только выбранных патчей, в порядке C,k1,...; без полного im2col."""\n    a=target.attrs; w=target.original; k=w.shape[2:]; dim=len(k)\n    if x.ndim!=dim+2 or y.ndim!=dim+2: raise ValueError(\'Conv rank\')\n    stride=tuple(a.get(\'strides\',[1]*dim)); dil=tuple(a.get(\'dilations\',[1]*dim))\n    eff=tuple((z-1)*d+1 for z,d in zip(k,dil))\n    auto=a.get(\'auto_pad\',b\'NOTSET\'); auto=auto.decode() if isinstance(auto,bytes) else auto\n    if auto in (\'SAME_UPPER\',\'SAME_LOWER\'):\n        total=[max(0,(o-1)*s+e-i) for o,s,e,i in zip(y.shape[2:],stride,eff,x.shape[2:])]\n        left=[(p+int(auto==\'SAME_LOWER\'))//2 for p in total]; right=[p-l for p,l in zip(total,left)]\n    elif auto==\'VALID\': left=right=[0]*dim\n    elif auto==\'NOTSET\':\n        pads=list(a.get(\'pads\',[0]*(2*dim))); left,right=pads[:dim],pads[dim:]\n    else: raise ValueError(\'auto_pad\')\n    if any(p<0 for p in left+right): raise ValueError(\'negative pad\')\n    xp=np.pad(x,[(0,0),(0,0)]+list(zip(left,right))) if any(left+right) else x\n    coords=np.array(np.unravel_index(idx,(y.shape[0],*y.shape[2:]))).T\n    cin=w.shape[1]; cout=w.shape[0]//target.groups\n    if x.shape[1]!=cin*target.groups: raise ValueError(\'Conv channels\')\n    result=[]\n    # Векторизация выбранных патчей, без полного im2col и копии полного y.\n    rank=2+dim\n    batch_idx=coords[:,0].reshape((len(idx),)+(1,)*(dim+1))\n    spatial=[]\n    for j,(size,st,di) in enumerate(zip(k,stride,dil)):\n        shape=[1]*rank;shape[0]=len(idx);base=(coords[:,j+1]*st).reshape(shape)\n        shape=[1]*rank;shape[2+j]=size;spatial.append(base+np.arange(size).reshape(shape)*di)\n    for group,mat in enumerate(target.matrices()):\n        channels=np.arange(group*cin,(group+1)*cin).reshape((1,cin)+(1,)*dim)\n        rows=xp[(batch_idx,channels,*spatial)].reshape(len(idx),-1)\n        ref=y[(coords[:,0,None],np.arange(group*cout,(group+1)*cout)[None,:],*[coords[:,j+1,None] for j in range(dim)])]\n        calc=rows@mat\n        if target.bias is not None: calc=calc+target.bias[group*cout:(group+1)*cout]\n        check_local(calc,ref)\n        result.append(rows)\n    return result\n\n\ndef check_local(calc, ref):\n    if not np.isfinite(calc).all() or not np.isfinite(ref).all(): raise ValueError(\'nonfinite activations\')\n    # Различный порядок FP32-накоплений ORT и NumPy допустим.\n    scale=max(1.0,float(np.max(np.abs(ref))))\n    if float(np.max(np.abs(calc-ref)))>2e-4*scale:\n        raise ValueError(f\'FP32 adapter mismatch: max={np.max(np.abs(calc-ref)):.3g}, scale={scale:.3g}\')\n\n\ndef sample_rows(x,y,t,cap,rng,mask=None):\n    if x.dtype!=np.float32 or y.dtype!=np.float32: raise ValueError(\'FP32 activations required\')\n    if t.op==\'Conv\':\n        n=y.shape[0]*math.prod(y.shape[2:]); idx=rng.choice(n,min(cap,n),replace=False)\n        return conv_rows(x,y,t,idx),n\n    if t.op==\'Gemm\' and t.attrs.get(\'transA\',0): x=x.T\n    m=t.matrices()[0]; rows=x.reshape(-1,m.shape[0]); ref=y.reshape(-1,m.shape[1])\n    if len(rows)!=len(ref) or x.shape[-1]!=m.shape[0]: raise ValueError(\'linear dimensions\')\n    valid=np.arange(len(rows))\n    if mask is not None and x.ndim>=3 and tuple(x.shape[:-1])==tuple(mask.shape):\n        valid=valid[np.asarray(mask).reshape(-1)!=0]\n    if not len(valid): return [np.empty((0,m.shape[0]),np.float32)],0\n    idx=rng.choice(valid,min(cap,len(valid)),replace=False)\n    calc=rows[idx]@m\n    if t.op==\'Gemm\':\n        calc=calc*float(t.attrs.get(\'alpha\',1.))\n        if t.bias is not None: calc=calc+float(t.attrs.get(\'beta\',1.))*np.broadcast_to(t.bias,y.shape).reshape(ref.shape)[idx]\n    check_local(calc,ref[idx]); return [rows[idx].copy()],len(valid)\n\n\ndef with_outputs(model, names):\n    import onnx\n    from onnx import helper\n    p=copy.deepcopy(model)\n    try: inferred=onnx.shape_inference.infer_shapes(p)\n    except Exception: inferred=p\n    info={v.name:v for v in list(inferred.graph.value_info)+list(inferred.graph.input)+list(inferred.graph.output)}\n    known={v.name for v in p.graph.output}\n    for name in dict.fromkeys(names):\n        if name not in known:\n            p.graph.output.append(info.get(name,helper.make_tensor_value_info(name,onnx.TensorProto.FLOAT,None)))\n    return p\n\n\ndef export(model, targets, bank):\n    """Одна DQ на исходный тензор, первоначальные имена выхода и порядок узлов сохранены."""\n    import onnx\n    from onnx import helper,numpy_helper\n    p=copy.deepcopy(model); g=p.graph\n    known={v.name for v in list(g.input)+list(g.output)+list(g.value_info)} | {i.name for i in g.initializer} | {v for n in g.node for v in list(n.input)+list(n.output)}\n    def unique(name):\n        while name in known: name+=\'_\' \n        known.add(name);return name\n    dq=[]; originals=set()\n    for t in targets:\n        if t.weight not in bank: continue\n        q,s=t.stored(bank[t.weight]); assert q.shape==t.original.shape and s.shape==(q.shape[t.axis],)\n        if q.dtype!=np.int8 or s.dtype!=np.float32 or not np.all(np.isfinite(s)&(s>0)) or np.any(q==-128): raise ValueError(\'invalid INT8 representation\')\n        qname=unique(t.weight+\'__int8\'); sname=unique(t.weight+\'__scale\')\n        g.initializer.extend([numpy_helper.from_array(q,qname),numpy_helper.from_array(s,sname)])\n        dq.append(helper.make_node(\'DequantizeLinear\',[qname,sname],[t.weight],axis=t.axis,name=unique(t.weight+\'__DQ\')))\n        originals.add(t.weight)\n    keep=[i for i in g.initializer if i.name not in originals]; del g.initializer[:]; g.initializer.extend(keep)\n    nodes=list(g.node); del g.node[:];g.node.extend(dq+nodes)\n    onnx.checker.check_model(p)\n    return p\n', 'independent.py': '"""Новые предобученные модели; CIFAR-10 для fidelity (без ложных ImageNet labels), AG News для embeddings."""\nfrom pathlib import Path\nimport gc\nimport numpy as np\nfrom models import Case\nfrom pretrained import indices,export_model,check_export,sha\nVISION_NEW=(\'efficientnet_b0\',\'vit_tiny\')\nNLP_NEW=(\'minilm_embeddings\',)\n\ndef build(name,seed,cfg,root):\n    import torch,onnx\n    torch.set_num_threads(cfg.threads)\n    cache=Path(root)/\'pretrained_cache\';cache.mkdir(parents=True,exist_ok=True);path=cache/(name+\'.onnx\')\n    if name in VISION_NEW:\n        from torchvision.datasets import CIFAR10\n        if name==\'efficientnet_b0\':\n            from torchvision.models import efficientnet_b0,EfficientNet_B0_Weights\n            weights=EfficientNet_B0_Weights.IMAGENET1K_V1;model=efficientnet_b0(weights=weights).eval();transform=weights.transforms();repo=weights.url\n        else:\n            import timm\n            repo=\'vit_tiny_patch16_224.augreg_in21k_ft_in1k\';model=timm.create_model(repo,pretrained=True).eval()\n            for mod in model.modules():\n                if hasattr(mod,\'fused_attn\'):mod.fused_attn=False\n            transform=timm.data.create_transform(**timm.data.resolve_model_data_config(model),is_training=False)\n        names=[\'input\'];args=(torch.randn(1,3,224,224,generator=torch.Generator().manual_seed(731)),)\n        p=onnx.load(path) if path.exists() else export_model(model,args,path,names,{\'input\':{0:\'batch\'},\'logits\':{0:\'batch\'}})\n        diff=check_export(model,args,p,names,cfg);del model;gc.collect()\n        tr=CIFAR10(str(cache),train=True,download=True,transform=transform);te=CIFAR10(str(cache),train=False,download=True,transform=transform)\n        ti=indices(len(tr),cfg.fit+cfg.dev,seed);vi=indices(len(te),cfg.test,731)\n        def fetch(ds,ids):return np.stack([ds[int(i)][0].numpy() for i in ids]).astype(np.float32)\n        x=fetch(tr,ti);y=fetch(te,vi);shift=y*.8+.15\n        case=Case(name,p,{\'input\':x[:cfg.fit]},{\'input\':x[cfg.fit:]},{\'input\':y},\'classification\',None,{\'input\':shift},None,\'ImageNet checkpoint, CIFAR-10 images; only FP32 fidelity and prediction agreement, not CIFAR accuracy.\')\n        meta=dict(checkpoint=repo,dataset=\'torchvision CIFAR10\',fit_ids=ti[:cfg.fit].tolist(),dev_ids=ti[cfg.fit:].tolist(),test_ids=vi.tolist(),shift=\'affine normalized pixels\')\n    elif name in NLP_NEW:\n        from transformers import AutoModel,AutoTokenizer\n        from datasets import load_dataset\n        repo=\'sentence-transformers/all-MiniLM-L6-v2\';tok=AutoTokenizer.from_pretrained(repo);base=AutoModel.from_pretrained(repo,attn_implementation=\'eager\').eval()\n        rev=getattr(base.config,\'_commit_hash\',None)\n        class Wrapper(torch.nn.Module):\n            def __init__(self,model):super().__init__();self.model=model\n            def forward(self,input_ids,attention_mask):\n                h=self.model(input_ids=input_ids,attention_mask=attention_mask).last_hidden_state\n                mask=attention_mask.unsqueeze(-1).to(h.dtype);z=(h*mask).sum(1)/mask.sum(1).clamp_min(1.)\n                return torch.nn.functional.normalize(z,p=2,dim=1)\n        model=Wrapper(base).eval();names=[\'input_ids\',\'attention_mask\'];args=(torch.ones((1,128),dtype=torch.long),torch.ones((1,128),dtype=torch.long))\n        p=onnx.load(path) if path.exists() else export_model(model,args,path,names,{k:{0:\'batch\'} for k in names+[\'logits\']})\n        diff=check_export(model,args,p,names,cfg);del model,base;gc.collect()\n        ds=load_dataset(\'fancyzhx/ag_news\');ti=indices(len(ds[\'train\']),cfg.fit+cfg.dev,seed);vi=indices(len(ds[\'test\']),cfg.test,731)\n        def enc(split,ids):\n            e=tok(list(ds[split].select(ids.tolist())[\'text\']),padding=\'max_length\',truncation=True,max_length=128,return_tensors=\'np\');return {k:e[k].astype(np.int64) for k in names}\n        tr=enc(\'train\',ti);te=enc(\'test\',vi);shift={k:v.copy() for k,v in te.items()};shift[\'input_ids\'][:,64:]=tok.pad_token_id;shift[\'attention_mask\'][:,64:]=0\n        case=Case(name,p,{k:v[:cfg.fit] for k,v in tr.items()},{k:v[cfg.fit:] for k,v in tr.items()},te,\'generic\',None,shift,None,\'Normalized mean-pooled MiniLM embeddings on AG News; no task accuracy claimed.\')\n        meta=dict(checkpoint=repo,checkpoint_revision=rev,dataset=\'fancyzhx/ag_news\',fit_ids=ti[:cfg.fit].tolist(),dev_ids=ti[cfg.fit:].tolist(),test_ids=vi.tolist(),shift=\'truncate to 64 tokens\')\n    else:raise ValueError(name)\n    meta.update(model_sha256=sha(path),export_max_abs=diff);case.provenance=meta;return case\n\n\ndef lstm_case(seed,cfg,root):\n    """Реальная обученная LSTM; recurrent W/R остаются FP32, проверяем границы адаптеров."""\n    import torch,onnx\n    from sklearn.datasets import load_digits\n    from models import split_indices\n    torch.set_num_threads(cfg.threads);torch.manual_seed(731)\n    data=load_digits();x=(data.data/16).astype(np.float32).reshape(-1,8,8);y=data.target.astype(np.int64)\n    tr,fi,de,te=split_indices(len(x),seed,cfg.fit,cfg.dev,cfg.test)\n    class Net(torch.nn.Module):\n        def __init__(self):\n            super().__init__();self.rnn=torch.nn.LSTM(8,32,batch_first=True);self.fc=torch.nn.Linear(32,10)\n        def forward(self,x):\n            h=x.new_zeros((1,x.shape[0],32));c=x.new_zeros((1,x.shape[0],32))\n            z,_=self.rnn(x,(h,c));return self.fc(z[:,-1])\n    net=Net();opt=torch.optim.Adam(net.parameters(),lr=.01);xt=torch.from_numpy(x[tr]);yt=torch.from_numpy(y[tr]);gen=torch.Generator().manual_seed(731)\n    for _ in range(20):\n        for idx in torch.randperm(len(xt),generator=gen).split(64):\n            opt.zero_grad();loss=torch.nn.functional.cross_entropy(net(xt[idx]),yt[idx]);loss.backward();opt.step()\n    net.eval();cache=Path(root)/\'pretrained_cache\';cache.mkdir(exist_ok=True,parents=True);path=cache/\'lstm_head.onnx\';args=(torch.from_numpy(x[:1]),)\n    p=onnx.load(path) if path.exists() else export_model(net,args,path,[\'input\'],{\'input\':{0:\'batch\'},\'logits\':{0:\'batch\'}})\n    diff=check_export(net,args,p,[\'input\'],cfg);shift=np.clip(x[te]+np.random.default_rng(731).normal(0,.18,x[te].shape).astype(np.float32),0,1)\n    case=Case(\'lstm_head\',p,{\'input\':x[fi]},{\'input\':x[de]},{\'input\':x[te]},\'classification\',y[te],{\'input\':shift},y[te],\'Digits trained LSTM; fused LSTM operator is unsupported, only supported output weights quantized.\')\n    case.provenance=dict(train_ids=tr.tolist(),fit_ids=fi.tolist(),dev_ids=de.tolist(),test_ids=te.tolist(),model_sha256=sha(path),export_max_abs=diff)\n    return case\n', 'mixed.py': '"""Политики из одного банка; измерения реальной ONNX-модели, не сумма локальных MSE."""\nfrom pathlib import Path\nimport json,time,gc\nimport numpy as np\nfrom adaptive_policies import POLICIES,VARIANTS,aggregate,policy,check,Portfolio\nfrom algorithms.adaptive_selector import search_priority\nfrom algorithms import logit_int8 as v2\n\nEXTRA=POLICIES+[\'Progressive-Local\',\'Single-Robust-Local\',\'Single-Dev-Local\']\n\ndef mixed_runs(case,cfg,root,valid,banks,requested,evaluate,search_feeds,check_feeds,base_sess,cal_time):\n    from benchmark import session,infer,export,save_bank\n    if not requested:return\n    prior=Portfolio(json.loads((Path(__file__).parent/\'prior.json\').read_text()))\n    records={a:{l[\'weight\']:l for l in json.loads((root/(m+\'.json\')).read_text())[\'layers\']} for a,m in VARIANTS.items()}\n    stats={};vectors={};costs={}\n    for t in valid:\n        k=t.weight;stats[k]={};costs[k]={}\n        for a in VARIANTS:\n            l=records[a][k];loss,vec=aggregate(l);stats[k][a]=loss;costs[k][a]=l[\'layer_wall_seconds\']\n            if a==\'minmax\':vectors[k]=vec\n    baseline=dict.fromkeys(stats,\'minmax\')\n    def payload(c):return {k:banks[VARIANTS[a]][k] for k,a in c.items()}\n    def run(c,feeds):\n        if c is None:return infer(base_sess,feeds,cfg.batch)\n        p=export(case.model,valid,payload(c));s=session(p,cfg.threads);z=infer(s,feeds,cfg.batch);del s,p;return z\n    ref_check=run(None,check_feeds);base_check=run(baseline,check_feeds)\n    ref_search=run(None,search_feeds)\n    check_cache={tuple(baseline.items()):base_check}\n    # Cache only a few small outputs, not model graphs.\n    def measured(c):\n        key=tuple(c.items())\n        if key not in check_cache:check_cache[key]=run(c,check_feeds)\n        return check_cache[key]\n    def choose_global(local,plans):\n        """Original V2 on classifiers, exact greedy on generic outputs. Only built candidates."""\n        keys=[k for k in local if any(a!=local[k] for a in (\'minmax\',)+plans[k])]\n        keys=sorted(keys,key=lambda k:(-search_priority({a:stats[k][a] for a in (\'minmax\',)+plans[k]}),k))\n        active=keys[:min(cfg.v2_layers or 4,4)];calls=0\n        if not active:return local,{\'kind\':\'none\',\'calls\':0,\'active\':[]}\n        opts={k:tuple(dict.fromkeys((local[k],\'minmax\')+plans[k])) if k in active else (local[k],) for k in local}\n        if case.task==\'classification\' and len(ref_search)==1 and next(iter(ref_search.values())).ndim==2 and next(iter(ref_search.values())).shape[1]>=2:\n            b={k:[banks[VARIANTS[a]][k] for a in opts[k]] for k in local}\n            cap=5+sum(len(v)-1 for v in opts.values())\n            def cb(weights,feeds):\n                nonlocal calls\n                if calls>=cap:raise ValueError(\'V2 budget\')\n                calls+=1\n                if weights is None:return next(iter(run(None,feeds).values()))\n                p=export(case.model,valid,weights);s=session(p,cfg.threads);z=next(iter(infer(s,feeds,cfg.batch).values()));del s,p;return z\n            idx=v2.select_int8(b,cb,case.fit,search_feeds,width=cfg.beam)\n            return {k:opts[k][i] for k,i in idx.items()},{\'kind\':\'original_V2\',\'calls\':calls,\'cap\':cap,\'active\':active}\n        cur=dict(local);cur_z=run(cur,search_feeds);calls=1;cap=9\n        for k in active:\n            for a in opts[k]:\n                if calls>=cap:break\n                if a==cur[k]:continue\n                trial={**cur,k:a};z=run(trial,search_feeds);calls+=1\n                if check(ref_search,cur_z,z)[0]:cur,cur_z=trial,z\n        return cur,{\'kind\':\'exact_multioutput_greedy\',\'calls\':calls,\'cap\':cap,\'active\':active}\n    # Map exact selections to measured model outputs. Same bank/selection means same exported graph.\n    seen={tuple(dict.fromkeys(stats,a).items()):m for a,m in VARIANTS.items()}\n    for name in requested:\n        if (root/(name+\'.json\')).exists():\n            previous=json.loads((root/(name+\'.json\')).read_text())\n            if \'selection\' in previous:seen[tuple((k,previous[\'selection\'][k]) for k in stats)]=name\n            continue\n        print(\'    policy\',name,flush=True);t0=time.perf_counter();local={};plans={};details=[];build_seconds=0.;guard_seconds=0.\n        local_only=name.endswith(\'-Local\');kind=name.removesuffix(\'-Local\')\n        for k in stats:\n            start=time.perf_counter();winner,planned,info=policy(kind,stats[k],vectors[k],prior);guard_seconds+=time.perf_counter()-start\n            local[k]=winner;plans[k]=planned;build_seconds+=costs[k][\'minmax\']+sum(costs[k][a] for a in planned)\n            details.append(dict(weight=k,local=winner,attempted=list(planned),bounds=info.get(\'upper\'),radius=info.get(\'radius\'),pre=info.get(\'pre\'),dev_ratio=stats[k][winner].dev/stats[k][\'minmax\'].dev if stats[k][\'minmax\'].dev else None))\n        candidate=local;global_info={\'kind\':\'none\',\'calls\':0};global_seconds=0.\n        if kind==\'Progressive-Global\':\n            tt=time.perf_counter();candidate,global_info=choose_global(local,plans);global_seconds=time.perf_counter()-tt\n        checks=[];final=local if local_only else baseline\n        # Independent model check: comparison only to all-MinMax, every output must be non-worse.\n        # Local-only variants deliberately expose unguarded test outcomes for ablation.\n        best=None\n        for c in [local]+([candidate] if candidate!=local else []):\n            ok,diag=check(ref_check,base_check,measured(c));checks.append(dict(diag,selection=\'local\' if c==local else \'global\'))\n            if ok and not local_only:\n                score=float(np.mean([r for r in diag[\'ratios\'] if r is not None]))\n                if best is None or score<best:best=score;final=c\n        if local_only:final=local\n        elapsed=time.perf_counter()-t0;bank=payload(final);p=export(case.model,valid,bank)\n        # Actual policy replay time excludes all-bank generation. Estimated standalone cost uses measured layer builds.\n        extra=dict(calibration_seconds=cal_time,policy_replay=True,policy_build_seconds_estimate=build_seconds,policy_replay_seconds=elapsed,policy_guard_decision_seconds=guard_seconds,global_search_seconds=global_seconds,global_search=global_info,global_checks=checks,global_guard_enabled=not local_only,selection=final,local_selection=local,policy_decisions=details,expensive_builds=sum(len(v) for v in plans.values()),fallback_to_all_minmax=final==baseline,payload_bytes=sum(q.nbytes+s.nbytes for q,s in bank.values()),quantized_parameters=sum(q.size for q,s in bank.values()))\n        # Compact operator-level coverage, no duplicate per-group diagnostics.\n        layers=[]\n        for k,a in final.items():\n            l=records[a][k];gs=l[\'groups\'];layers.append({\'weight\':k,\'operator\':l[\'operator\'],\'groups\':[{\'channels\':sum(g[\'channels\'] for g in gs),\'accepted\':sum(g.get(\'accepted\',0) for g in gs),\'attempted_channels\':sum(g.get(\'attempted_channels\',0) for g in gs)}]})\n        key=tuple((k,final[k]) for k in stats)\n        if key in seen:extra[\'reuse_from\']=seen[key]\n        evaluate(name,p,build_seconds+elapsed,layers,extra);seen[key]=name;del p,bank;gc.collect()\n', 'models.py': '"""Небольшие воспроизводимые модели и раздельные наборы данных; без загрузок из сети."""\nfrom dataclasses import dataclass\nfrom pathlib import Path\nimport json,zipfile\nimport numpy as np\n\n@dataclass\nclass Case:\n    name: str\n    model: object\n    fit: dict\n    dev: dict\n    test: dict\n    task: str\n    labels: np.ndarray | None\n    shift: dict | None = None\n    shift_labels: np.ndarray | None = None\n    note: str = \'\'\n    historical: bool = False\n\nclass Builder:\n    def __init__(self):\n        self.nodes=[]; self.weights=[]\n    def const(self,name,a):\n        from onnx import numpy_helper\n        self.weights.append(numpy_helper.from_array(np.ascontiguousarray(a),name)); return name\n    def node(self,op,ins,out,**attrs):\n        from onnx import helper\n        self.nodes.append(helper.make_node(op,ins,[out],name=out,**attrs)); return out\n    def dense(self,x,w,b,out,trans=False):\n        return self.node(\'Gemm\',[x,self.const(out+\'_W\',np.asarray(w.T if trans else w,np.float32)),self.const(out+\'_b\',np.asarray(b,np.float32))],out,transB=int(trans))\n    def model(self,name,inputs,outputs):\n        import onnx\n        from onnx import helper,TensorProto\n        make=lambda item:helper.make_tensor_value_info(item[0],TensorProto.FLOAT,item[1])\n        g=helper.make_graph(self.nodes,name,[make(i) for i in inputs],[make(i) for i in outputs],self.weights)\n        p=helper.make_model(g,opset_imports=[helper.make_opsetid(\'\',17)],producer_name=\'quant-benchmark\',ir_version=8)\n        onnx.checker.check_model(p); return p\n\n\ndef split_indices(n,seed,fit_count,dev_count,test_count):\n    # Test фиксирован между seed, обучение не пересекается с calibration/dev/test.\n    perm=np.random.default_rng(731).permutation(n)\n    train=perm[:int(.6*n)]; pool=perm[int(.6*n):int(.8*n)]; test=perm[int(.8*n):]\n    pool=np.random.default_rng(seed).permutation(pool)\n    f=min(fit_count,max(1,len(pool)*2//3)); d=min(dev_count,len(pool)-f)\n    return train,pool[:f],pool[f:f+d],test[:test_count]\n\n\ndef ridge(x,y,lam=1e-2):\n    x=np.asarray(x,np.float64); y=np.asarray(y,np.float64)\n    a=x-x.mean(0); b=y-y.mean(0); w=np.linalg.solve(a.T@a+lam*np.eye(a.shape[1]),a.T@b)\n    return w.astype(np.float32),(y.mean(0)-x.mean(0)@w).astype(np.float32)\n\n\ndef conv_numpy(x,w,groups=1):\n    # Только для построения небольших демонстрационных моделей 3x3, stride 1, pad 1.\n    k=w.shape[-1]; p=k//2\n    padded=np.pad(x,((0,0),(0,0),(p,p),(p,p)))\n    patches=np.lib.stride_tricks.sliding_window_view(padded,(k,k),axis=(2,3))\n    parts=[]\n    for a,b in zip(np.split(patches,groups,axis=1),np.split(w,groups,axis=0)):\n        parts.append(np.einsum(\'nchwij,ocij->nohw\',a,b,optimize=True))\n    return np.concatenate(parts,axis=1).astype(np.float32)\n\n\ndef make_case(name,seed,cfg,root,archive=None):\n    from sklearn.datasets import load_digits,load_diabetes\n    from sklearn.neural_network import MLPClassifier\n    from sklearn.decomposition import PCA\n    if name==\'bert_tiny\': return bert_case(seed,cfg,archive)\n    rng=np.random.default_rng(1207); b=Builder()\n    if name==\'regression\':\n        xx,yy=load_diabetes(return_X_y=True); x=xx.astype(np.float32); y=yy.astype(np.float32)[:,None]\n        tr,fit,dev,test=split_indices(len(x),seed,cfg.fit,cfg.dev,cfg.test)\n        mean=x[tr].mean(0); sd=x[tr].std(0); x=(x-mean)/np.maximum(sd,1e-6)\n        w1=rng.normal(0,.35,(x.shape[1],32)).astype(np.float32); b1=rng.normal(0,.1,32).astype(np.float32)\n        h=np.tanh(x@w1+b1); w2,b2=ridge(h[tr],y[tr],.2)\n        b.dense(\'x\',w1,b1,\'h\',trans=True);b.node(\'Tanh\',[\'h\'],\'a\');b.dense(\'a\',w2,b2,\'prediction\')\n        model=b.model(name,[(\'x\',[None,x.shape[1]])],[(\'prediction\',[None,1])]); task=\'regression\'; label=y\n        shift=x[test]+np.float32(.35)*rng.normal(size=x[test].shape).astype(np.float32)\n        # Сдвиг входов без известных истинных меток; оцениваем только fidelity.\n        return Case(name,model,{\'x\':x[fit]},{\'x\':x[dev]},{\'x\':x[test]},task,label[test],{\'x\':shift},None,\'Diabetes; фиксированные случайные признаки, обученный ridge-выход.\')\n    data=load_digits(); x=(data.data/16).astype(np.float32); y=data.target.astype(np.int64)\n    tr,fit,dev,test=split_indices(len(x),seed,cfg.fit,cfg.dev,cfg.test)\n    task=\'classification\'; label=y; out=[(\'logits\',[None,10])]; shape=[None,64]; note=\'Digits; независимые train/calibration/dev/test.\'\n    if name==\'mlp\':\n        model_fit=MLPClassifier(hidden_layer_sizes=(32,),max_iter=200,random_state=731,early_stopping=True).fit(x[tr],y[tr])\n        b.dense(\'x\',model_fit.coefs_[0],model_fit.intercepts_[0],\'h\',trans=True);b.node(\'Relu\',[\'h\'],\'a\');b.dense(\'a\',model_fit.coefs_[1],model_fit.intercepts_[1],\'logits\')\n    elif name in (\'cnn\',\'depthwise_cnn\'):\n        x=x.reshape(-1,1,8,8); shape=[None,1,8,8]; f=rng.normal(0,.35,(4,1,3,3)).astype(np.float32)\n        z=b.node(\'Conv\',[\'x\',b.const(\'conv0_W\',f)],\'conv0\',pads=[1]*4); z=b.node(\'Relu\',[z],\'relu0\')\n        h=np.maximum(conv_numpy(x[tr],f),0)\n        if name==\'depthwise_cnn\':\n            f2=rng.normal(0,.35,(4,1,3,3)).astype(np.float32);f3=rng.normal(0,.35,(6,2,1,1)).astype(np.float32)\n            z=b.node(\'Conv\',[z,b.const(\'depth_W\',f2)],\'depth\',group=4,auto_pad=\'SAME_UPPER\');z=b.node(\'Relu\',[z],\'relu1\');h=np.maximum(conv_numpy(h,f2,4),0)\n            z=b.node(\'Conv\',[z,b.const(\'group_W\',f3)],\'group\',group=2);z=b.node(\'Relu\',[z],\'relu2\');h=np.maximum(conv_numpy(h,f3,2),0)\n        z=b.node(\'AveragePool\',[z],\'pool\',kernel_shape=[2,2],strides=[2,2]);b.node(\'Flatten\',[z],\'features\',axis=1)\n        feat=h.reshape(len(h),h.shape[1],4,2,4,2).mean((3,5)).reshape(len(h),-1)\n        w,bb=ridge(feat,np.eye(10)[y[tr]],.1);b.dense(\'features\',w,bb,\'logits\',trans=True)\n        note+=\' Фиксированные случайные свёртки, обученная ridge-голова; не pretrained CNN.\'\n    elif name==\'autoencoder\':\n        p=PCA(n_components=16,random_state=731).fit(x[tr]); w=p.components_.T.astype(np.float32); mu=p.mean_.astype(np.float32)\n        b.dense(\'x\',w,-mu@w,\'latent\');b.dense(\'latent\',w.T,mu,\'reconstruction\',trans=True);out=[(\'reconstruction\',[None,64])];task=\'reconstruction\';label=x\n        note+=\' PCA-автоэнкодер, обученный на train.\'\n    elif name==\'multi_output\':\n        w=rng.normal(0,.4,(64,24)).astype(np.float32); bb=np.zeros(24,np.float32); h=np.maximum(x@w,0)\n        b.dense(\'x\',w,bb,\'h\');b.node(\'Relu\',[\'h\'],\'a\')\n        w1,b1=ridge(h[tr],np.eye(10)[y[tr]],.1);w2,b2=ridge(h[tr],x[tr],.1)\n        b.dense(\'a\',w1,b1,\'logits\');b.dense(\'a\',w2,b2,\'reconstruction\');out=[(\'logits\',[None,10]),(\'reconstruction\',[None,64])];task=\'multi\';label=None\n        note+=\' Две головы: классификация и реконструкция; V2 пропускается.\'\n    elif name==\'rnn\':\n        # Явная развёртка рекуррентной ячейки на 8 шагов, общие W/R не дублируются.\n        x=x.reshape(-1,8,8);shape=[None,8,8]; w=rng.normal(0,.35,(8,16)).astype(np.float32);r=rng.normal(0,.15,(16,16)).astype(np.float32)\n        b.const(\'W_shared\',w);b.const(\'R_shared\',r)\n        hidden=np.zeros((len(tr),16),np.float32); prev=None\n        for i in range(8):\n            idx=b.const(f\'idx{i}\',np.array([i],np.int64));g=b.node(\'Gather\',[\'x\',idx],f\'g{i}\',axis=1)\n            g=b.node(\'Flatten\',[g],f\'f{i}\',axis=1);a=b.node(\'MatMul\',[g,\'W_shared\'],f\'u{i}\')\n            if prev is not None:\n                rpart=b.node(\'MatMul\',[prev,\'R_shared\'],f\'r{i}\');a=b.node(\'Add\',[a,rpart],f\'add{i}\')\n            prev=b.node(\'Tanh\',[a],f\'h{i}\');hidden=np.tanh(x[tr,i]@w+hidden@r)\n        w2,b2=ridge(hidden,np.eye(10)[y[tr]],.1);b.dense(prev,w2,b2,\'logits\');note+=\' Развёрнутый RNN; общие W/R намеренно пропускаются, квантуется выходная голова.\'\n    else: raise ValueError(name)\n    model=b.model(name,[(\'x\',shape)],out)\n    shift=np.clip(x[test]+rng.normal(0,.18,x[test].shape).astype(np.float32),0,1)\n    sl=label[test] if label is not None else None\n    if task==\'reconstruction\': shift_label=shift\n    else: shift_label=sl\n    return Case(name,model,{\'x\':x[fit]},{\'x\':x[dev]},{\'x\':x[test]},task,sl,{\'x\':shift},shift_label,note)\n\n\ndef bert_case(seed,cfg,archive):\n    import onnx\n    from tokenizers import Tokenizer\n    if archive is None: raise FileNotFoundError(\'Поместите onnx-quant-lab-handoff-v0.5.zip рядом со стендом или передайте --archive.\')\n    path=Path(archive); prefix=\'onnx-quant-lab/demo-assets/\'\n    if path.is_dir():\n        d=path/\'demo-assets\' if (path/\'demo-assets\').exists() else path\n        read=lambda n:(d/n).read_bytes()\n    else:\n        with zipfile.ZipFile(path) as z:\n            blobs={n:z.read(prefix+n) for n in [\'bert_tiny_fp32.onnx\',\'calibration_samples.json\',\'evaluation_samples.json\',\'tokenizer/tokenizer.json\']}\n        read=blobs.__getitem__\n    model=onnx.load_model_from_string(read(\'bert_tiny_fp32.onnx\')); tok=Tokenizer.from_str(read(\'tokenizer/tokenizer.json\').decode())\n    tok.enable_truncation(max_length=64);tok.enable_padding(length=64,pad_id=tok.token_to_id(\'[PAD]\') or 0,pad_token=\'[PAD]\')\n    cal=json.loads(read(\'calibration_samples.json\')); eva=json.loads(read(\'evaluation_samples.json\'))\n    perm=np.random.default_rng(seed).permutation(len(cal)); f=min(cfg.fit,192);d=min(cfg.dev,len(cal)-f)\n    inputs={i.name for i in model.graph.input}\n    def encode(rows):\n        enc=tok.encode_batch([r[\'text\'] for r in rows]); allfeeds={\'input_ids\':np.array([e.ids for e in enc],np.int64),\'attention_mask\':np.array([e.attention_mask for e in enc],np.int64),\'token_type_ids\':np.array([e.type_ids for e in enc],np.int64)}\n        return {k:v for k,v in allfeeds.items() if k in inputs}\n    # Фиксированная подвыборка исторического test между seed.\n    chosen=[eva[i] for i in np.random.default_rng(731).permutation(len(eva))[:cfg.test]]\n    return Case(\'bert_tiny\',model,encode([cal[i] for i in perm[:f]]),encode([cal[i] for i in perm[f:f+d]]),encode(chosen),\'classification\',np.array([r[\'label\'] for r in chosen]),note=\'Архив SST-2, исторический test повторно использован; не независимое подтверждение.\',historical=True)\n\n\ndef custom_case(model_path,fit_path,dev_path,test_path,task=\'generic\',shift_path=None):\n    import onnx\n    def load(path):\n        with np.load(path,allow_pickle=False) as z:\n            return {k:z[k] for k in z.files if k!=\'__labels__\'},z[\'__labels__\'] if \'__labels__\' in z.files else None\n    a,_=load(fit_path);b,_=load(dev_path);c,y=load(test_path);s,sy=load(shift_path) if shift_path else (None,None)\n    return Case(Path(model_path).stem,onnx.load(model_path),a,b,c,task,y,s,sy,\'Пользовательская ONNX; непересечение выборок обеспечивается пользователем.\')\n', 'pretrained.py': '"""Реальные предобученные модели; выборки фиксируются независимо от методов."""\nfrom pathlib import Path\nimport io, tarfile, urllib.request, json, hashlib, gc\nimport numpy as np\nfrom models import Case\n\nVISION=(\'resnet18\',\'mobilenet_v2\',\'squeezenet1_1\')\nNLP=(\'distilbert_sst2\',)\nIMAGENETTE={\'n01440764\':0,\'n02102040\':217,\'n02979186\':482,\'n03000684\':491,\'n03028079\':497,\'n03394916\':566,\'n03417042\':569,\'n03425413\':571,\'n03445777\':574,\'n03888257\':701}\nURL=\'https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-320.tgz\'\n\ndef indices(n, k, seed):\n    return np.random.default_rng(seed).permutation(n)[:min(k,n)]\n\ndef download(path):\n    if not path.exists():\n        tmp=path.with_suffix(\'.partial\')\n        print(\'Download:\',URL,flush=True)\n        urllib.request.urlretrieve(URL,tmp);tmp.replace(path)\n\ndef sha(path):\n    h=hashlib.sha256()\n    with open(path,\'rb\') as f:\n        for chunk in iter(lambda:f.read(2**20),b\'\'):h.update(chunk)\n    return h.hexdigest()\n\ndef vision_data(cache, transform, cfg, seed):\n    from PIL import Image\n    path=cache/\'imagenette2-320.tgz\';download(path)\n    # Только чтение выбранных файлов, без распаковки путей из архива.\n    with tarfile.open(path,\'r:gz\') as tar:\n        items={\'train\':[],\'val\':[]}\n        for m in tar.getmembers():\n            p=Path(m.name).parts\n            if m.isfile() and len(p)==4 and p[1] in items and p[2] in IMAGENETTE and p[-1].lower().endswith((\'.jpeg\',\'.jpg\',\'.png\')):items[p[1]].append(m)\n        for a in items.values():a.sort(key=lambda m:m.name)\n        tr=indices(len(items[\'train\']),cfg.fit+cfg.dev,seed);te=indices(len(items[\'val\']),cfg.test,731)\n        if len(tr)!=cfg.fit+cfg.dev or len(te)!=cfg.test:raise ValueError(\'Not enough Imagenette examples\')\n        chosen=[items[\'train\'][i] for i in tr]+[items[\'val\'][i] for i in te]\n        x=np.empty((len(chosen),3,224,224),np.float32);y=np.empty(len(chosen),np.int64)\n        # Читаем в порядке смещений архива, не перематывая gzip для каждого изображения.\n        for j in sorted(range(len(chosen)),key=lambda j:chosen[j].offset_data):\n            m=chosen[j]\n            with tar.extractfile(m) as f:raw=f.read()\n            with Image.open(io.BytesIO(raw)) as img:x[j]=transform(img.convert(\'RGB\')).numpy()\n            y[j]=IMAGENETTE[Path(m.name).parts[2]]\n    a,b=cfg.fit,cfg.fit+cfg.dev\n    test=x[b:]; shift=test.copy();shift*=np.float32(.8);shift+=np.float32(.15)\n    return {\'input\':x[:a]},{\'input\':x[a:b]},{\'input\':test},y[b:],{\'input\':shift},y[b:],{\'fit_ids\':[m.name for m in chosen[:a]],\'dev_ids\':[m.name for m in chosen[a:b]],\'test_ids\':[m.name for m in chosen[b:]],\'dataset\':URL,\'dataset_sha256\':sha(path)}\n\ndef export_model(model, args, path, names, axes):\n    import torch,onnx\n    tmp=path.with_suffix(\'.tmp.onnx\')\n    with torch.no_grad():\n        torch.onnx.export(model.eval(),args,str(tmp),input_names=names,output_names=[\'logits\'],dynamic_axes=axes,opset_version=17,dynamo=False,do_constant_folding=True)\n    p=onnx.load(tmp);onnx.checker.check_model(p);tmp.replace(path)\n    return p\n\ndef check_export(model, args, p, names, cfg):\n    import torch\n    from benchmark import session\n    with torch.no_grad():ref=model(*args).detach().cpu().numpy()\n    s=session(p,cfg.threads);out=s.run(None,{k:v.cpu().numpy() for k,v in zip(names,args)})[0]\n    np.testing.assert_allclose(out,ref,rtol=2e-3,atol=5e-4)\n    return float(np.max(abs(out-ref)))\n\ndef make_pretrained(name,seed,cfg,root):\n    import torch,onnx\n    torch.set_num_threads(cfg.threads)\n    cache=Path(root)/\'pretrained_cache\';cache.mkdir(parents=True,exist_ok=True)\n    path=cache/(name+\'.onnx\')\n    if name in VISION:\n        import torchvision.models as tv\n        factory,weights={\'resnet18\':(tv.resnet18,tv.ResNet18_Weights.IMAGENET1K_V1),\'mobilenet_v2\':(tv.mobilenet_v2,tv.MobileNet_V2_Weights.IMAGENET1K_V1),\'squeezenet1_1\':(tv.squeezenet1_1,tv.SqueezeNet1_1_Weights.IMAGENET1K_V1)}[name]\n        # Явно фиксируем checkpoint, не DEFAULT.\n        model=factory(weights=weights).eval(); args=(torch.randn(1,3,224,224,generator=torch.Generator().manual_seed(731)),);names=[\'input\']\n        p=onnx.load(path) if path.exists() else export_model(model,args,path,names,{\'input\':{0:\'batch\'},\'logits\':{0:\'batch\'}})\n        diff=check_export(model,args,p,names,cfg);del model;gc.collect()\n        fit,dev,test,labels,shift,sy,meta=vision_data(cache,weights.transforms(),cfg,seed)\n        meta.update(checkpoint=weights.url,export_max_abs=diff,model_sha256=sha(path),shift=\'affine change of normalized pixels; labels unchanged\')\n        case=Case(name,p,fit,dev,test,\'classification\',labels,shift,sy,\'ImageNet pretrained; Imagenette train/val separated for PTQ, not necessarily unseen during ImageNet pretraining; 1000-class output, original class IDs.\')\n    elif name in NLP:\n        from transformers import AutoModelForSequenceClassification,AutoTokenizer\n        from datasets import load_dataset\n        repo=\'distilbert/distilbert-base-uncased-finetuned-sst-2-english\'\n        tok=AutoTokenizer.from_pretrained(repo);base=AutoModelForSequenceClassification.from_pretrained(repo,attn_implementation=\'eager\').eval()\n        class Wrapper(torch.nn.Module):\n            def __init__(self,base):super().__init__();self.base=base\n            def forward(self,input_ids,attention_mask):return self.base(input_ids=input_ids,attention_mask=attention_mask).logits\n        model=Wrapper(base).eval();names=[\'input_ids\',\'attention_mask\'];args=(torch.ones((1,128),dtype=torch.long),torch.ones((1,128),dtype=torch.long))\n        p=onnx.load(path) if path.exists() else export_model(model,args,path,names,{k:{0:\'batch\'} for k in names+[\'logits\']})\n        diff=check_export(model,args,p,names,cfg);rev=getattr(base.config,\'_commit_hash\',None);del model,base;gc.collect()\n        ds=load_dataset(\'stanfordnlp/imdb\');tr=indices(len(ds[\'train\']),cfg.fit+cfg.dev,seed);te=indices(len(ds[\'test\']),cfg.test,731)\n        train=ds[\'train\'].select(tr.tolist());test=ds[\'test\'].select(te.tolist())\n        def enc(texts,max_len=128):\n            e=tok(list(texts),padding=\'max_length\',truncation=True,max_length=max_len,return_tensors=\'np\');return {k:e[k].astype(np.int64) for k in names}\n        fit=enc(train[:cfg.fit][\'text\']);dev=enc(train[cfg.fit:][\'text\']);testfeeds=enc(test[\'text\']);shift=enc(test[\'text\']);shift[\'attention_mask\'][:,64:]=0;shift[\'input_ids\'][:,64:]=tok.pad_token_id\n        labels=np.asarray(test[\'label\'],np.int64)\n        meta={\'checkpoint\':repo,\'checkpoint_revision\':rev,\'model_sha256\':sha(path),\'export_max_abs\':diff,\'dataset\':\'stanfordnlp/imdb\',\'dataset_fingerprints\':{k:v._fingerprint for k,v in ds.items()},\'fit_ids\':tr[:cfg.fit].tolist(),\'dev_ids\':tr[cfg.fit:].tolist(),\'test_ids\':te.tolist(),\'shift\':\'truncate to 64 tokens; fixed shape 128\'}\n        case=Case(name,p,fit,dev,testfeeds,\'classification\',labels,shift,labels,\'SST-2 pretrained checkpoint; IMDb train for PTQ, official IMDb test; cross-domain sentiment.\')\n    else:raise ValueError(name)\n    case.provenance=meta;return case\n', 'prior.json': '[[[[1.3862943611198906,0.02725802414254978,0.4075766428187993,1.5305988942994255],[1.3862943611198906,0.025443634831999754,0.4972051391204795,1.8751445694550601],[0.5596157879354227,0.22651251637250908,0.076896515158215,2.846824083343942],[0.5596157879354227,0.23428288057163163,0.0756954290813315,3.1159789617111704],[0.5596157879354227,0.23560161734849305,0.0767502216374135,1.9147672686824546],[0.5596157879354227,0.08069057133586309,0.190783787831595,3.105786986799515],[0.5596157879354227,0.032633948728759125,0.1046841474495382,2.234839234574024],[1.3862943611198906,0.03388611863115673,0.0917748981944739,3.8863714412155206],[0.5596157879354227,0.1026463450893067,0.0773094327831942,2.4032698865853495],[0.5596157879354227,0.10267711596720086,0.0772612918178347,2.447732102543025],[0.5596157879354227,0.10361478380366711,0.0748395076210696,2.1232806321249695],[0.5596157879354227,0.11332916101515217,0.1147365555032689,2.625932419346971],[0.5596157879354227,0.06332130112201964,0.102207119886267,2.7953604914773824],[1.3862943611198906,0.02000592574626931,0.0942672664325143,4.549291468828566],[0.5596157879354227,0.1378094454583842,0.0784443249009858,2.2231635908814855],[0.5596157879354227,0.13888165926754212,0.0767979788447563,2.169802195218116],[0.5596157879354227,0.13711305451086778,0.0784344333281885,1.885158236182969],[0.5596157879354227,0.06401505256222884,0.1080320579839387,2.2821302938103125],[0.5596157879354227,0.08912491266300593,0.1029057167005199,3.2953003210842895],[1.3862943611198906,0.15561592548434036,0.116763537889929,5.408061844311521],[0.5596157879354227,0.13568480567220162,0.0883516725693934,1.9732039194720223],[0.5596157879354227,0.1352395386412876,0.0848378775582524,2.3684073241711148],[0.5596157879354227,0.13755646521204098,0.0863949572074479,2.104685493769493],[0.5596157879354227,0.17292206143962194,0.0628258793656715,2.033547847711628],[0.5596157879354227,0.11521628280579652,0.0956893413529189,2.498144536344329],[1.3862943611198906,0.08813697372648541,0.1465844563687362,4.357754740207214],[0.5596157879354227,0.1348342589980552,0.0898967092360742,1.8389743418739248],[0.5596157879354227,0.13735114526626863,0.0884913435813333,2.67893030739065],[0.5596157879354227,0.1340499125042344,0.0882640103307784,1.8692412060569923],[0.5596157879354227,0.08046416592772733,0.1180908462896497,2.380871167189764],[0.5596157879354227,0.07070466992608947,0.1564813332024421,3.0783024675168096],[1.3862943611198906,0.02983758948962538,0.2947628438825379,4.626033871670849],[0.5596157879354227,0.0771047781316961,0.1420104861353844,2.640727362591262],[0.5596157879354227,0.07642735093992847,0.1421812640485925,2.323905484845081],[0.5596157879354227,0.07460555700507224,0.1456312038864307,2.016610942673226],[0.5596157879354227,0.03287005352718375,0.3512746325077045,2.4962988894073264],[0.5596157879354227,0.04659459470986663,0.3105690198911831,2.623851209071839],[1.3862943611198906,0.040015637312977324,0.4565484434111921,4.464096237754022]],[[0.8881612125125582,0.885356080030085,0.9673405155593334],[0.2971352492664341,0.2958755710492229,0.3070247094346874],[0.07370525294411723,0.07245173036997499,0.09199304533354435],[0.08130994810872205,0.07803385077091218,0.09581073607763924],[0.07001214450406001,0.06751059843428642,0.09703172444746777],[0.13322155445727035,0.13019135536770876,0.15264752576905227],[0.052806389106812346,0.05246485086608155,0.06007902314657576],[0.0,0.0,0.1095324054866007],[0.08593662951879821,0.08333828117426045,0.09885707410139699],[0.08258788646971382,0.08032678428017925,0.0969124939189896],[0.09130366499231002,0.08875243556630119,0.10250005640994031],[0.14215208847422178,0.13822838584725095,0.13346779721343427],[0.048301304711174775,0.048364960998330986,0.05243711252873284],[0.0,0.0,0.05039526964315899],[0.10440425784699403,0.10302577127957324,0.10684610015902929],[0.10557468452530172,0.10604232400624003,0.11006691442716343],[0.10196601359992297,0.1010985291807236,0.1037562068775173],[0.06570155605284045,0.060822064695987055,0.09134312754504059],[0.038475176325933225,0.03836161099410862,0.03898941258776245],[0.0,0.0,0.43141772552863555],[0.10674678381896174,0.10476664476151987,0.11129726948573448],[0.1046885403745561,0.10378888511763462,0.10863353626582617],[0.10653065279619534,0.10582047628617952,0.11240290238620776],[0.03247681148033643,0.02894705439910139,0.06704743371273725],[0.039677392753824425,0.03929757750729501,0.03952658876189996],[0.0,0.0,0.048379273389422384],[0.10681284213067344,0.10745587143669444,0.11093227143052098],[0.12532112139008178,0.126611574311967,0.13702551662605156],[0.11807115795015999,0.11618218302414918,0.12096841566890393],[0.11581883027760065,0.11539970053031623,0.15901776547768343],[0.03885804872487375,0.0385923922623298,0.039648824372867986],[0.14103333314301658,0.10317257818013303,0.35992454413520003],[0.1824581176332869,0.18244112651775868,0.19899227059553193],[0.15377839649223324,0.151458062793281,0.16229150206912135],[0.15419581726483944,0.1524606111528577,0.16575354776653728],[0.2119949714739866,0.21189897433977878,0.23377008526862986],[0.060223962837108846,0.06097438177793724,0.06330744382607456],[0.1482354986603398,0.13572820310650247,0.17409794347635055]]],[[[1.791759469228055,0.030369369976641455,0.4947376586970732,1.9445594782603906],[0.026025565277498032,0.04344387473883759,0.8465709392793564,1.9546599583738455],[0.008750663520185453,0.11540322652650453,0.9302350513772238,1.1163172535276964],[0.030771658666753687,0.03905388320483675,0.4474626983909928,1.6167477097527174],[0.015504186535965254,0.3071986442932082,0.1857412610788119,1.7800705751170143],[0.008750663520185453,0.15414171814316463,0.671534013323936,1.0872893865799222],[0.08961215868968714,0.024002943972536585,0.393784288746501,1.5779753141869735],[0.02316705928153438,0.17237127909020147,0.1881199048034983,1.7200379359533724],[0.008750663520185453,0.14210311476401477,0.7539446910158267,1.1511871638075732],[0.13157635778871926,0.0271508022200145,0.3525884984377891,1.858424685655123],[0.02316705928153438,0.33642172757686356,0.1286570195184073,1.8537575413663097],[0.008750663520185453,0.2101822956800428,0.4634709981251927,0.97201479537967],[0.13157635778871926,0.0238208351821081,0.4608050657918809,1.7763156524764807],[0.030771658666753687,0.1923813034966069,0.1477586902284018,1.800170530506718],[0.008750663520185453,0.13376848349894108,0.8086966037981226,1.1818055139557806],[0.17185025692665923,0.03371492638954613,0.2572592555031601,1.7300678467229833],[0.030771658666753687,0.2650351227241647,0.1024599373861684,1.7881887518210298],[0.008750663520185453,0.15355707838228339,0.6851096413997407,1.1280025807157696],[0.17185025692665923,0.027032359705870077,0.2635435998037306,1.7518076744932025],[0.030771658666753687,0.285629180476916,0.1019196901338856,1.8222966826029634],[0.008750663520185453,0.18753006817164797,0.5391409989954754,0.9863697491884109],[0.17185025692665923,0.02282096277209639,0.5411952266068786,1.8012279526876183],[0.06062462181643484,0.12718287820402274,0.1273047025427439,1.9164453933996637],[0.008750663520185453,0.13911366634316247,0.7707135022889033,1.139756035789541],[0.3184537311185346,0.042236498970046686,0.2897371027380022,1.9505484848229853],[0.06062462181643484,0.19192826141704925,0.0993798421893237,1.7968981050874497],[0.008750663520185453,0.163937805337619,0.6302627313396639,1.1133586469993044],[0.3184537311185346,0.034258711529098505,0.281749964356185,1.9646809917940908],[0.06062462181643484,0.23479297975959443,0.0826469434497273,1.7703599240394592],[0.008750663520185453,0.18978567176454375,0.522003961464339,1.1178407648216553],[0.3184537311185346,0.032304336792042185,0.2513287235009653,2.0914885094143036],[0.06062462181643484,0.24684935396941768,0.0749115369940452,1.8495338610583678],[0.008750663520185453,0.19232665402386828,0.5195111481950496,1.2972379586408154],[0.3184537311185346,0.02480673755020711,0.4409559288194086,2.093696372961446],[0.08961215868968714,0.20301714436336124,0.0913634211824515,1.8992096246767394],[0.008750663520185453,0.20517282821313365,0.4695974600961454,1.1527388726266377],[0.44628710262841953,0.03503796133160006,0.2823847254709169,1.8311195968846392],[0.08961215868968714,0.25941638259999417,0.0771145362819622,2.2957221848846094],[0.008750663520185453,0.21735548912937577,0.4346276251861376,1.1414666741517983],[0.44628710262841953,0.02759400457797685,0.2980826119819382,2.447729336226114],[0.08961215868968714,0.16875823423744532,0.0813616825764756,2.0565024417285507],[0.008750663520185453,0.22204738126960655,0.2711995076475625,1.0002472616575069],[0.44628710262841953,0.029482376304809713,0.4970396466852457,1.8511476539254015],[0.1451820098444979,0.04053893124426774,0.2669360449223062,2.3655555190963775],[0.008750663520185453,0.20163195356853988,0.4865145052870371,1.0641425422802433],[0.661398482245365,0.03642675887663289,0.2892674048425528,1.9142220788754685],[0.1451820098444979,0.06370943206410881,0.1637123603422716,2.1385101508641777],[0.008750663520185453,0.23696229272192204,0.3931958720690026,1.0858454783884859],[0.661398482245365,0.032884885654314536,0.3569149460949127,2.0427428643651573],[0.1451820098444979,0.022793921909155676,0.3835192351133948,2.3007044923626263],[0.008750663520185453,0.28052035358720045,0.2944299126210235,0.8166887971392642],[0.661398482245365,0.02075638496490143,0.5707881525307191,1.820342767371216],[0.27193371548364176,0.2800356399042445,0.0695559992316045,1.8378935541235843]],[[0.13927147780414917,0.1304083239697994,0.16525977986299056],[0.9024703360169513,0.9472740172848753,0.976116279473901],[0.8511688207783529,0.8649333723351178,0.8465982118257872],[0.7845349586234867,0.8773609363523295,0.8247045907578451],[0.3610200820452609,0.37175537627841826,0.3278561910527855],[0.7885462393454065,0.8078616586325487,0.8021999626753686],[0.7236797362613314,0.7612604550064471,0.7297985006531276],[0.4080262694069845,0.42951166131013874,0.3599789038666341],[0.7090730036336572,0.7444877451673735,0.7344378024768334],[0.5803080858822791,0.5943109321698369,0.5142913375439485],[0.27289659323329807,0.2789105897200579,0.21792886885553753],[0.5189246215479784,0.5495319846505521,0.5141312611404685],[0.690829176969403,0.7361581478287831,0.6841067259611378],[0.34542555178181444,0.368916241673351,0.2909065708526791],[0.6828259038755542,0.7268938142587906,0.7234970887147443],[0.5277819839796306,0.5484254873322293,0.47368179048326253],[0.26065872068861107,0.26546507293165766,0.18660933232528376],[0.5968421502239896,0.6615546445709108,0.6778476222945323],[0.607931985066794,0.6259798998067059,0.6134686459241632],[0.22002469035781258,0.22687990470242114,0.1501173399054081],[0.6761629407970329,0.7107314837333364,0.6914063736460949],[0.7888504392834744,0.8089006124108788,0.7934821686834566],[0.3110274678708924,0.3174290083785616,0.23091879267621462],[0.6387131107233894,0.6911171953853972,0.7029745602529587],[0.5164746910005281,0.5318736433326823,0.5132983270089867],[0.21546972560006794,0.22155860993942778,0.1451333911128302],[0.6796711726529554,0.7463564083738283,0.7295532742457456],[0.5722626745956353,0.5830606295297878,0.5778339194684257],[0.15628972152055942,0.1560398706932955,0.09455668843857923],[0.5435269109903655,0.5934266022907198,0.5808438175952425],[0.5643004454009188,0.5807925575855256,0.5806441228645378],[0.1370563234579265,0.13744812547005547,0.07448741936452963],[0.5284131184003894,0.5921717596130935,0.6071873597884228],[0.6926071753302726,0.7124030518251208,0.6931077869599337],[0.16760709840592147,0.16838775618825552,0.11282180066019443],[0.5212215194028222,0.5742304885706956,0.5805441556714255],[0.506339250821376,0.5104932140760678,0.5384790364337634],[0.11423177745566648,0.11633201385414693,0.06598673053345117],[0.4473630029493083,0.49092740145958524,0.47665691077603534],[0.5194900840500246,0.541273812683732,0.5460319012071665],[0.09943015700364355,0.10086715487715225,0.053475293958404446],[0.558430617089828,0.583312138749304,0.5170008196845972],[0.8068225299313069,0.8196647118426144,0.822922989362989],[0.13098391321148417,0.1336154689215392,0.12632230171341519],[0.4332688313740659,0.48233165318491455,0.4817953591311075],[0.6027921783457202,0.6015836195458476,0.5995603997749082],[0.15225494823183905,0.15296054636975442,0.13888012746995781],[0.39017595210215417,0.4251486599025661,0.4016173196777222],[0.717534494868891,0.7291402918915181,0.7451327112822439],[0.14215230652131083,0.1484817771901883,0.13571474737650624],[0.4674803200347456,0.5077557581939516,0.42917040026258824],[0.5511182285289892,0.5689060231417971,0.5913253664941511],[0.021387174557069843,0.019876615130800634,0.01249717893097202]]],[[[1.0986122886681098,0.029153669626368624,0.5298041946711886,2.424560416664792],[0.13414155799826424,0.019904837447150853,0.780199392882611,2.5889056885563604],[0.44628710262841953,0.020125894754709106,0.4982621025463526,2.6940438795091635],[0.44628710262841953,0.027278211701814832,0.4340374218818421,2.5854788185824655],[0.44628710262841953,0.0223289073241326,0.5474479097417572,2.4772272129163184],[0.44628710262841953,0.040162261832528454,0.2814095399612499,2.571155146949139],[0.44628710262841953,0.02583925366653461,0.4929232461154372,2.6010986311181905],[0.7537718023763802,0.050060720341057935,0.2196510242641252,2.8314701294554783],[0.06062462181643484,0.025565727832487905,0.5105361220140959,2.536685172186171],[0.7537718023763802,0.044632831909662794,0.2790370163285028,2.6237825643515094],[0.7537718023763802,0.10251030445692132,0.1316026786519624,2.872141049731871],[0.7537718023763802,0.050855688446639444,0.2520971360074416,2.605978634401443],[1.1786549963416462,0.07081622383409028,0.1818331251375974,3.0657518515354214],[0.11778303565638346,0.04628074761418598,0.2887758013954352,2.8381627151455757],[1.1786549963416462,0.05352644951286165,0.2373004658189202,2.8170189134273893],[1.1786549963416462,0.12465998362903592,0.089602333988326,3.1763052896564465],[1.1786549963416462,0.04817998418651342,0.2284836103301848,3.0476367188533144],[1.7047480922384253,0.10389520777922046,0.0877569190181526,3.1587067354896177],[0.22314355131420976,0.041275721517030906,0.266051395493378,2.8537129837312083],[1.7047480922384253,0.10216633726691549,0.0965049608344744,3.0450967068479353],[1.7047480922384253,0.11915372634723409,0.1007187888393092,2.8755245257198334]],[[0.15291391895148287,0.1470325241159326,0.1708874236327267],[0.8116954515863234,0.8910751773205147,0.9400615023182045],[0.7907954643515842,0.799449192370097,0.9011972451576218],[0.6788142346620669,0.6755291507677055,0.8134283974071919],[0.7458914745010872,0.754151217742214,0.8420955399446912],[0.4589943147904798,0.427316291082087,0.6326872217416522],[0.6449487780386238,0.6506370143385494,0.7769573776557066],[0.23450822790257408,0.21342548996984723,0.4687130208049999],[0.6225983504876427,0.6556154644265182,0.62751846239895],[0.3586899350859927,0.3084494080207515,0.5671543631716747],[0.0,0.0,0.3087024832567079],[0.22949284822911442,0.15729102282925,0.5028859550858651],[0.0,0.0,0.34752650357329573],[0.37015555762630714,0.38239462637482413,0.33919367498192377],[0.1803510658226345,0.0772833531730226,0.5799601257588528],[0.0,0.0,0.3919206252413342],[0.012068975010119831,0.0,0.2934305580931833],[0.0,0.0,0.0],[0.16598277494096625,0.16864424930375221,0.15829047181745548],[0.0,0.0,0.0],[0.0,0.0,0.0]]],[[[0.4054651081081644,0.17156045117223825,0.0574198761510358,1.9465646884740724],[0.026025565277498032,0.04346793061498956,0.8458486960010689,1.5181244935878453],[0.0458095360312942,0.035392281924954326,0.561615448648869,2.2918655174630373],[0.3519764231571782,0.02898389130888669,0.5144157656451911,2.352830259553702],[0.3184537311185346,0.04661617612168514,0.2692166073575337,2.1701409908951392],[0.06062462181643484,0.035639822432457315,0.4108624058790768,2.0713276608169156],[0.44628710262841953,0.03958000606114386,0.3669096561008977,2.4322606793547914],[0.3184537311185346,0.05882966977145773,0.2422538118540422,2.2001495139631086],[0.06062462181643484,0.033548746230495856,0.4411978654254468,1.8855049863107642],[0.44628710262841953,0.03722376788215862,0.3963618151620716,2.2919636256899305],[0.4054651081081644,0.07758684587239353,0.1738766600359323,2.0930249946236694],[0.015504186535965254,0.09888523406609423,0.498979807427422,1.8576809449663292],[0.13157635778871926,0.026211048403611697,0.4989047949794654,2.4888280869442583],[0.06062462181643484,0.024670328991329066,0.5523016991286668,1.855171763910155],[0.015504186535965254,0.10200640682066044,0.5690520551607225,1.8238609164298163],[0.13157635778871926,0.02622486819473362,0.5719380605328133,1.9884121313506633],[0.11778303565638346,0.03789806942170563,0.2804458923500023,1.7745363556781497],[0.030771658666753687,0.047059840226459856,0.6148793134910168,1.9812297433468846],[0.24783616390458127,0.024942828570794894,0.5844301744593896,2.490728050545931],[0.11778303565638346,0.02521842965956774,0.5707239673416633,2.1324297748204666],[0.030771658666753687,0.05618998445469413,0.5173931097848724,2.0998198326972273],[0.24783616390458127,0.02995908707504079,0.4919079663208409,2.4449041220422614],[0.22314355131420976,0.034255333305266845,0.2923849193108388,2.1057610247755303],[0.0458095360312942,0.03228718823527006,0.5996889686791875,1.91956010308666],[0.3519764231571782,0.026524871040968224,0.5480427309306994,2.724798392319609],[0.22314355131420976,0.024677428231261806,0.5629234307538051,2.255002129013776]],[[0.0,0.0,0.0012432025604132146],[0.8012501300481897,0.887004698823944,0.9168959823208698],[0.5606370301469876,0.6024423895029475,0.5969994663997208],[0.5904594708677171,0.6019697502654453,0.750284899797903],[0.5607129714676933,0.5774456557349954,0.5308410280018923],[0.4713688577409396,0.5100961494600273,0.47811864984734564],[0.4887207905944433,0.4811272959297259,0.6760423680033696],[0.5342247542492455,0.5340128209446173,0.48403173178613146],[0.43904526611969585,0.4693553953544847,0.4612716661997195],[0.5065660450882143,0.4851038816570469,0.7034305245892287],[0.36877318172861695,0.3641699844731259,0.33506480596794275],[0.5977466785571517,0.5994977230573444,0.5509395695091459],[0.6576585177802416,0.7155888884284665,0.7871349862511193],[0.8268222509174391,0.8470682748962501,0.8253265540654258],[0.4290138597577281,0.5095224552925294,0.4943422992969574],[0.655057967121494,0.697643152629102,0.8025665770505758],[0.6933149222390197,0.6954982152703314,0.623379543679483],[0.5642458580167597,0.6128880189569175,0.5888735360304961],[0.7113679850190238,0.733874779785959,0.8383395604472244],[0.7654657698246761,0.7908793332141618,0.7540682236081944],[0.5098653708654965,0.5608966995356136,0.5411681378955677],[0.6502973399319503,0.6716906179211339,0.7978895306585698],[0.6684087750313594,0.6764964470891663,0.6110104007870073],[0.5635615466989444,0.6299818472830048,0.6074578092804368],[0.6537826679600012,0.6545181895165296,0.7945876379296981],[0.7423518706865523,0.769021577078449,0.7345353582232431]]]]', 'requirements-kaggle.txt': '# Не переустанавливать PyTorch/CUDA из Kaggle!\nonnx>=1.16,<2\ntransformers>=4.45,<5\ndatasets>=3.0,<5\npillow>=10,<13\nthreadpoolctl>=3.4\npsutil>=5.9\nscikit-learn>=1.4\npandas>=2.0\nmatplotlib>=3.8\n\ntimm>=1.0.15,<2\n'}
for name, source in SOURCES.items():
    path = ROOT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source, encoding="utf-8")
print("Written", len(SOURCES), "files")

### Интеграционные тесты

In [ ]:
SOURCES = {'tests/test_adapters.py': "import importlib.util,unittest\nimport numpy as np\nfrom graph import Target,sample_rows,conv_rows\nfrom models import conv_numpy,split_indices\nfrom benchmark import quant,PROFILES,fidelity\n\nclass AdapterTests(unittest.TestCase):\n    def test_gemm_transpose_alpha_beta(self):\n        r=np.random.default_rng(2);x=r.normal(size=(5,4)).astype(np.float32);w=r.normal(size=(3,4)).astype(np.float32);b=r.normal(size=3).astype(np.float32)\n        y=2*(x@w.T)+.5*b;t=Target('w','n','Gemm','x','y',w,{'transA':1,'transB':1,'alpha':2.,'beta':.5},b)\n        rows,n=sample_rows(x.T,y,t,5,r);self.assertEqual(n,5);self.assertEqual(rows[0].shape,(5,4))\n        q=np.arange(12,dtype=np.int8).reshape(4,3);s=np.ones(3,np.float32)\n        a,b=t.pack([(q,s)]);np.testing.assert_array_equal(a,q.T);np.testing.assert_array_equal(t.stored(t.canonical(a,b))[0],q.T)\n    def test_conv_groups_depthwise_and_padding(self):\n        r=np.random.default_rng(2);x=r.normal(size=(2,4,5,6)).astype(np.float32)\n        for groups,out in [(1,6),(2,6),(4,8)]:\n            w=r.normal(size=(out,4//groups,3,3)).astype(np.float32);y=conv_numpy(x,w,groups)\n            t=Target('w','n','Conv','x','y',w,{'group':groups,'auto_pad':b'SAME_UPPER'})\n            rows=conv_rows(x,y,t,np.arange(2*5*6));self.assertEqual(len(rows),groups)\n            self.assertEqual(rows[0].shape,(60,4//groups*9))\n            pairs=[(np.rint(m).astype(np.int8),np.ones(m.shape[1],np.float32)) for m in t.matrices()]\n            q,s=t.pack(pairs);np.testing.assert_array_equal(q,np.rint(w));np.testing.assert_array_equal(t.stored(t.canonical(q,s))[0],q)\n    def test_conv_stride_dilation(self):\n        x=np.arange(49,dtype=np.float32).reshape(1,1,7,7);w=np.array([[[[1,2],[3,4]]]],np.float32)\n        y=np.array([[[[np.sum(x[0,0,i:i+3:2,j:j+3:2]*w[0,0]) for j in (0,2,4)] for i in (0,2,4)]]],np.float32)\n        t=Target('w','n','Conv','x','y',w,{'strides':[2,2],'dilations':[2,2]});rows=conv_rows(x,y,t,np.arange(9));self.assertEqual(rows[0].shape,(9,4))\n    def test_matmul_mask(self):\n        r=np.random.default_rng(1);x=r.normal(size=(2,3,4)).astype(np.float32);w=r.normal(size=(4,2)).astype(np.float32);mask=np.array([[1,1,0],[1,0,0]])\n        t=Target('w','n','MatMul','x','y',w,{})\n        rows,n=sample_rows(x,x@w,t,99,r,mask);self.assertEqual(n,3);self.assertEqual(rows[0].shape,(3,4))\n    def test_split_and_metrics(self):\n        tr,a,b,c=split_indices(100,42,12,6,20);_,a2,b2,c2=split_indices(100,123,12,6,20)\n        self.assertEqual(len(set(np.concatenate([tr,a,b,c]))),sum(map(len,[tr,a,b,c])));np.testing.assert_array_equal(c,c2)\n        r={'out':np.array([[0.,1.],[1.,0.]])};p={'out':np.array([[0.,1.],[0.,1.]])};m,e=fidelity(r,p,'classification',np.array([1,0]));self.assertEqual(m['changed'],1);self.assertEqual(m['accuracy'],.5);self.assertEqual(m['mse_equal_output_mean'],.5)\n    def test_bank_formats(self):\n        r=np.random.default_rng(2);x=r.normal(size=(32,12)).astype(np.float32);w=r.normal(size=(12,5)).astype(np.float32);base,_=quant(w,x,'MinMax',PROFILES['smoke']);loss=lambda q,s:np.sum((x.astype(float)@((q*s).astype(float)-w))**2)\n        for name in ('Single-3','Paired-3','SDR'):\n            (q,s),info=quant(w,x,name,PROFILES['smoke']);self.assertEqual(q.dtype,np.int8);self.assertEqual(s.dtype,np.float32);self.assertLessEqual(loss(q,s),loss(*base)+1e-10)\n\n@unittest.skipUnless(importlib.util.find_spec('onnx') and importlib.util.find_spec('onnxruntime'),'ONNX/ORT недоступны')\nclass RuntimeTests(unittest.TestCase):\n    def test_real_graph(self):\n        import tempfile\n        from pathlib import Path\n        from models import make_case\n        from benchmark import session,infer,capture,quant\n        from graph import discover,export\n        cfg=PROFILES['smoke']\n        for name in ('cnn','depthwise_cnn','regression','rnn','autoencoder','multi_output'):\n            case=make_case(name,42,cfg,Path('.'));targets,skipped=discover(case.model);self.assertTrue(targets)\n            bank={}\n            for i,t in enumerate(targets):\n                xs,_=capture(case.model,t,case.fit,cfg,42+i);pairs=[quant(w,x,'SDR',cfg)[0] for w,x in zip(t.matrices(),xs)];bank[t.weight]=t.canonical(*t.pack(pairs))\n            model=export(case.model,targets,bank);a=infer(session(case.model,1),case.test,cfg.batch);b=infer(session(model,1),case.test,cfg.batch)\n            self.assertEqual(set(a),set(b));self.assertEqual(sum(n.op_type=='DequantizeLinear' for n in model.graph.node),len(targets))\n            for k in a:self.assertEqual(a[k].shape,b[k].shape);self.assertTrue(np.isfinite(b[k]).all())\nif __name__=='__main__':unittest.main()\n", 'tests/test_cuda_math.py': "import unittest,os\nfrom unittest.mock import patch\nimport numpy as np\nimport torch\nfrom algorithms import activation_int8 as v1,spectral_rounding as sr\nfrom cuda_math import v1_torch,sdr_torch\n\nclass TorchMath(unittest.TestCase):\n    def devices(self):return ['cpu']+(['cuda'] if torch.cuda.is_available() else [])\n    def test_v1_against_original(self):\n        for seed in (11,23,47):\n            rng=np.random.default_rng(seed);x=rng.normal(size=(40,24)).astype('f');w=rng.normal(size=(24,35)).astype('f');x[:,3]=0;w[:,0]=0\n            for paired in (False,True):\n                if paired:q,s=v1.quantize_int8(x,w)\n                else:\n                    with patch.object(v1,'_pairs',return_value=[]):q,s=v1.quantize_int8(x,w)\n                for device in self.devices():\n                    qq,ss=v1_torch(x,w,paired,device=device)\n                    np.testing.assert_array_equal(qq,q);np.testing.assert_allclose(ss,s,rtol=1e-6,atol=0)\n    def test_sdr_against_original(self):\n        for seed in (11,23,47):\n            rng=np.random.default_rng(seed);x=rng.normal(size=(40,24)).astype('f');w=rng.normal(size=(24,9)).astype('f');x[:,3]=0;w[:,0]=0\n            q=sr.quantize(w,x);base=sr.quantize(w,None)\n            for device in self.devices():\n                (a,b),info=sdr_torch(x,w,device=device)\n                np.testing.assert_array_equal(b,q.scales)\n                np.testing.assert_array_equal(a,q.codes)\n                self.assertEqual(info['accepted'],q.diagnostics['accepted'])\n                loss=lambda z,s:np.sum((x.astype(float)@(w-(z*s)))**2,axis=0)\n                self.assertTrue(np.all(loss(a,b)<=loss(base.codes,base.scales)))\n    def test_sdr_budget(self):\n        x=np.eye(3,dtype='f');w=x.copy();(a,b),i=sdr_torch(x,w,sr.Config(work_limit=0),device='cpu');self.assertEqual(i['reason'],'budget_or_width')\n\nif __name__=='__main__':unittest.main()\n", 'tests/test_policies.py': "import unittest\nimport numpy as np\nfrom adaptive_policies import policy, check, aggregate, Loss, Portfolio\nclass Policies(unittest.TestCase):\n    def test_progressive_early_stop(self):\n        l={'minmax':Loss(1,1,1),'single':Loss(.5,.5,1),'paired':Loss(.2,.2,1),'sdr':Loss(.1,.1,1)}\n        self.assertEqual(policy('Progressive',l,np.zeros(4),Portfolio([]))[:2],('single',('single',)))\n        l['single']=Loss(.1,1.1,1)\n        self.assertEqual(policy('Progressive',l,np.zeros(4),Portfolio([]))[:2],('paired',('single','paired')))\n    def test_global_check(self):\n        r={'a':np.zeros(2),'b':np.zeros(2)};b={k:np.ones(2) for k in r}\n        self.assertFalse(check(r,b,{'a':np.zeros(2),'b':np.full(2,2.)})[0])\n        self.assertTrue(check(r,b,{'a':np.zeros(2),'b':np.ones(2)})[0])\n    def test_group_energy(self):\n        g=dict(channels=2,fit_rows=3,dev_rows=2,shape=[4,2],fit_mse=.5,dev_mse=.6,weight_mse=.25,profile_sampled_features=4,stable_rank_sample=2,mean_abs_corr_sample=.1,weight_abs_max=1,weight_rms=.5)\n        l,f=aggregate({'groups':[g,g]});self.assertAlmostEqual(l.energy,1.);self.assertAlmostEqual(l.dev,.6)\n    def test_astra_outside(self):\n        l={'minmax':Loss(1,1,1),'single':Loss(.1,.1,1)}\n        self.assertEqual(policy('Astra-Adaptive',l,np.zeros(4),Portfolio([]))[:2],('minmax',()))\n", 'tests/test_research.py': "import unittest\nfrom dataclasses import replace\nimport numpy as np\nfrom benchmark import quant,PROFILES,fidelity\nfrom pretrained import indices,IMAGENETTE\nfrom graph import Target,conv_rows\n\nclass ResearchTests(unittest.TestCase):\n    def test_caps_leave_other_channels_at_minmax(self):\n        rng=np.random.default_rng(52);w=rng.normal(size=(16,13)).astype(np.float32);x=rng.normal(size=(48,16)).astype(np.float32);x[:,1]=x[:,0]\n        cfg=replace(PROFILES['smoke'],channel_cap=5,sdr_work=10**9)\n        base,_=quant(w,x,'MinMax',cfg);chosen=np.linspace(0,12,5,dtype=int);other=np.setdiff1d(np.arange(13),chosen)\n        for name in ('Single-3','Paired-3','SDR'):\n            (q,s),info=quant(w,x,name,cfg);np.testing.assert_array_equal(q[:,other],base[0][:,other]);np.testing.assert_array_equal(s[other],base[1][other]);self.assertEqual(info['selected_channels'],chosen.tolist());self.assertEqual(q.nbytes+s.nbytes,w.size+4*w.shape[1])\n    def test_research_splits(self):\n        a=indices(1000,192,42);b=indices(1000,192,123)\n        self.assertEqual(len(set(a)),192);self.assertFalse(set(a[:128])&set(a[128:]));self.assertFalse(np.array_equal(a,b));np.testing.assert_array_equal(indices(500,100,731),indices(500,100,731));self.assertEqual(len(set(IMAGENETTE.values())),10)\n    def test_top5_full_class_space(self):\n        y=np.array([1,5]);r=np.array([[0,9,8,7,6,5],[0,1,2,3,4,5]],np.float32);p=r.copy();m,_=fidelity({'o':r},{'o':p},'classification',y);self.assertEqual(m['top5_accuracy'],1);self.assertEqual(m['accuracy'],1)\n    def test_conv3d_selected_patches(self):\n        x=np.arange(64,dtype=np.float32).reshape(1,1,4,4,4);w=np.ones((1,1,2,2,2),np.float32);y=np.array([x[0,0,i:i+2,j:j+2,k:k+2].sum() for i in range(3) for j in range(3) for k in range(3)],np.float32).reshape(1,1,3,3,3);t=Target('w','n','Conv','x','y',w,{})\n        a=conv_rows(x,y,t,np.array([0,12,26]));self.assertEqual(a[0].shape,(3,8));np.testing.assert_array_equal(a[0].sum(1),y.ravel()[[0,12,26]])\nif __name__=='__main__':unittest.main()\n"}
for name, source in SOURCES.items():
    path = ROOT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source, encoding="utf-8")
print("Written", len(SOURCES), "files")

## 2. Окружение и preflight
Проверка окружения CUDA и ONNX Runtime. Установка не переустанавливает PyTorch. Для новых моделей добавлен timm. Реальные CUDA-проверки выполняются на каждой GPU.

In [ ]:
import importlib.metadata as im

def installed(name):
    try: return im.version(name)
    except im.PackageNotFoundError: return None

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", *args], check=True)

cuda = subprocess.check_output([sys.executable, "-c", "import torch; print(torch.version.cuda)"], text=True).strip()
if PROVIDER == "cuda":
    if cuda.startswith("13."): ort_version = "1.30.0"
    elif cuda.startswith("12."): ort_version = "1.22.0"
    else: raise RuntimeError(f"Unsupported PyTorch CUDA: {cuda}")
    print("PyTorch CUDA:", cuda, "target ORT:", ort_version)
if INSTALL_MISSING:
    pip("install", "-q", "-r", str(ROOT/"requirements-kaggle.txt"), "flatbuffers")
    if PROVIDER == "cuda":
        if installed("onnxruntime") is not None or installed("onnxruntime-gpu") != ort_version:
            pip("uninstall", "-y", "onnxruntime", "onnxruntime-gpu")
            pip("install", "-q", f"onnxruntime-gpu=={ort_version}")
    elif installed("onnxruntime") is None and installed("onnxruntime-gpu") is None:
        pip("install", "-q", "onnxruntime>=1.22,<2")

# CUDA runtime из той же major-версии, что и PyTorch; подготавливаем окружение дочерних процессов.
env = {**os.environ, "QUANT_PROVIDER": PROVIDER, "PYTHONUTF8":"1", **{k:str(CPU_THREADS) for k in ("OPENBLAS_NUM_THREADS","OMP_NUM_THREADS","MKL_NUM_THREADS","BLIS_NUM_THREADS")}}
if PROVIDER == "cuda":
    import site
    major=cuda.split(".")[0]
    libs=[]
    for base in site.getsitepackages():
        libs.extend(Path(base).glob("nvidia/*/lib"))
        libs.extend(Path(base).glob("nvidia/*/lib64"))
    # Если найден нужный libcudart, явно задаём его cuDNN Frontend.
    candidates=[p for d in libs for p in d.glob(f"libcudart.so.{major}")]
    candidates += list(Path("/usr/local/cuda/lib64").glob(f"libcudart.so.{major}"))
    if candidates: env["CUDNN_FRONTEND_CUDART_LIB_NAME"]=str(candidates[0])
    env["LD_LIBRARY_PATH"]=os.pathsep.join(dict.fromkeys([str(p) for p in libs]+[env.get("LD_LIBRARY_PATH","")])).rstrip(os.pathsep)
    subprocess.run(["nvidia-smi"],check=False)
print("ORT:", installed("onnxruntime-gpu") or installed("onnxruntime"))
if RUN_PREFLIGHT:
    # CPU-математика + интеграционные адаптеры; отдельные CUDA-проверки в главном запуске.
    test_env={**env,"CUDA_VISIBLE_DEVICES":"","QUANT_PROVIDER":"cpu","QUANT_INFER_PROVIDER":"cpu"}
    for folder in ("algorithms","tests"):
        subprocess.run([sys.executable,"-m","unittest","discover","-s",folder,"-v"],cwd=ROOT,env=test_env,check=True)


## 3. Запуск с компактным прогрессом
Логи пишутся на диск. В браузере обновляется один блок, для уменьшения нагрузки на интерфейс. Параллельные GPU-воркеры сохраняют checkpoint и отчёты.

In [ ]:
from IPython.display import display
from html import escape
from IPython.display import HTML
args=[sys.executable,"-u",str(ROOT/"benchmark.py"),"--profile",PROFILE,"--provider",PROVIDER,"--threads",str(CPU_THREADS),"--gpus",str(GPU_COUNT),"--math-backend",MATH_BACKEND,"--capture-group",str(CAPTURE_GROUP),"--output",str(OUT),"--methods",*METHODS]
for opt,value in [("--rows",ROWS_OVERRIDE),("--channel-cap",CHANNEL_CAP),("--v2-layers",V2_LAYER_CAP),("--sdr-work",SDR_WORK_OVERRIDE)]:
    if value is not None: args += [opt,str(value)]
if MODELS: args += ["--models",*MODELS]
if SEEDS: args += ["--seeds",*map(str,SEEDS)]
if SAVE_MODELS: args += ["--save-models"]
if ALLOW_CONV_CPU_FALLBACK: args += ["--conv-cpu-fallback"]
stop=threading.Event()
def monitor():
    path=OUT/"gpu_usage.csv"
    with path.open("a",encoding="utf-8") as f:
        if path.stat().st_size==0:f.write("unix_time,gpu,util_percent,memory_used_mib,memory_total_mib\n")
        while not stop.is_set():
            try:
                r=subprocess.run(["nvidia-smi","--query-gpu=index,utilization.gpu,memory.used,memory.total","--format=csv,noheader,nounits"],capture_output=True,text=True,timeout=10)
                if r.returncode==0:
                    for line in r.stdout.strip().splitlines():f.write(f"{time.time():.3f},{line}\n")
                    f.flush()
            except (OSError,subprocess.TimeoutExpired):pass
            stop.wait(15)
def tail(path,limit=65536):
    if not path.exists():return []
    with path.open("rb") as f:
        f.seek(max(0,path.stat().st_size-limit));return f.read().decode("utf-8",errors="replace").splitlines()
watch=threading.Thread(target=monitor,daemon=True)
if PROVIDER=="cuda":watch.start()
mainlog=OUT/"run.log";started=time.monotonic();view=display(HTML("<pre>Starting…</pre>"),display_id=True)
proc=None
try:
    with mainlog.open("w",encoding="utf-8") as f:
        proc=subprocess.Popen(args,cwd=ROOT,env=env,stdout=f,stderr=subprocess.STDOUT,start_new_session=True)
        while True:
            rc=proc.poll();lines=tail(mainlog);complete=[x for x in lines if x.startswith("COMPLETED")];jobs=sorted((OUT/"logs").glob("*.log"),key=lambda p:p.stat().st_mtime,reverse=True)[:GPU_COUNT]
            status=[f"Elapsed: {(time.monotonic()-started)/60:.1f} min | {'Running' if rc is None else 'Exit '+str(rc)}",complete[-1] if complete else "Preflight / initialization"]
            for job in jobs:
                useful=[x for x in tail(job) if ' MSE=' in x or 'capture ' in x or x.startswith('    ') and ('/' in x or 'policy' in x)]
                if useful:status.append(job.stem+": "+useful[-1][:240])
            status.extend(lines[-3:]);view.update(HTML("<pre style='white-space:pre-wrap'>"+escape("\n".join(status))+"</pre>"))
            if rc is not None:break
            time.sleep(15)
except KeyboardInterrupt:
    if proc and proc.poll() is None:
        import signal
        os.killpg(proc.pid,signal.SIGTERM);proc.wait()
    raise
finally:
    stop.set()
    if watch.is_alive():watch.join()
print("Exit code:",rc,"(partial reports retained)")


## 4. Сводка и заранее определённые критерии
Оценка по каждому выходу, отдельно для модели и seed. Пропущенные/упавшие конфигурации не считаются успехом. Применимость не равна 100% поддержке операторов.

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display
report_path=OUT/"research_report.json"
if report_path.exists():
    report=json.loads(report_path.read_text())
    df=pd.read_csv(OUT/"summary.csv");display(df[[k for k in ["model","seed","method","status","ratio_to_minmax","accuracy","agreement","prepare_s","fallback_to_all_minmax"] if k in df]])
    ok={(r["model"],r["seed"],r["method"]):r for r in report["runs"] if r["status"]=="ok"}
    rows=[]
    def ratio(a,b):return a/b if b>0 else (1.0 if a==0 else float("inf"))
    for (model,seed,method),r in ok.items():
        if method=="FP32" or (model,seed,"MinMax") not in ok:continue
        b=ok[model,seed,"MinMax"];a=r["metrics"];z=b["metrics"]
        rs=[ratio(v["mse"],z["outputs"][k]["mse"]) for k,v in a["outputs"].items()]
        sh=r.get("shift");bs=b.get("shift");sr=[ratio(v["mse"],bs["outputs"][k]["mse"]) for k,v in sh["outputs"].items()] if sh and bs else []
        rows.append(dict(model=model,seed=seed,method=method,worst_test=max(rs),worst_shift=max(sr) if sr else np.nan,accuracy_delta=a.get("accuracy",np.nan)-z.get("accuracy",np.nan),rmse_ratio=ratio(a["task_rmse"],z["task_rmse"]) if "task_rmse" in a and "task_rmse" in z else np.nan,size_equal=r["model_bytes"]==b["model_bytes"],prepare_s=r["prepare_seconds"],builds=r.get("expensive_builds",np.nan),guarded=r.get("global_guard_enabled",False)))
    metrics=pd.DataFrame(rows)
    if len(metrics):
        metrics.to_csv(OUT/"decision_per_seed.csv",index=False)
        med=metrics.groupby(["model","method"]).agg(seeds=("seed","nunique"),test=("worst_test","median"),shift=("worst_shift","median"),accuracy_delta=("accuracy_delta","median"),rmse_ratio=("rmse_ratio","median"),size_equal=("size_equal","all"),prepare_s=("prepare_s","median"),builds=("builds","median"),guarded=("guarded","all")).reset_index()
        med.to_csv(OUT/"decision_by_model.csv",index=False);display(med)
        expected=set(MODELS or (["efficientnet_b0","minilm_embeddings","multi_output"] if PROFILE=="final_smoke" else ["efficientnet_b0","vit_tiny","minilm_embeddings","mlp","regression","autoencoder","multi_output","lstm_head"]));expected_seeds=set(SEEDS or ([42] if PROFILE=="final_smoke" else [42,123,731] if PROFILE=="final" else [42,123,731,2026,91]));counts={m:len(expected_seeds) for m in expected};decisions=[]
        for method,t in med.groupby("method"):
            if method=="MinMax" or not t.guarded.all():continue
            complete=set(t.model)==expected and all(n==counts.get(m,0) for m,n in zip(t.model,t.seeds)) and all(set(metrics[(metrics.model==m)&(metrics.method==method)].seed)==expected_seeds for m in expected)
            pass_gate=bool(complete and t.size_equal.all() and (t.test<=1.02).all() and (t['shift'].dropna()<=1.05).all() and (t.accuracy_delta.dropna()>=-.01).all() and (t.rmse_ratio.dropna()<=1.02).all() and (t.test<=.95).sum()>=2)
            decisions.append(dict(method=method,complete=complete,passes_engineering_gate=pass_gate,models_improved=int((t.test<=.95).sum()),mean_model_median_worst_test=float(t.test.mean()),sum_model_median_prepare_s=float(t.prepare_s.sum())))
        decision=pd.DataFrame(decisions);display(decision);decision.to_csv(OUT/"decision_gate.csv",index=False)
else:print("No report yet. Inspect run.log and preflight logs.")


## 5. Скачать результаты
ZIP содержит отчёты, компактные массивы ошибок и полные логи, но не банки и модели. Если шаг анализа не прошёл, эту ячейку можно выполнить отдельно.

In [ ]:
from IPython.display import display, FileLink
package=OUT.parent / ("quant_v4_"+PROFILE+"_reports.zip")
with zipfile.ZipFile(package,"w",zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file() and (p.suffix in (".json",".csv",".log") or p.name.endswith(".mse.npy")) and ".layer_" not in p.name:z.write(p,p.relative_to(OUT))
print(package);display(FileLink(str(package.relative_to(Path.cwd()))) if package.is_relative_to(Path.cwd()) else FileLink(str(package)))


## Воспроизведение
После `final_smoke` убедиться, что обе GPU и новые экспорты работают. Затем `PROFILE="final"`, `INSTALL_MISSING=False` **только в той же среде**, Run All. Профиль `final_full` предназначен для расширенного эксперимента. Порог Progressive фиксируется до оценки на test. Результаты используются для выбора продуктовых режимов; независимая оценка требует новых входных моделей. Сравнение с опубликованными методами GPTQ/COMQ/DiscQuant требуется для научных заявлений о превосходстве.